# Hard RGB Diffusion Policy — staged local-first run
Execute each stage separately; **do not use Run All**. All action switches start False.
Reuse `/content/berlin-marso-hackathon` and `/content/marso-py312/bin/python`.
Parent stages `il/demos/hard/trajectory.rgb.pd_ee_delta_pos.physx_cuda.{h5,json}`.
This notebook does not install packages or restart the kernel.

1. Mount check; inspect expected Drive root.
2. Enable reviewed source setup once. Every destination must match its base/current hash;
   unknown sources and existing output directories are rejected. Originals are backed up.
3. Enable dataset/CUDA/reference checks. The reference records valid Hard failures honestly.
4. Enable 100-step CUDA smoke. Inspect finite updates and `smoke_passed`.
5. Enable `RUN_HARD` only after reviewing smoke. Fresh seed 1, 40,000 iterations, AMP,
   batch 128, lr 1e-4, ResNet18 scene RGB, horizons 2/8/16; latest saved every 1,000.
   Training eval: 32 episodes / 8 envs every 5,000. Six-hour training limit.
6. Monitor by exact saved plan path. Re-executing a training phase is rejected.

Outputs: `/content/marso-hard/hard_<timestamp>` and
`MyDrive/marso/recoveries/hard/hard_<timestamp>`. Local completion and remote pending are separate.
The best sort-accuracy checkpoint is evaluated on four local seed-5000 configurations, then
recorded with one video env and eight diagnostic episodes. Scores are local validation,
not official or heldout. Shared eval/diagnostic loops execute 799 steps with an 800-step limit.
A missing best checkpoint (all tracked sort metrics zero) uses final with an explicit fallback record.


In [ ]:
import os
from pathlib import Path
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
assert os.path.ismount('/content/drive'), 'A real Drive mount is required'
assert Path('/content/drive/MyDrive/marso').is_dir(), 'Existing marso Drive root is required'
print('Drive mounted. Parent stages Hard demos separately; a Drive Hard marker is not required.')


In [ ]:
EMBEDDED_SOURCES = {'conf/config.yaml': {'base_sha256': '7117e091b029d493277524e8cd0d61de0a850f74f32b654adc75103f94fae158', 'sha256': '7117e091b029d493277524e8cd0d61de0a850f74f32b654adc75103f94fae158', 'content_b64': 'IyBSb290IGNvbmZpZy4gQ29tcG9zZXMgYSBkaWZmaWN1bHR5IChkZWZhdWx0cyBsaXN0KSArIHNoYXJlZCBwYXJhbXMuCiMgT3ZlcnJpZGUgYW55dGhpbmcgb24gdGhlIENMSSwgZS5nLiAgcHl0aG9uIGV2YWwucHkgZGlmZmljdWx0eT1oYXJkIGNoZWNrcG9pbnQ9PHBhdGg+CmRlZmF1bHRzOgogIC0gZGlmZmljdWx0eTogZWFzeQogIC0gX3NlbGZfCgpzZWVkOiAwCm51bV9lbnZzOiA4ICAgICAgICAgICAgICAgIyBmb3IgZXZhbC90ZXN0OyBzY2FsZSB1cCBhcyBHUFUgYWxsb3dzCmRldmljZTogY3VkYQpjb250cm9sX21vZGU6IHBkX2VlX2RlbHRhX3BvcwoKIyBFcGlzb2RlIGJ1ZGdldCBpcyBQRVIgRElGRklDVUxUWSAoY29uZi9kaWZmaWN1bHR5LyoueWFtbCwga2V5IGRpZmZpY3VsdHkubWF4X2VwaXNvZGVfc3RlcHMpLgojIFRoZSBzY3JpcHRlZCBkZW1vcyBuZWVkIH4xMTUgLyB+MjM2IC8gfjM4OCBzdGVwcyBmb3IgZWFzeSAvIG1lZGl1bSAvIGhhcmQsIHNvIHRoZSBvbGQgZ2xvYmFsIDIwMAojIG1hZGUgbWVkaXVtL2hhcmQgaW1wb3NzaWJsZSB0byBmaW5pc2guIEEgY29ycmVjdCBwbGFjZW1lbnQgaXMgc3RpY2t5IGluIGVudi5ldmFsdWF0ZSgpLCBzbyBhIGxvbmdlcgojIGJ1ZGdldCBjYW4gb25seSByYWlzZSBzb3J0X2FjY3VyYWN5LiBOT1RFOiBIeWRyYSBsb2FkcyBgZGlmZmljdWx0eWAgQkVGT1JFIGBfc2VsZl9gLCBzbyBhIGxpdGVyYWwKIyB2YWx1ZSBoZXJlIHdvdWxkIHNpbGVudGx5IG92ZXJyaWRlIHRoZSBwZXItbGV2ZWwgb25lIC0tIGtlZXAgdGhlIGludGVycG9sYXRpb24uCm1heF9lcGlzb2RlX3N0ZXBzOiAke2RpZmZpY3VsdHkubWF4X2VwaXNvZGVfc3RlcHN9CgojIG9ic2VydmF0aW9uIG1vZGUgKHN0YXRlIHwgcmdiKS4gcmdiID0gc2NlbmUtY2FtZXJhIGltYWdlICsgcHJvcHJpb2NlcHRpb24gKG91ciBtYWluIHRyYWNrKS4Kb2JzX21vZGU6IHJnYgoKY2FtZXJhOgogIHdpZHRoOiAxMjgKICBoZWlnaHQ6IDEyOAoKIyB0aGUgcmdiIG9icyBjb21lcyBmcm9tIGEgc2luZ2xlIGZpeGVkIHRoaXJkLXBlcnNvbiAic2NlbmUiIGNhbWVyYSAodGhlIG9ubHkgaW1hZ2UgaW5wdXQpCm9ic19jYW1lcmE6IHNjZW5lCgojIGNoZWNrcG9pbnQgcGF0aCAoZm9yIGV2YWwucHkgLyB0aGUganVkZ2UpCmNoZWNrcG9pbnQ6IG51bGwKIyBldmFsIGNvbmZpZyBmaWxlIHBhdGggKGV2YWwucHkgb25seSkKZXZhbF9jb25maWc6IG51bGwKIyBwb2xpY3kgZW50cnlwb2ludCAibW9kdWxlOmZ1bmN0aW9uIiAtLSByZXF1aXJlZCBmb3IgZXZhbC5weSAvIHRoZSBqdWRnZQojIGUuZy4gIHBvbGljeT13YXJlaG91c2Vfc29ydC5pbF9wb2xpY3k6bG9hZF9kcF9yZ2IKcG9saWN5OiBudWxsCgojIE9wdGlvbmFsIGt3YXJncyBmb3J3YXJkZWQgdG8gdGhlIHBvbGljeSBsb2FkZXIgKGV2YWwtb25seSBrbm9iczsgdGhlIGp1ZGdlIHBhc3NlcyBub25lLCBzbwojIGFueXRoaW5nIHRoYXQgbXVzdCBob2xkIGF0IGp1ZGdpbmcgdGltZSBoYXMgdG8gbGl2ZSBpbiB0aGUgY2hlY2twb2ludCdzIGBjb25maWdgKS4KIyAgIHB5dGhvbiBldmFsLnB5IC4uLiArcG9saWN5X2t3YXJncy5hY3RfaG9yaXpvbj00ICtwb2xpY3lfa3dhcmdzLm51bV9pbmZlcmVuY2Vfc3RlcHM9MzIKcG9saWN5X2t3YXJnczoge30KCiMgZXZhbC5weSBleHRyYXMKcmVjb3JkX3ZpZGVvOiB0cnVlICAgICAgICAjIHNhdmUgb25lIHJvbGxvdXQgbXA0IGFmdGVyIHRoZSBtZXRyaWNzIChWdWxrYW4pOyBmYWxzZSA9IG51bWJlcnMgb25seQp2aWRlb19lbnZzOiAxICAgICAgICAgICAgICMgZW52cyBpbiB0aGUgdmlkZW8gKGZyYW1lcyBhcmUgYnVmZmVyZWQgaW4gUkFNIC0tIGtlZXAgc21hbGwgb24gQ29sYWIpCnJlc3VsdHNfZmlsZTogbnVsbCAgICAgICAgIyBpZiBzZXQsIGFwcGVuZCBvbmUgSlNPTiBsaW5lIHdpdGggdGhlIG1ldHJpY3MgKENvbGFiIGJvb2trZWVwaW5nKQo='}, 'conf/difficulty/easy.yaml': {'base_sha256': '06f70a4cbdc4ddacdbf21bba342494cc9bbcb435aeeabd04506c61dc46954df4', 'sha256': '06f70a4cbdc4ddacdbf21bba342494cc9bbcb435aeeabd04506c61dc46954df4', 'content_b64': 'IyBAcGFja2FnZSBfZ2xvYmFsXwojIEVhc3kgaXMgQUxXQVlTIGZ1bGx5IGZpeGVkOiBzYW1lIDIgcGFyY2VscyBpbiB0aGUgc2FtZSBwb3NpdGlvbnMgZXZlcnkgZXBpc29kZSwgYmlucwojIG5ldmVyIHN3YXAuIFRoaXMgaXMgdGhlIGd1YXJhbnRlZWQtc29sdmFibGUgYmFzZWxpbmUgbGV2ZWwuCmRpZmZpY3VsdHk6CiAgbmFtZTogZWFzeQogIG51bV9wYXJjZWxzOiAyCiAgZml4ZWRfcG9zZXM6IHRydWUKICBtYXhfZXBpc29kZV9zdGVwczogMjUwICAgIyBkZW1vcyBuZWVkIH4xMTUgc3RlcHMgKG1heCAxMTUpOyBwbGFjZW1lbnQgaXMgc3RpY2t5LCBzbyBnZW5lcm91cyBpcyBzYWZlCgpyYW5kb21pemF0aW9uOgogIHBhcmNlbF9wb3NlOgogICAgeHlfaml0dGVyOiBbMC4wLCAwLjBdCiAgICB5YXdfaml0dGVyOiBbMC4wLCAwLjBdCiAgYmluX3Bvc2l0aW9uOgogICAgc2lkZV9zd2FwX3Byb2I6IDAuMAogICAgeHlfaml0dGVyOiBbMC4wLCAwLjBdCg=='}, 'conf/difficulty/hard.yaml': {'base_sha256': '1e4a9eb5d0b0be1b5086fab37289cfbfae210ddab6a84bffaf0b5e1342fe0def', 'sha256': '1e4a9eb5d0b0be1b5086fab37289cfbfae210ddab6a84bffaf0b5e1342fe0def', 'content_b64': 'IyBAcGFja2FnZSBfZ2xvYmFsXwojIEhhcmQgPSBtZWRpdW0gd2l0aCBtb3JlIHBhcmNlbHMgKDYpIHBsdXMgdGhlIGJpbnMgb2NjYXNpb25hbGx5IHN3YXBwaW5nIHNpZGVzLCBzbyB0aGUgcG9saWN5CiMgbXVzdCByb3V0ZSBieSB0YWcgY29sb3VyIHJhdGhlciB0aGFuIG1lbW9yaXNpbmcgYSBmaXhlZCBzaWRlLiBPcmllbnRhdGlvbiBqaXR0ZXIgaXMgdmVyeQojIHNsaWdodCBhbmQgcG9zaXRpb25zIHZhcnkgb25seSBhIGxpdHRsZS4KZGlmZmljdWx0eToKICBuYW1lOiBoYXJkCiAgbnVtX3BhcmNlbHM6IDYKICBmaXhlZF9wb3NlczogZmFsc2UKICBtYXhfZXBpc29kZV9zdGVwczogODAwICAgIyBkZW1vcyBuZWVkIH4zODggc3RlcHMgKG1heCA3NzkpOyBwbGFjZW1lbnQgaXMgc3RpY2t5LCBzbyBnZW5lcm91cyBpcyBzYWZlCgpyYW5kb21pemF0aW9uOgogIHBhcmNlbF9wb3NlOgogICAgeHlfaml0dGVyOiBbLTAuMDIsIDAuMDJdICAgICAgICMgc21hbGwgc3Bhd24gaml0dGVyIHdpdGhpbiB0aGUgaW5ib3VuZCB6b25lIChtKQogICAgeWF3X2ppdHRlcjogWy0wLjEsIDAuMV0gICAgICAgICMgdmVyeSBzbGlnaHQgb3JpZW50YXRpb24gaml0dGVyIChyYWQsIHotb25seSB+NiBkZWcpCiAgYmluX3Bvc2l0aW9uOgogICAgc2lkZV9zd2FwX3Byb2I6IDAuNSAgICAgICAgICAgICMgcmVkL2JsdWUgYmlucyBzd2FwIHNpZGVzIGluIH5oYWxmIG9mIGVwaXNvZGVzCiAgICB4eV9qaXR0ZXI6IFswLjAsIDAuMF0K'}, 'conf/difficulty/medium.yaml': {'base_sha256': '0b25b513a50bf85f043ded73621f59ac59d611e54c56ef62906fad43ac5f9c87', 'sha256': '0b25b513a50bf85f043ded73621f59ac59d611e54c56ef62906fad43ac5f9c87', 'content_b64': 'IyBAcGFja2FnZSBfZ2xvYmFsXwpkaWZmaWN1bHR5OgogIG5hbWU6IG1lZGl1bQogIG51bV9wYXJjZWxzOiA0CiAgZml4ZWRfcG9zZXM6IGZhbHNlCiAgbWF4X2VwaXNvZGVfc3RlcHM6IDUwMCAgICMgZGVtb3MgbmVlZCB+MjM2IHN0ZXBzIChtYXggMjM4KTsgcGxhY2VtZW50IGlzIHN0aWNreSwgc28gZ2VuZXJvdXMgaXMgc2FmZQoKcmFuZG9taXphdGlvbjoKICBwYXJjZWxfcG9zZToKICAgIHh5X2ppdHRlcjogWy0wLjAxNSwgMC4wMTVdICAgICAjIHNtYWxsIHNwYXduIGppdHRlciB3aXRoaW4gdGhlIGluYm91bmQgem9uZSAobSkKICAgIHlhd19qaXR0ZXI6IFswLjAsIDAuMF0gICAgICAgICAjIGZpeGVkIG9yaWVudGF0aW9uIChib3hlcyBhbHdheXMgYXhpcy1hbGlnbmVkKQogIGJpbl9wb3NpdGlvbjoKICAgIHNpZGVfc3dhcF9wcm9iOiAwLjAKICAgIHh5X2ppdHRlcjogWzAuMCwgMC4wXQo='}, 'conf/eval/default.yaml': {'base_sha256': 'a37dc23ca54d90207205e054dd452adccbcb1272c45232e2228a0cafee4c7dcf', 'sha256': 'a37dc23ca54d90207205e054dd452adccbcb1272c45232e2228a0cafee4c7dcf', 'content_b64': 'ZXZhbDoKICBuX2VwaXNvZGVzOiA0CiAgc2VlZHM6IFs1MDAwLCA1MDAxLCA1MDAyLCA1MDAzLCA1MDA0LCA1MDA1LCA1MDA2LCA1MDA3LAogICAgICAgICAgNTAwOCwgNTAwOSwgNTAxMCwgNTAxMSwgNTAxMiwgNTAxMywgNTAxNCwgNTAxNV0K'}, 'conf/eval/eval32.yaml': {'base_sha256': 'fe2009cceaec85a7b69d81f42dd1e8d558f7cc7ae66bf75eb6aad2f8d0f37a22', 'sha256': 'fe2009cceaec85a7b69d81f42dd1e8d558f7cc7ae66bf75eb6aad2f8d0f37a22', 'content_b64': 'IyAzMiBlcGlzb2RlcyAoPSA0IGJhdGNoZXMgb2YgbnVtX2VudnM9OCkuIFNhbWUgc2VlZCBsaXN0IGFzIGRlZmF1bHQueWFtbCwgZXhwYW5kZWQgYnkgZXhwYW5kX3NlZWRzLgpldmFsOgogIG5fZXBpc29kZXM6IDMyCiAgc2VlZHM6IFs1MDAwLCA1MDAxLCA1MDAyLCA1MDAzLCA1MDA0LCA1MDA1LCA1MDA2LCA1MDA3LAogICAgICAgICAgNTAwOCwgNTAwOSwgNTAxMCwgNTAxMSwgNTAxMiwgNTAxMywgNTAxNCwgNTAxNV0K'}, 'conf/eval/eval64.yaml': {'base_sha256': 'a99c1cc7625fe4967eda0cfe3b7922b74bd027faf21e97c557dfb9535f54984c', 'sha256': 'a99c1cc7625fe4967eda0cfe3b7922b74bd027faf21e97c557dfb9535f54984c', 'content_b64': 'IyA2NCBlcGlzb2RlcyBmb3IgZGVjaXNpb24tZ3JhZGUgZWFzeSBldmFscyAoZWFzeSA9IDIgcGFyY2Vscy9lcGlzb2RlIC0+IDEyOCBwYXJjZWxzKS4KZXZhbDoKICBuX2VwaXNvZGVzOiA2NAogIHNlZWRzOiBbNTAwMCwgNTAwMSwgNTAwMiwgNTAwMywgNTAwNCwgNTAwNSwgNTAwNiwgNTAwNywKICAgICAgICAgIDUwMDgsIDUwMDksIDUwMTAsIDUwMTEsIDUwMTIsIDUwMTMsIDUwMTQsIDUwMTVdCg=='}, 'eval.py': {'base_sha256': '2b1577b4a4701f519792b6fd674851521d2ae30c434c6e5b24d11aa0a527b7a6', 'sha256': '2106aaa9a34cac05cd0fb756bade193de682898267101d9c4ef3f079bd63d574', 'content_b64': 'IiIiRXZhbHVhdGUgYSBjaGVja3BvaW50IG9uIGFuIGV2YWwgY29uZmlnLCByZXBvcnRpbmcgdGhlIMKnOS4xIG1ldHJpY3Mgb3ZlciBOIGVwaXNvZGVzLgpUaGUgaW50ZXJmYWNlIGhlcmUgaXMgSURFTlRJQ0FMIHRvIHRoZSBoZWxkLW91dCBqdWRnaW5nIGhhcm5lc3MuCgogIHB5dGhvbiBldmFsLnB5IGRpZmZpY3VsdHk9aGFyZCBjaGVja3BvaW50PTxwYXRoPiBldmFsX2NvbmZpZz1jb25mL2V2YWwvZGVmYXVsdC55YW1sCiAgIyBqdWRnZXMgcnVuIHRoZSBzYW1lIGNvbW1hbmQgd2l0aCB0aGUgaGVsZC1vdXQgY29uZmlnOgogIHB5dGhvbiBldmFsLnB5IGRpZmZpY3VsdHk9aGFyZCBjaGVja3BvaW50PTxwYXRoPiBldmFsX2NvbmZpZz1qdWRnZS9oZWxkb3V0LnlhbWwKCkNyaXRpY2FsIGJlaGF2aW91cjoKICAqIG9ic19tb2RlPXJnYiAoc2NlbmUgaW1hZ2UgKyBwcm9wcmlvY2VwdGlvbikgaXMgb3VyIHRyYWNrOyBzdGF0ZSBpcyBvcHRpb25hbCAocGVyLWxldmVsIGNrcHQpLgogICogRnVsbHkgZHJpdmVuIGJ5IHRoZSBgZXZhbF9jb25maWdgIGZpbGU6IGl0IHN1cHBsaWVzIG5fZXBpc29kZXMsIHRoZSBzZWVkIGxpc3QsIGFuZAogICAgKG9wdGlvbmFsbHkpIHJhbmRvbWlzYXRpb24tcmFuZ2UgT1ZFUlJJREVTLiBOb3RoaW5nIGFib3V0IHRoZSBldmFsIGNvbmRpdGlvbnMgaXMKICAgIGhhcmRjb2RlZCBoZXJlLgogICogQSB0cmFpbi5weSBjaGVja3BvaW50IGxvYWRzIGFuZCBydW5zIHdpdGggbm8gY29kZSBjaGFuZ2VzOyBkZWZhdWx0LnlhbWwgYW5kIHRoZSBoZWxkLW91dAogICAgY29uZmlnIHVzZSB0aGUgc2FtZSBwaXBlbGluZSAtLSBvbmx5IHRoZSByYW5kb21pc2F0aW9uIHZhbHVlcyBhbmQgc2VlZCBsaXN0IGRpZmZlci4KIiIiCgppbXBvcnQgb3MKaW1wb3J0IHRpbWUKCmltcG9ydCBoeWRyYQppbXBvcnQgdG9yY2gKZnJvbSBvbWVnYWNvbmYgaW1wb3J0IE9tZWdhQ29uZgoKZnJvbSB3YXJlaG91c2Vfc29ydC51dGlscyBpbXBvcnQgKAogICAgYXBwZW5kX2pzb25sLCBnaXRfaGFzaCwgbG9hZF9hZ2VudCwgbG9nX3J1bl9oZWFkZXIsIG1ha2VfZW52LCBwcmludF9tZXRyaWNzLCByZWNvcmRfZXZhbF92aWRlbywKICAgIHJvbGxvdXRfbWV0cmljcywKKQoKCmRlZiBfcG9saWN5X2t3YXJnc190b19jb250YWluZXIocG9saWN5X2t3YXJncyk6CiAgICBpZiBwb2xpY3lfa3dhcmdzIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIHt9CiAgICBpZiBPbWVnYUNvbmYuaXNfY29uZmlnKHBvbGljeV9rd2FyZ3MpOgogICAgICAgIHJldHVybiBPbWVnYUNvbmYudG9fY29udGFpbmVyKHBvbGljeV9rd2FyZ3MsIHJlc29sdmU9VHJ1ZSkgb3Ige30KICAgIHJldHVybiBkaWN0KHBvbGljeV9rd2FyZ3MpCgoKQGh5ZHJhLm1haW4odmVyc2lvbl9iYXNlPU5vbmUsIGNvbmZpZ19wYXRoPSJjb25mIiwgY29uZmlnX25hbWU9ImNvbmZpZyIpCmRlZiBtYWluKGNmZyk6CiAgICBhc3NlcnQgY2ZnLmNoZWNrcG9pbnQsICJwYXNzIGNoZWNrcG9pbnQ9PHBhdGggdG8gY2twdC5wdD4iCiAgICBhc3NlcnQgY2ZnLmdldCgiZXZhbF9jb25maWciKSwgInBhc3MgZXZhbF9jb25maWc9PHBhdGggdG8gZXZhbCB5YW1sPiIKICAgIGxvZ19ydW5faGVhZGVyKGNmZywgImV2YWwiKQogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKGNmZy5kZXZpY2UgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKQoKICAgIGV2YWxfY2ZnID0gT21lZ2FDb25mLmxvYWQoY2ZnLmV2YWxfY29uZmlnKQogICAgbl9lcGlzb2RlcyA9IGludChldmFsX2NmZy5ldmFsLm5fZXBpc29kZXMpCiAgICBzZWVkcyA9IGxpc3QoZXZhbF9jZmcuZXZhbC5zZWVkcykKICAgICMgcmFuZG9taXNhdGlvbjogdXNlIHRoZSBkaWZmaWN1bHR5J3MgdHJhaW5pbmcgcmFuZ2VzIHVubGVzcyB0aGUgZXZhbCBjb25maWcgb3ZlcnJpZGVzCiAgICAjIHRoZW0gKGhlbGQtb3V0IHdpZGVucy9yZWNvbWJpbmVzIHZpYSB0aGlzIG92ZXJyaWRlKS4KICAgIHJhbmRvbWl6YXRpb24gPSBldmFsX2NmZy5nZXQoInJhbmRvbWl6YXRpb24iLCBOb25lKSBvciBjZmcucmFuZG9taXphdGlvbgoKICAgIG9ic19tb2RlID0gY2ZnLm9ic19tb2RlCiAgICBwb2xpY3lfa3dhcmdzID0gX3BvbGljeV9rd2FyZ3NfdG9fY29udGFpbmVyKGNmZy5nZXQoInBvbGljeV9rd2FyZ3MiKSkKCiAgICBuX2VudnMgPSBtaW4oY2ZnLm51bV9lbnZzLCBuX2VwaXNvZGVzKQogICAgZW52LCBfID0gbWFrZV9lbnYoY2ZnLCBvYnNfbW9kZSwgcmFuZG9taXphdGlvbiwgbnVtX2VudnM9bl9lbnZzKQogICAgYWdlbnQsIF8gPSBsb2FkX2FnZW50KGNmZy5jaGVja3BvaW50LCBlbnYsIGRldmljZSwgZW50cnlwb2ludD1jZmcucG9saWN5LCBwb2xpY3lfa3dhcmdzPXBvbGljeV9rd2FyZ3MpCiAgICBpZiBnZXRhdHRyKGFnZW50LCAiY2ZnIiwgTm9uZSk6CiAgICAgICAgcHJpbnQoZiJbZXZhbF0gcG9saWN5IGNvbmZpZzoge2FnZW50LmNmZ30iLCBmbHVzaD1UcnVlKQoKICAgIHQwID0gdGltZS50aW1lKCkKICAgIG0gPSByb2xsb3V0X21ldHJpY3MoZW52LCBhZ2VudCwgZGV2aWNlLCBuX2VwaXNvZGVzLCBzZWVkcywgY2ZnLm1heF9lcGlzb2RlX3N0ZXBzKQogICAgbVsiZXZhbF9zZWNvbmRzIl0gPSByb3VuZCh0aW1lLnRpbWUoKSAtIHQwLCAxKQogICAgcHJpbnRfbWV0cmljcygiRVZBTCIsIGNmZy5kaWZmaWN1bHR5Lm5hbWUsIG9ic19tb2RlLCBtLAogICAgICAgICAgICAgICAgICBoYXJkPShjZmcuZGlmZmljdWx0eS5uYW1lID09ICJoYXJkIikpCiAgICBlbnYuY2xvc2UoKQoKICAgIGlmIGNmZy5nZXQoInJlc3VsdHNfZmlsZSIpOgogICAgICAgIGFwcGVuZF9qc29ubChjZmcucmVzdWx0c19maWxlLCBkaWN0KAogICAgICAgICAgICB0cz10aW1lLnN0cmZ0aW1lKCIlWS0lbS0lZCAlSDolTTolUyIpLCBnaXQ9Z2l0X2hhc2goKVs6OF0sIGxldmVsPWNmZy5kaWZmaWN1bHR5Lm5hbWUsCiAgICAgICAgICAgIG9ic19tb2RlPW9ic19tb2RlLCBjaGVja3BvaW50PWNmZy5jaGVja3BvaW50LCBwb2xpY3k9Y2ZnLnBvbGljeSwgcG9saWN5X2t3YXJncz1wb2xpY3lfa3dhcmdzLAogICAgICAgICAgICBldmFsX2NvbmZpZz1jZmcuZXZhbF9jb25maWcsIHJlcXVlc3RlZF9uX2VwaXNvZGVzPW5fZXBpc29kZXMsIHNlZWQwPWludChzZWVkc1swXSksCiAgICAgICAgICAgIG1heF9lcGlzb2RlX3N0ZXBzPWludChjZmcubWF4X2VwaXNvZGVfc3RlcHMpLCAqKm0pKQogICAgICAgIHByaW50KGYiW2V2YWxdIGFwcGVuZGVkIG1ldHJpY3MgLT4ge2NmZy5yZXN1bHRzX2ZpbGV9IiwgZmx1c2g9VHJ1ZSkKCiAgICAjIG9wdGlvbmFsbHkgc2F2ZSBhIHJvbGxvdXQgdmlkZW8gKFJlY29yZEVwaXNvZGUsIGFsbCB2aWV3czogcmVuZGVyICsgc2NlbmUgc2Vuc29yIGNhbSkuCiAgICAjIEZyYW1lcyBhcmUgYnVmZmVyZWQgaW4gc3lzdGVtIFJBTSB1bnRpbCB0aGUgZXBpc29kZSBlbmRzLCBzbyBrZWVwIHZpZGVvX2VudnMgc21hbGwgb24gQ29sYWIuCiAgICBpZiBub3QgY2ZnLmdldCgicmVjb3JkX3ZpZGVvIiwgVHJ1ZSk6CiAgICAgICAgcHJpbnQoIltldmFsXSByZWNvcmRfdmlkZW89ZmFsc2UgLT4gc2tpcHBpbmcgdmlkZW8iLCBmbHVzaD1UcnVlKQogICAgICAgIHJldHVybgogICAgb3V0X2RpciA9IGh5ZHJhLmNvcmUuaHlkcmFfY29uZmlnLkh5ZHJhQ29uZmlnLmdldCgpLnJ1bnRpbWUub3V0cHV0X2RpcgogICAgdmlkX2RpciA9IG9zLnBhdGguam9pbihvdXRfZGlyLCAidmlkZW9zIikKICAgIG5fdmlkID0gbWF4KDEsIG1pbihpbnQoY2ZnLmdldCgidmlkZW9fZW52cyIsIDEpKSwgbl9lbnZzKSkKICAgIHJlY29yZF9ldmFsX3ZpZGVvKGNmZywgb2JzX21vZGUsIHJhbmRvbWl6YXRpb24sIGFnZW50LCBkZXZpY2UsIHZpZF9kaXIsCiAgICAgICAgICAgICAgICAgICAgICBuX2VudnM9bl92aWQsIHNlZWQ9aW50KHNlZWRzWzBdKSkKICAgIHByaW50KGYiW2V2YWxdIHNhdmVkIHJvbGxvdXQgdmlkZW8gKHJlbmRlciArIHNlbnNvciB2aWV3cykgLT4ge3ZpZF9kaXJ9IiwgZmx1c2g9VHJ1ZSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=='}, 'examples/scripted_policy.py': {'base_sha256': 'c66028df0e4322bc29c4e0c2b52a67b180f1a826b8314c339d255b0c45c50cbd', 'sha256': 'c66028df0e4322bc29c4e0c2b52a67b180f1a826b8314c339d255b0c45c50cbd', 'content_b64': 'IiIiU2NyaXB0ZWQgcGljay1hbmQtcGxhY2UgcG9saWN5IGZvciBXYXJlaG91c2VTb3J0RW52LgoKQSBkZXRlcm1pbmlzdGljIG1vdGlvbi1wcmltaXRpdmUgc3RhdGUgbWFjaGluZSB0aGF0IHNvbHZlcyB0aGUgdGFzayB3aXRob3V0IGFueSBsZWFybmluZy4KUHVycG9zZTogcHJvdmUgdGhlIGVudmlyb25tZW50IGlzIHBoeXNpY2FsbHkgc29sdmFibGUgYmVmb3JlIHNwZW5kaW5nIEdQVSB0aW1lIG9uIFJMLgoKUGhhc2VzIChwZXIgcGFyY2VsLCBpbiBvcmRlcik6CiAgT1BFTiAgICAg4oaSIG9wZW4gZ3JpcHBlcgogIEFCT1ZFICAgIOKGkiBtb3ZlIFRDUCBkaXJlY3RseSBhYm92ZSBwYXJjZWwgYXQgc2FmZSBoZWlnaHQKICBERVNDRU5EICDihpIgbG93ZXIgb250byBwYXJjZWwgdG9wIGZhY2UKICBHUkFTUCAgICDihpIgY2xvc2UgZ3JpcHBlcgogIExJRlQgICAgIOKGkiByYWlzZSB0byBjYXJyeSBoZWlnaHQgKGNsZWFycyBiaW4gd2FsbHMpCiAgQ0FSUlkgICAg4oaSIG1vdmUgbGF0ZXJhbGx5IHRvIGFib3ZlIGNvcnJlY3QgYmluCiAgRFJPUCAgICAg4oaSIGxvd2VyIGludG8gYmluCiAgUkVMRUFTRSAg4oaSIG9wZW4gZ3JpcHBlciwgcmV0cmVhdAoKQWN0aW9uczogcGRfZWVfZGVsdGFfcG9zLCA0LWRpbSwgcmFuZ2UgWy0xLCAxXS4KICBkaW1zIDAtMjogZGVsdGEgeHl6IG9mIGVuZC1lZmZlY3RvciAoYWN0aW9uICogMC4xIG0vc3RlcCkKICBkaW0gIDMgIDogZ3JpcHBlciAoKzEgPSBvcGVuLCAtMSA9IGNsb3NlKQoKVXNhZ2U6CiAgcGl4aSBydW4gcHl0aG9uIGV4YW1wbGVzL3NjcmlwdGVkX3BvbGljeS5weQogICMgdmlkZW8gc2F2ZWQgdG8gb3V0cHV0cy9zY3JpcHRlZC92aWRlb3MvCiIiIgoKaW1wb3J0IG9zCmltcG9ydCBneW1uYXNpdW0gYXMgZ3ltCmltcG9ydCB0b3JjaAppbXBvcnQgd2FyZWhvdXNlX3NvcnQgICMgbm9xYSDigJQgcmVnaXN0ZXJzIFdhcmVob3VzZVNvcnQtdjEKCmZyb20gbWFuaV9za2lsbC51dGlscy53cmFwcGVycy5yZWNvcmQgaW1wb3J0IFJlY29yZEVwaXNvZGUKCiMg4pSA4pSAIGdlb21ldHJ5IGNvbnN0YW50cyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAgIwpTQ0FMRSAgPSAwLjEgICAgICMgbWV0cmVzIHBlciB1bml0IGFjdGlvbiAoY29udHJvbGxlciBwb3NfdXBwZXIpCkhPVkVSICAgPSAwLjIyICAgIyBzYWZlIHRyYW5zaXQgaGVpZ2h0IGFib3ZlIHRhYmxlIChjbGVhcnMgcGFyY2VscykKR1JBU1BfWiA9IDAuMDYwICAjIFRDUCB6IGF0IHBhcmNlbCB0b3AtZmFjZSBsZXZlbCAoZmluZ2VydGlwcyBhdCB+MC4wNjEsIGdyaXAgdXBwZXIgYm94IHNpZGVzKQpDQVJSWSAgID0gMC4yNiAgICMgY2FycnkgaGVpZ2h0IOKAlCBsaWZ0IGhpZ2ggc28gdGhlIGJveCBjbGVhcnMgYWxsIHBhcmNlbHMvYmluIHdhbGxzIHdoZW4gbW92aW5nCkRST1BfWiAgPSAwLjA4ICAgIyBUQ1AgeiBpbnNpZGUgYmluIChwYXJjZWwgc2V0dGxlcyB+MC4wMzEpClNQRUVEICAgPSAwLjcgICAgIyBtYXggYWN0aW9uIG1hZ25pdHVkZSBwZXIgc3RlcCAoZnJhY3Rpb24gb2YgMC4xbSkg4oCUIDAuNyDihpIgNyBjbS9zdGVwClRPTCAgICAgPSAwLjAxNSAgIyAiY2xvc2UgZW5vdWdoIiB0aHJlc2hvbGQgKG0pIGJlZm9yZSBhZHZhbmNpbmcgcGhhc2UKCiMg4pSA4pSAIHBoYXNlIGNvZGVzIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgCAjCk9QRU4sIEFCT1ZFLCBERVNDRU5ELCBHUkFTUCwgTElGVCwgQ0FSUllfUCwgRFJPUCwgUkVMRUFTRSA9IHJhbmdlKDgpClBIQVNFX05BTUVTID0gWyJPUEVOIiwgIkFCT1ZFIiwgIkRFU0NFTkQiLCAiR1JBU1AiLCAiTElGVCIsICJDQVJSWSIsICJEUk9QIiwgIlJFTEVBU0UiXQoKCmRlZiBfYWN0KGRlbHRhX3h5eiwgZ3JpcHBlcik6CiAgICAiIiJCdWlsZCBhICgxLCA0KSBhY3Rpb24gdGVuc29yIGZyb20gYSBudW1weSBhcnJheSBhbmQgc2NhbGFyIGdyaXBwZXIuIiIiCiAgICBkeCwgZHksIGR6ID0gKGZsb2F0KHYpIGZvciB2IGluIGRlbHRhX3h5eikKICAgIGcgPSBmbG9hdChncmlwcGVyKQogICAgcmV0dXJuIHRvcmNoLnRlbnNvcihbW2R4LCBkeSwgZHosIGddXSwgZHR5cGU9dG9yY2guZmxvYXQzMikKCgpkZWYgX21vdmUodGNwLCB0YXJnZXQsIGdyaXBwZXI9MS4wLCBzcGVlZD1TUEVFRCk6CiAgICAiIiJQcm9wb3J0aW9uYWwtY29udHJvbCBzdGVwIHRvd2FyZCB0YXJnZXQ7IGNsYW1wZWQgdG8gWy1zcGVlZCwgc3BlZWRdLiIiIgogICAgaW1wb3J0IG51bXB5IGFzIG5wCiAgICBkZWx0YSA9ICh0YXJnZXQgLSB0Y3ApLmNsaXAoLXNwZWVkLCBzcGVlZCkKICAgIHJldHVybiBfYWN0KGRlbHRhIC8gU0NBTEUsIGdyaXBwZXIpCgoKZGVmIF9hdCh0Y3AsIHRhcmdldCwgdG9sPVRPTCk6CiAgICBpbXBvcnQgbnVtcHkgYXMgbnAKICAgIHJldHVybiBmbG9hdCgoKHRjcCAtIHRhcmdldCkgKiogMikuc3VtKCkgKiogMC41KSA8IHRvbAoKCmRlZiBzY3JpcHRlZF9lcGlzb2RlKGVudiwgbWF4X3N0ZXBzPTMwMCwgc2VlZD00MiwgYWN0aW9uX25vaXNlPTAuMCwgcm5nPU5vbmUpOgogICAgIiIiUnVuIG9uZSBzY3JpcHRlZCBlcGlzb2RlIG9uIGEgc2luZ2xlLWVudiBXYXJlaG91c2VTb3J0RW52IHdyYXBwZXIuCgogICAgUmV0dXJucyBhIGxpc3Qgb2YgKG9icywgYWN0aW9uLCByZXdhcmQsIGluZm8pIHR1cGxlcy4gVGhlIGVwaXNvZGUgZW5kcyBhcyBzb29uIGFzIGV2ZXJ5CiAgICBwYXJjZWwgaXMgY29ycmVjdGx5IHNvcnRlZC4gYGBzZWVkYGAgaXMgZm9yd2FyZGVkIHRvIGBgZW52LnJlc2V0YGAuIGBgYWN0aW9uX25vaXNlYGAgKHN0ZCwKICAgIGluIGFjdGlvbiB1bml0cykgb3B0aW9uYWxseSBpbmplY3RzIEdhdXNzaWFuIG5vaXNlIGludG8gdGhlIHh5eiBhY3Rpb24gZGltczsgdGhlIHBvbGljeSBpcwogICAgY2xvc2VkLWxvb3Agc28gaXQgc2VsZi1jb3JyZWN0cy4gYGBybmdgYCAoYSBudW1weSBHZW5lcmF0b3IpIG1ha2VzIHRoYXQgbm9pc2UgcmVwcm9kdWNpYmxlLgogICAgIiIiCiAgICBpbXBvcnQgbnVtcHkgYXMgbnAKCiAgICBpZiBybmcgaXMgTm9uZToKICAgICAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZCkKICAgIGJhc2UgPSBlbnYudW53cmFwcGVkCiAgICBvYnMsIF8gPSBlbnYucmVzZXQoc2VlZD1zZWVkKQogICAgZGV2aWNlID0gImNwdSIKCiAgICBwaGFzZSAgICAgICA9IE9QRU4KICAgIHBhcmNlbF9pZHggID0gMAogICAgcGhhc2Vfc3RlcHMgPSAwICAgICAgICMgc3RlcHMgc3BlbnQgaW4gdGhlIGN1cnJlbnQgcGhhc2UgKHJlc2V0IG9uIGV2ZXJ5IHRyYW5zaXRpb24pCiAgICBncmFzcF90cmllcyA9IDAgICAgICAgIyBncmFzcCBhdHRlbXB0cyBvbiB0aGUgY3VycmVudCBwYXJjZWwgKGdpdmUgdXAgYWZ0ZXIgYSBmZXcpCiAgICBuX3BhcmNlbHMgICA9IGJhc2UubnVtX3BhcmNlbHMKICAgIGhpc3RvcnkgICAgID0gW10KCiAgICBkZWYgZ290byhwKToKICAgICAgICBub25sb2NhbCBwaGFzZSwgcGhhc2Vfc3RlcHMKICAgICAgICBwaGFzZSwgcGhhc2Vfc3RlcHMgPSBwLCAwCgogICAgIyBTcHJlYWQgZHJvcCBwb2ludHMgd2l0aGluIGEgYmluIHNvIG11bHRpcGxlIHNhbWUtY29sb3VyIHBhcmNlbHMgZG9uJ3Qgc3RhY2s6CiAgICAjIGVhY2ggcGFyY2VsIGdldHMgYSAic2xvdCIgaW5kZXggYW1vbmcgcGFyY2VscyBzaGFyaW5nIGl0cyB0YWcsIG9mZnNldCBhbG9uZyB0aGUKICAgICMgYmluJ3MgeC1heGlzIChmb290cHJpbnQgaGFsZi14ID0gMC4xMSwgc28gKy8tMC4wNzUga2VlcHMgYm94ZXMgaW5zaWRlKS4KICAgIHRhZ3MwID0gYmFzZS5wYXJjZWxfdGFnc1swXS5jcHUoKS5sb25nKCkudG9saXN0KCkKICAgIHNsb3QsIHBlcl90YWcgPSBbMF0gKiBuX3BhcmNlbHMsIHt9CiAgICBmb3IgaiBpbiByYW5nZShuX3BhcmNlbHMpOgogICAgICAgIHQgPSB0YWdzMFtqXQogICAgICAgIHNsb3Rbal0gPSBwZXJfdGFnLmdldCh0LCAwKQogICAgICAgIHBlcl90YWdbdF0gPSBzbG90W2pdICsgMQogICAgdGFnX3RvdGFsID0gZGljdChwZXJfdGFnKQoKICAgIGZvciBzdGVwIGluIHJhbmdlKG1heF9zdGVwcyk6CiAgICAgICAgcGhhc2Vfc3RlcHMgKz0gMQogICAgICAgIHRjcCAgPSBiYXNlLmFnZW50LnRjcF9wb3NlLnBbMF0uY3B1KCkubnVtcHkoKQogICAgICAgIGJpbnMgPSBiYXNlLl9iaW5fcG9zaXRpb25zKClbMF0uY3B1KCkubnVtcHkoKSAgICAgIyAoMiwgMykKICAgICAgICB0YWdzID0gYmFzZS5wYXJjZWxfdGFnc1swXS5jcHUoKS5sb25nKCkudG9saXN0KCkgICMgW3RhZ19wMCwgdGFnX3AxLCAuLi5dCgogICAgICAgIGlmIHBhcmNlbF9pZHggPj0gbl9wYXJjZWxzOgogICAgICAgICAgICBhY3Rpb24gPSBfYWN0KFswLCAwLCAwXSwgMS4wKSAgICMgaG9sZCBvcGVuOyBsZXQgdGhlIGxhc3QgYm94IHNldHRsZSBpbnRvIHRoZSBiaW4KICAgICAgICBlbHNlOgogICAgICAgICAgICBwX3BvcyA9IGJhc2UucGFyY2Vsc1twYXJjZWxfaWR4XS5wb3NlLnBbMF0uY3B1KCkubnVtcHkoKQogICAgICAgICAgICB0YWcgICA9IHRhZ3NbcGFyY2VsX2lkeF0KICAgICAgICAgICAgYmluX3h5eiA9IGJpbnNbdGFnXSAgICAjIGNvcnJlY3QgYmluIGZvciB0aGlzIHBhcmNlbAogICAgICAgICAgICAjIHNsb3Qgb2Zmc2V0IHdpdGhpbiB0aGUgYmluLCBzcHJlYWQgYWxvbmcgdGhlIGJpbidzIGRlZXBlciB5LWF4aXMgKGZvb3RwcmludAogICAgICAgICAgICAjIGhhbGYteSA9IDAuMTMsIHZzIGhhbGYteCA9IDAuMTEpIHNvIDMgc2FtZS1jb2xvdXIgYm94ZXMgZml0IHdpdGhvdXQgb25lIGxhbmRpbmcKICAgICAgICAgICAgIyBvbiB0aGUgcmltLiArLy0wLjA3IGtlZXBzIGVhY2ggYm94IChoYWxmIDAuMDI2KSB3ZWxsIGluc2lkZSB0aGUgZm9vdHByaW50LgogICAgICAgICAgICBvZmYgPSAoc2xvdFtwYXJjZWxfaWR4XSAtICh0YWdfdG90YWxbdGFnXSAtIDEpIC8gMi4wKSAqIDAuMDcKCiAgICAgICAgICAgIGFib3ZlX3AgID0gbnAuYXJyYXkoW3BfcG9zWzBdLCAgIHBfcG9zWzFdLCAgICAgICAgICBIT1ZFUl0pCiAgICAgICAgICAgIGdyYXNwX3AgID0gbnAuYXJyYXkoW3BfcG9zWzBdLCAgIHBfcG9zWzFdLCAgICAgICAgICBHUkFTUF9aXSkKICAgICAgICAgICAgY2FycnlfcCAgPSBucC5hcnJheShbYmluX3h5elswXSwgYmluX3h5elsxXSArIG9mZiwgIENBUlJZXSkKICAgICAgICAgICAgZHJvcF9wICAgPSBucC5hcnJheShbYmluX3h5elswXSwgYmluX3h5elsxXSArIG9mZiwgIERST1BfWl0pCgogICAgICAgICAgICBkZWYgYWR2YW5jZV9wYXJjZWwoKToKICAgICAgICAgICAgICAgIG5vbmxvY2FsIHBhcmNlbF9pZHgsIGdyYXNwX3RyaWVzCiAgICAgICAgICAgICAgICBwYXJjZWxfaWR4ICs9IDEKICAgICAgICAgICAgICAgIGdyYXNwX3RyaWVzID0gMAogICAgICAgICAgICAgICAgZ290byhBQk9WRSBpZiBwYXJjZWxfaWR4IDwgbl9wYXJjZWxzIGVsc2UgT1BFTikKCiAgICAgICAgICAgIGlmIHBoYXNlID09IE9QRU46CiAgICAgICAgICAgICAgICBhY3Rpb24gPSBfYWN0KFswLCAwLCAwXSwgMS4wKQogICAgICAgICAgICAgICAgaWYgcGhhc2Vfc3RlcHMgPj0gNDogICAgICAgICMgYnJpZWYgZ3JpcHBlci1vcGVuIHNldHRsZSwgdGhlbiBtb3ZlCiAgICAgICAgICAgICAgICAgICAgZ290byhBQk9WRSkKCiAgICAgICAgICAgIGVsaWYgcGhhc2UgPT0gQUJPVkU6CiAgICAgICAgICAgICAgICAjIEFsaWduIFZFUlkgdGlnaHRseSBpbiB4eSBhdCBob3ZlciBoZWlnaHQgYmVmb3JlIGRlc2NlbmRpbmcuIFdpdGggYSB+NWNtIGJveAogICAgICAgICAgICAgICAgIyByb3RhdGVkIGJ5IHVwIHRvIDAuNSByYWQsIHRoZSBvcGVuIGdyaXBwZXIgaGFzIG9ubHkgfjVtbSBjbGVhcmFuY2UgcGVyIHNpZGUsCiAgICAgICAgICAgICAgICAjIHNvIHRoZSBsYXRlcmFsIGVycm9yIG11c3QgYmUgc21hbGwgb3IgYSBmaW5nZXIgY2xpcHMgdGhlIGJveCBvbiB0aGUgd2F5IGRvd24uCiAgICAgICAgICAgICAgICBhY3Rpb24gPSBfbW92ZSh0Y3AsIGFib3ZlX3AsIGdyaXBwZXI9MS4wLCBzcGVlZD0wLjQpCiAgICAgICAgICAgICAgICBsYXQgPSBmbG9hdChucC5saW5hbGcubm9ybSh0Y3BbOjJdIC0gYWJvdmVfcFs6Ml0pKQogICAgICAgICAgICAgICAgaWYgKGxhdCA8IDAuMDA1IGFuZCBhYnModGNwWzJdIC0gSE9WRVIpIDwgMC4wNSkgb3IgcGhhc2Vfc3RlcHMgPiA3MDoKICAgICAgICAgICAgICAgICAgICBnb3RvKERFU0NFTkQpCgogICAgICAgICAgICBlbGlmIHBoYXNlID09IERFU0NFTkQ6CiAgICAgICAgICAgICAgICAjIFNsb3csIG5lYXItdmVydGljYWwgZGVzY2VudCAoeHkgYWxyZWFkeSBhbGlnbmVkKS4gRXhpdCBvbiBsb3cgeiB3aXRoIHh5IHN0aWxsCiAgICAgICAgICAgICAgICAjIHRpZ2h0OyBrZXlpbmcgb24geiAobm90IGEgc3RlcCBjb3VudCkgc3RvcHMgaXQgZ3Jhc3BpbmcgdG9vIGhpZ2guCiAgICAgICAgICAgICAgICBhY3Rpb24gPSBfbW92ZSh0Y3AsIGdyYXNwX3AsIGdyaXBwZXI9MS4wLCBzcGVlZD0wLjEyKQogICAgICAgICAgICAgICAgbGF0ID0gZmxvYXQobnAubGluYWxnLm5vcm0odGNwWzoyXSAtIGdyYXNwX3BbOjJdKSkKICAgICAgICAgICAgICAgIGlmICh0Y3BbMl0gPD0gR1JBU1BfWiArIDAuMDA4IGFuZCBsYXQgPCAwLjAwOCkgb3IgcGhhc2Vfc3RlcHMgPiA2MDoKICAgICAgICAgICAgICAgICAgICBnb3RvKEdSQVNQKQoKICAgICAgICAgICAgZWxpZiBwaGFzZSA9PSBHUkFTUDoKICAgICAgICAgICAgICAgIGFjdGlvbiA9IF9hY3QoWzAsIDAsIDBdLCAtMS4wKQogICAgICAgICAgICAgICAgaWYgcGhhc2Vfc3RlcHMgPj0gMTI6CiAgICAgICAgICAgICAgICAgICAgaWYgYmFzZS5hZ2VudC5pc19ncmFzcGluZyhiYXNlLnBhcmNlbHNbcGFyY2VsX2lkeF0pWzBdLml0ZW0oKToKICAgICAgICAgICAgICAgICAgICAgICAgZ290byhMSUZUKQogICAgICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgICAgIGdyYXNwX3RyaWVzICs9IDEKICAgICAgICAgICAgICAgICAgICAgICAgZ290byhBQk9WRSBpZiBncmFzcF90cmllcyA8IDMgZWxzZSBSRUxFQVNFKSAgIyBnaXZlIHVwIOKGkiBza2lwIHBhcmNlbAoKICAgICAgICAgICAgZWxpZiBwaGFzZSA9PSBMSUZUOgogICAgICAgICAgICAgICAgIyByaXNlIFNUUkFJR0hUIFVQIChubyBsYXRlcmFsIG1vdGlvbikgc28gdGhlIGNhcnJpZWQgYm94IGNsZWFycyBuZWlnaGJvdXJpbmcKICAgICAgICAgICAgICAgICMgcGFyY2VscyBhbmQgYmluIHdhbGxzIGJlZm9yZSB0aGUgbGF0ZXJhbCBjYXJyeSBiZWdpbnMuCiAgICAgICAgICAgICAgICBhY3Rpb24gPSBfYWN0KFswLCAwLCAxLjBdLCAtMS4wKQogICAgICAgICAgICAgICAgaWYgdGNwWzJdID4gQ0FSUlkgLSAwLjAxIG9yIHBoYXNlX3N0ZXBzID4gMzU6CiAgICAgICAgICAgICAgICAgICAgZ290byhDQVJSWV9QKQoKICAgICAgICAgICAgZWxpZiBwaGFzZSA9PSBDQVJSWV9QOgogICAgICAgICAgICAgICAgYWN0aW9uID0gX21vdmUodGNwLCBjYXJyeV9wLCBncmlwcGVyPS0xLjApCiAgICAgICAgICAgICAgICBpZiBfYXQodGNwLCBjYXJyeV9wLCB0b2w9MC4wNCkgb3IgcGhhc2Vfc3RlcHMgPiA0MDoKICAgICAgICAgICAgICAgICAgICBnb3RvKERST1ApCgogICAgICAgICAgICBlbGlmIHBoYXNlID09IERST1A6CiAgICAgICAgICAgICAgICBhY3Rpb24gPSBfbW92ZSh0Y3AsIGRyb3BfcCwgZ3JpcHBlcj0tMS4wLCBzcGVlZD0wLjMpCiAgICAgICAgICAgICAgICBpZiBfYXQodGNwLCBkcm9wX3AsIHRvbD0wLjAyNSkgb3IgcGhhc2Vfc3RlcHMgPiAzMDoKICAgICAgICAgICAgICAgICAgICBnb3RvKFJFTEVBU0UpCgogICAgICAgICAgICBlbGlmIHBoYXNlID09IFJFTEVBU0U6CiAgICAgICAgICAgICAgICBhY3Rpb24gPSBfYWN0KFswLCAwLCAwLjFdLCAxLjApICAgIyBvcGVuICsgc21hbGwgbGlmdAogICAgICAgICAgICAgICAgaWYgcGhhc2Vfc3RlcHMgPj0gNjogICAgICAgICAgICAgICMgcmVsZWFzZSBhbmQgbW92ZSBzdHJhaWdodCB0byB0aGUgbmV4dCBwYXJjZWwKICAgICAgICAgICAgICAgICAgICBhZHZhbmNlX3BhcmNlbCgpCgogICAgICAgIGlmIGFjdGlvbl9ub2lzZSA+IDAuMDoKICAgICAgICAgICAgbm9pc2UgPSB0b3JjaC5hc190ZW5zb3IoCiAgICAgICAgICAgICAgICBybmcubm9ybWFsKDAuMCwgYWN0aW9uX25vaXNlLCBzaXplPWFjdGlvbi5zaGFwZSksIGR0eXBlPWFjdGlvbi5kdHlwZQogICAgICAgICAgICApCiAgICAgICAgICAgICMga2VlcCB0aGUgZ3JpcHBlciBjb21tYW5kIChkaW0gMykgY3Jpc3A7IG9ubHkgcGVydHVyYiB0aGUgeHl6IGRlbHRhcyAoZGltcyAwLTIpCiAgICAgICAgICAgIG5vaXNlWzosIDNdID0gMC4wCiAgICAgICAgICAgIGFjdGlvbiA9IChhY3Rpb24gKyBub2lzZSkuY2xhbXAoLTEuMCwgMS4wKQogICAgICAgIG9icywgcmV3YXJkLCB0ZXJtLCB0cnVuYywgaW5mbyA9IGVudi5zdGVwKGFjdGlvbi50byhkZXZpY2UpKQogICAgICAgIGhpc3RvcnkuYXBwZW5kKChvYnMsIGFjdGlvbiwgZmxvYXQocmV3YXJkKSwgaW5mbykpCgogICAgICAgIHNjID0gaW5mby5nZXQoInN1Y2Nlc3NfY291bnQiLCBOb25lKQogICAgICAgIGlmIHNjIGlzIG5vdCBOb25lOgogICAgICAgICAgICBzY192YWwgPSBzYy5pdGVtKCkgaWYgaGFzYXR0cihzYywgIml0ZW0iKSBlbHNlIHNjCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2NfdmFsID0gIj8iCiAgICAgICAgaWYgc3RlcCAlIDIwID09IDAgb3Igc3RlcCA9PSBtYXhfc3RlcHMgLSAxOgogICAgICAgICAgICBwcmludChmIiAgc3RlcCB7c3RlcDozZH0gIHBoYXNlPXtQSEFTRV9OQU1FU1twaGFzZV0gaWYgcGFyY2VsX2lkeCA8IG5fcGFyY2VscyBlbHNlICdET05FJzo4c30iCiAgICAgICAgICAgICAgICAgIGYiICBwYXJjZWw9e3BhcmNlbF9pZHh9ICB0Y3A9e3RjcC5yb3VuZCgzKX0gIHNvcnRlZD17c2NfdmFsfSIsIGZsdXNoPVRydWUpCgogICAgICAgIGlmIHRydW5jIG9yIChpc2luc3RhbmNlKHNjX3ZhbCwgKGludCwgZmxvYXQpKSBhbmQgc2NfdmFsID49IG5fcGFyY2Vscyk6CiAgICAgICAgICAgIGJyZWFrCgogICAgcmV0dXJuIGhpc3RvcnkKCgojIERpZmZpY3VsdHkg4oaSIFdhcmVob3VzZVNvcnRFbnYga3dhcmdzLiBNVVNUIG1pcnJvciBjb25mL2RpZmZpY3VsdHkvKi55YW1sIHNvIHRoZSBkZW1vcyBhcmUKIyBkcmF3biBmcm9tIHRoZSBzYW1lIGRpc3RyaWJ1dGlvbiB0aGUgcG9saWN5IGlzIGV2YWx1YXRlZCBvbjoKIyAgIGVhc3kgICA9IDIgcGFyY2VscywgZnVsbHkgZml4ZWQgKHNhbWUgc2NlbmUgZXZlcnkgZXBpc29kZSkKIyAgIG1lZGl1bSA9IDQgcGFyY2VscywgZml4ZWQgb3JpZW50YXRpb24sIHNtYWxsIHBvc2l0aW9uIGppdHRlciwgYmlucyBmaXhlZAojICAgaGFyZCAgID0gNiBwYXJjZWxzLCB2ZXJ5IHNsaWdodCBvcmllbnRhdGlvbiBqaXR0ZXIsIHNtYWxsIHBvc2l0aW9uIGppdHRlciwgYmlucyBzd2FwIH5oYWxmCl9SQU5EX01FRElVTSA9IHsKICAgICJwYXJjZWxfcG9zZSI6ICB7Inh5X2ppdHRlciI6IFstMC4wMTUsIDAuMDE1XSwgInlhd19qaXR0ZXIiOiBbMC4wLCAwLjBdfSwKICAgICJiaW5fcG9zaXRpb24iOiB7InNpZGVfc3dhcF9wcm9iIjogMC4wLCAieHlfaml0dGVyIjogWzAuMCwgMC4wXX0sCn0KCiMgSGFyZDogYmlucyBzd2FwIHNpZGVzIGluIH5oYWxmIG9mIGVwaXNvZGVzIHNvIGRlbW9zIGNvdmVyIGJvdGggbGF5b3V0cy4gVGhlIHNjcmlwdGVkIHBvbGljeQojIHJlYWRzIGJpbiBwb3NpdGlvbnMgbGl2ZSwgc28gaXQgc29ydHMgY29ycmVjdGx5IHJlZ2FyZGxlc3Mgb2Ygd2hpY2ggc2lkZSBlYWNoIGJpbiBpcyBvbi4KX1JBTkRfSEFSRCA9IHsKICAgICJwYXJjZWxfcG9zZSI6ICB7Inh5X2ppdHRlciI6IFstMC4wMiwgMC4wMl0sICJ5YXdfaml0dGVyIjogWy0wLjEsIDAuMV19LAogICAgImJpbl9wb3NpdGlvbiI6IHsic2lkZV9zd2FwX3Byb2IiOiAwLjUsICJ4eV9qaXR0ZXIiOiBbMC4wLCAwLjBdfSwKfQoKIyBGdWxseS1maXhlZCBzY2VuZSAodXNlZCBieSBlYXN5KTogaWRlbnRpY2FsIGxheW91dCBldmVyeSBlcGlzb2RlLgpfUkFORF9OT05FID0gewogICAgInBhcmNlbF9wb3NlIjogIHsieHlfaml0dGVyIjogWzAuMCwgMC4wXSwgInlhd19qaXR0ZXIiOiBbMC4wLCAwLjBdfSwKICAgICJiaW5fcG9zaXRpb24iOiB7InNpZGVfc3dhcF9wcm9iIjogMC4wLCAieHlfaml0dGVyIjogWzAuMCwgMC4wXX0sCn0KCkRJRkZJQ1VMVFlfS1dBUkdTID0gewogICAgImVhc3kiOiAgIGRpY3QobnVtX3BhcmNlbHM9MiwgZml4ZWRfcG9zZXM9VHJ1ZSwgIHJhbmRvbWl6YXRpb249X1JBTkRfTk9ORSksCiAgICAibWVkaXVtIjogZGljdChudW1fcGFyY2Vscz00LCBmaXhlZF9wb3Nlcz1GYWxzZSwgcmFuZG9taXphdGlvbj1fUkFORF9NRURJVU0pLAogICAgImhhcmQiOiAgIGRpY3QobnVtX3BhcmNlbHM9NiwgZml4ZWRfcG9zZXM9RmFsc2UsIHJhbmRvbWl6YXRpb249X1JBTkRfSEFSRCksCn0KCgpkZWYgcnVuX2RpZmZpY3VsdHkoZGlmZmljdWx0eTogc3RyLCBzZWVkOiBpbnQgPSA0Mik6CiAgICBvdXRfZGlyID0gZiJvdXRwdXRzL3NjcmlwdGVkL3tkaWZmaWN1bHR5fSIKICAgIG9zLm1ha2VkaXJzKG91dF9kaXIsIGV4aXN0X29rPVRydWUpCgogICAga3dhcmdzID0gRElGRklDVUxUWV9LV0FSR1MuZ2V0KGRpZmZpY3VsdHksIERJRkZJQ1VMVFlfS1dBUkdTWyJlYXN5Il0pCiAgICBuX3BhcmNlbHMgPSBrd2FyZ3NbIm51bV9wYXJjZWxzIl0KICAgICMgYnVkZ2V0IH4xMDAgc3RlcHMgcGVyIHBhcmNlbCBmb3IgdGhlIGZ1bGwgbGlmdC1oaWdoIHBpY2stY2FycnktcGxhY2UgY3ljbGUgKCsgcmV0cnkpCiAgICBtYXhfc3RlcHMgPSBtYXgoMTUwLCAxMDAgKiBuX3BhcmNlbHMpCiAgICBlbnYgPSBneW0ubWFrZSgKICAgICAgICAiV2FyZWhvdXNlU29ydC12MSIsCiAgICAgICAgbnVtX2VudnM9MSwKICAgICAgICBvYnNfbW9kZT0ic3RhdGUiLAogICAgICAgIGNvbnRyb2xfbW9kZT0icGRfZWVfZGVsdGFfcG9zIiwKICAgICAgICBzaW1fYmFja2VuZD0iZ3B1IiwKICAgICAgICByZW5kZXJfbW9kZT0iYWxsIiwKICAgICAgICBtYXhfZXBpc29kZV9zdGVwcz1tYXhfc3RlcHMsCiAgICAgICAgKiprd2FyZ3MsCiAgICApCiAgICBlbnYgPSBSZWNvcmRFcGlzb2RlKAogICAgICAgIGVudiwKICAgICAgICBvdXRwdXRfZGlyPW91dF9kaXIsCiAgICAgICAgc2F2ZV90cmFqZWN0b3J5PUZhbHNlLAogICAgICAgIHNhdmVfdmlkZW89VHJ1ZSwKICAgICAgICB2aWRlb19mcHM9MjAsCiAgICAgICAgbWF4X3N0ZXBzX3Blcl92aWRlbz1tYXhfc3RlcHMsCiAgICApCgogICAgcHJpbnQoZiJcbj09PSBzY3JpcHRlZCBwb2xpY3kgIGRpZmZpY3VsdHk9e2RpZmZpY3VsdHl9ICAiCiAgICAgICAgICBmIih7bl9wYXJjZWxzfSBwYXJjZWxzKSAgc2VlZD17c2VlZH0gPT09IikKICAgIGhpc3RvcnkgPSBzY3JpcHRlZF9lcGlzb2RlKGVudiwgbWF4X3N0ZXBzPW1heF9zdGVwcykKICAgIGVudi5jbG9zZSgpCgogICAgZmluYWxfaW5mbyA9IGhpc3RvcnlbLTFdWy0xXQogICAgc2MgPSBmaW5hbF9pbmZvLmdldCgic3VjY2Vzc19jb3VudCIpCiAgICBzY192YWwgPSBzYy5pdGVtKCkgaWYgaGFzYXR0cihzYywgIml0ZW0iKSBlbHNlIHNjCiAgICBwcmludChmIkZpbmFsIHNvcnRlZDoge3NjX3ZhbH0gLyB7bl9wYXJjZWxzfSIpCiAgICBwcmludChmIlZpZGVvIOKGkiB7b3V0X2Rpcn0vMC5tcDQiKQogICAgcmV0dXJuIHNjX3ZhbAoKCmRlZiBtYWluKCk6CiAgICBpbXBvcnQgc3lzCiAgICBkaWZmaWN1bHRpZXMgPSBzeXMuYXJndlsxOl0gaWYgbGVuKHN5cy5hcmd2KSA+IDEgZWxzZSBbImVhc3kiLCAibWVkaXVtIiwgImhhcmQiXQogICAgZm9yIGQgaW4gZGlmZmljdWx0aWVzOgogICAgICAgIHJ1bl9kaWZmaWN1bHR5KGQpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='}, 'il/baselines/diffusion_policy/diffusion_policy/__init__.py': {'base_sha256': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'sha256': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'content_b64': ''}, 'il/baselines/diffusion_policy/diffusion_policy/augment.py': {'base_sha256': '8f54248c12df0d53c0c991a45ff7f0298ec46cb67b4be668c26fdd64c9e2a999', 'sha256': '8f54248c12df0d53c0c991a45ff7f0298ec46cb67b4be668c26fdd64c9e2a999', 'content_b64': 'IiIiVHJhaW5pbmctb25seSwgcGFyYW1ldGVyLWZyZWUgaW1hZ2UgYXVnbWVudGF0aW9uIGZvciB0aGUgUkdCIERpZmZ1c2lvbiBQb2xpY3kuIiIiCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKCgpjbGFzcyBSYW5kb21TaGlmdHNBdWcobm4uTW9kdWxlKToKICAgICIiIkRyUS12MiBzdHlsZSByYW5kb20gdHJhbnNsYXRpb246IHJlcGxpY2F0ZS1wYWQgYnkgYHBhZGAgcHggdGhlbiByYW5kb20tY3JvcCBiYWNrIHZpYQogICAgZ3JpZF9zYW1wbGUuIE1pbWljcyBzbWFsbCBjYW1lcmEvb2JqZWN0IGppdHRlciBzbyB0aGUgcG9saWN5IGdlbmVyYWxpc2VzIHRvIHRoZSBoZWxkLW91dCB3aWRlcgogICAgcG9zaXRpb24gcmFuZG9taXNhdGlvbi4gQ29sb3VyLXNhZmUgKG5vIGh1ZSBjaGFuZ2UpIGFuZCBwYXJhbWV0ZXItZnJlZSwgc28gYSBjaGVja3BvaW50IHRyYWluZWQKICAgIHdpdGggaXQgbG9hZHMgaW50byBhbiBBZ2VudCBidWlsdCB3aXRob3V0IGl0LiBBcHBsaWVkIGJ5IEFnZW50LmVuY29kZV9vYnMgb25seSB3aGVuIHRyYWluaW5nLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHBhZD00KToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnBhZCA9IGludChwYWQpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyB4OiAoTiwgQywgSCwgVykgZmxvYXQKICAgICAgICBuLCBjLCBoLCB3ID0geC5zaXplKCkKICAgICAgICBhc3NlcnQgaCA9PSB3LCAiUmFuZG9tU2hpZnRzQXVnIGV4cGVjdHMgc3F1YXJlIGltYWdlcyIKICAgICAgICBwYWRkaW5nID0gdHVwbGUoW3NlbGYucGFkXSAqIDQpCiAgICAgICAgeCA9IEYucGFkKHgsIHBhZGRpbmcsICJyZXBsaWNhdGUiKQogICAgICAgIGVwcyA9IDEuMCAvIChoICsgMiAqIHNlbGYucGFkKQogICAgICAgIGFyYW5nZSA9IHRvcmNoLmxpbnNwYWNlKC0xLjAgKyBlcHMsIDEuMCAtIGVwcywgaCArIDIgKiBzZWxmLnBhZCwgZGV2aWNlPXguZGV2aWNlLCBkdHlwZT14LmR0eXBlKVs6aF0KICAgICAgICBhcmFuZ2UgPSBhcmFuZ2UudW5zcXVlZXplKDApLnJlcGVhdChoLCAxKS51bnNxdWVlemUoMikKICAgICAgICBiYXNlX2dyaWQgPSB0b3JjaC5jYXQoW2FyYW5nZSwgYXJhbmdlLnRyYW5zcG9zZSgxLCAwKV0sIGRpbT0yKQogICAgICAgIGJhc2VfZ3JpZCA9IGJhc2VfZ3JpZC51bnNxdWVlemUoMCkucmVwZWF0KG4sIDEsIDEsIDEpCiAgICAgICAgc2hpZnQgPSB0b3JjaC5yYW5kaW50KDAsIDIgKiBzZWxmLnBhZCArIDEsIHNpemU9KG4sIDEsIDEsIDIpLCBkZXZpY2U9eC5kZXZpY2UsIGR0eXBlPXguZHR5cGUpCiAgICAgICAgc2hpZnQgKj0gMi4wIC8gKGggKyAyICogc2VsZi5wYWQpCiAgICAgICAgZ3JpZCA9IGJhc2VfZ3JpZCArIHNoaWZ0CiAgICAgICAgcmV0dXJuIEYuZ3JpZF9zYW1wbGUoeCwgZ3JpZCwgcGFkZGluZ19tb2RlPSJ6ZXJvcyIsIGFsaWduX2Nvcm5lcnM9RmFsc2UpCg=='}, 'il/baselines/diffusion_policy/diffusion_policy/conditional_unet1d.py': {'base_sha256': '96b81b1a0df65f1be566026ce2f15905cd24bead8b9c191973b046786d63d80f', 'sha256': '96b81b1a0df65f1be566026ce2f15905cd24bead8b9c191973b046786d63d80f', 'content_b64': 'I0BtYXJrZG93biAjIyMgKipOZXR3b3JrKioKI0BtYXJrZG93bgojQG1hcmtkb3duIERlZmluZXMgYSAxRCBVTmV0IGFyY2hpdGVjdHVyZSBgQ29uZGl0aW9uYWxVbmV0MURgCiNAbWFya2Rvd24gYXMgdGhlIG5vaWVzIHByZWRpY3Rpb24gbmV0d29yawojQG1hcmtkb3duCiNAbWFya2Rvd24gQ29tcG9uZW50cwojQG1hcmtkb3duIC0gYFNpbnVzb2lkYWxQb3NFbWJgIFBvc2l0aW9uYWwgZW5jb2RpbmcgZm9yIHRoZSBkaWZmdXNpb24gaXRlcmF0aW9uIGsKI0BtYXJrZG93biAtIGBEb3duc2FtcGxlMWRgIFN0cmlkZWQgY29udm9sdXRpb24gdG8gcmVkdWNlIHRlbXBvcmFsIHJlc29sdXRpb24KI0BtYXJrZG93biAtIGBVcHNhbXBsZTFkYCBUcmFuc3Bvc2VkIGNvbnZvbHV0aW9uIHRvIGluY3JlYXNlIHRlbXBvcmFsIHJlc29sdXRpb24KI0BtYXJrZG93biAtIGBDb252MWRCbG9ja2AgQ29udjFkIC0tPiBHcm91cE5vcm0gLS0+IE1pc2gKI0BtYXJrZG93biAtIGBDb25kaXRpb25hbFJlc2lkdWFsQmxvY2sxRGAgVGFrZXMgdHdvIGlucHV0cyBgeGAgYW5kIGBjb25kYC4gXAojQG1hcmtkb3duIGB4YCBpcyBwYXNzZWQgdGhyb3VnaCAyIGBDb252MWRCbG9ja2Agc3RhY2tlZCB0b2dldGhlciB3aXRoIHJlc2lkdWFsIGNvbm5lY3Rpb24uCiNAbWFya2Rvd24gYGNvbmRgIGlzIGFwcGxpZWQgdG8gYHhgIHdpdGggW0ZpTE1dKGh0dHBzOi8vYXJ4aXYub3JnL2Ficy8xNzA5LjA3ODcxKSBjb25kaXRpb25pbmcuCgoiIiIKTm90ZTogVGhpcyBpcyBjb3BpZWQgZnJvbSB0aGUgY29sYWIgbm90ZWJvb2suClRoZSBtYWluIGRpZmZlcmVuY2Ugd2l0aCB0aGUgZ2l0aHViIHJlcG8gY29kZSBpcyBpbiBgY2xhc3MgQ29uZGl0aW9uYWxVbmV0MURgIC0tIHRoaXMgdmVyc2lvbiBtYWtlcyBzb21lIHNpbXBsaWZpY2F0aW9ucy4KIiIiCgoKZnJvbSB0eXBpbmcgaW1wb3J0IFVuaW9uCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmltcG9ydCBtYXRoCgpjbGFzcyBTaW51c29pZGFsUG9zRW1iKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgZGltKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmRpbSA9IGRpbQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIGRldmljZSA9IHguZGV2aWNlCiAgICAgICAgaGFsZl9kaW0gPSBzZWxmLmRpbSAvLyAyCiAgICAgICAgZW1iID0gbWF0aC5sb2coMTAwMDApIC8gKGhhbGZfZGltIC0gMSkKICAgICAgICBlbWIgPSB0b3JjaC5leHAodG9yY2guYXJhbmdlKGhhbGZfZGltLCBkZXZpY2U9ZGV2aWNlKSAqIC1lbWIpCiAgICAgICAgZW1iID0geFs6LCBOb25lXSAqIGVtYltOb25lLCA6XQogICAgICAgIGVtYiA9IHRvcmNoLmNhdCgoZW1iLnNpbigpLCBlbWIuY29zKCkpLCBkaW09LTEpCiAgICAgICAgcmV0dXJuIGVtYgoKCmNsYXNzIERvd25zYW1wbGUxZChubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGRpbSk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5jb252ID0gbm4uQ29udjFkKGRpbSwgZGltLCAzLCAyLCAxKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIHJldHVybiBzZWxmLmNvbnYoeCkKCmNsYXNzIFVwc2FtcGxlMWQobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBkaW0pOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuY29udiA9IG5uLkNvbnZUcmFuc3Bvc2UxZChkaW0sIGRpbSwgNCwgMiwgMSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICByZXR1cm4gc2VsZi5jb252KHgpCgpjbGFzcyBDb252MWRCbG9jayhubi5Nb2R1bGUpOgogICAgJycnCiAgICAgICAgQ29udjFkIC0tPiBHcm91cE5vcm0gLS0+IE1pc2gKICAgICcnJwoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbnBfY2hhbm5lbHMsIG91dF9jaGFubmVscywga2VybmVsX3NpemUsIG5fZ3JvdXBzPTgpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQoKICAgICAgICBzZWxmLmJsb2NrID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uQ29udjFkKGlucF9jaGFubmVscywgb3V0X2NoYW5uZWxzLCBrZXJuZWxfc2l6ZSwgcGFkZGluZz1rZXJuZWxfc2l6ZSAvLyAyKSwKICAgICAgICAgICAgbm4uR3JvdXBOb3JtKG5fZ3JvdXBzLCBvdXRfY2hhbm5lbHMpLAogICAgICAgICAgICBubi5NaXNoKCksCiAgICAgICAgKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIHJldHVybiBzZWxmLmJsb2NrKHgpCgoKY2xhc3MgQ29uZGl0aW9uYWxSZXNpZHVhbEJsb2NrMUQobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLAogICAgICAgICAgICBpbl9jaGFubmVscywKICAgICAgICAgICAgb3V0X2NoYW5uZWxzLAogICAgICAgICAgICBjb25kX2RpbSwKICAgICAgICAgICAga2VybmVsX3NpemU9MywKICAgICAgICAgICAgbl9ncm91cHM9OCk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCgogICAgICAgIHNlbGYuYmxvY2tzID0gbm4uTW9kdWxlTGlzdChbCiAgICAgICAgICAgIENvbnYxZEJsb2NrKGluX2NoYW5uZWxzLCBvdXRfY2hhbm5lbHMsIGtlcm5lbF9zaXplLCBuX2dyb3Vwcz1uX2dyb3VwcyksCiAgICAgICAgICAgIENvbnYxZEJsb2NrKG91dF9jaGFubmVscywgb3V0X2NoYW5uZWxzLCBrZXJuZWxfc2l6ZSwgbl9ncm91cHM9bl9ncm91cHMpLAogICAgICAgIF0pCgogICAgICAgICMgRmlMTSBtb2R1bGF0aW9uIGh0dHBzOi8vYXJ4aXYub3JnL2Ficy8xNzA5LjA3ODcxCiAgICAgICAgIyBwcmVkaWN0cyBwZXItY2hhbm5lbCBzY2FsZSBhbmQgYmlhcwogICAgICAgIGNvbmRfY2hhbm5lbHMgPSBvdXRfY2hhbm5lbHMgKiAyCiAgICAgICAgc2VsZi5vdXRfY2hhbm5lbHMgPSBvdXRfY2hhbm5lbHMKICAgICAgICBzZWxmLmNvbmRfZW5jb2RlciA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLk1pc2goKSwKICAgICAgICAgICAgbm4uTGluZWFyKGNvbmRfZGltLCBjb25kX2NoYW5uZWxzKSwKICAgICAgICAgICAgbm4uVW5mbGF0dGVuKC0xLCAoLTEsIDEpKQogICAgICAgICkKCiAgICAgICAgIyBtYWtlIHN1cmUgZGltZW5zaW9ucyBjb21wYXRpYmxlCiAgICAgICAgc2VsZi5yZXNpZHVhbF9jb252ID0gbm4uQ29udjFkKGluX2NoYW5uZWxzLCBvdXRfY2hhbm5lbHMsIDEpIFwKICAgICAgICAgICAgaWYgaW5fY2hhbm5lbHMgIT0gb3V0X2NoYW5uZWxzIGVsc2Ugbm4uSWRlbnRpdHkoKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgsIGNvbmQpOgogICAgICAgICcnJwogICAgICAgICAgICB4IDogWyBiYXRjaF9zaXplIHggaW5fY2hhbm5lbHMgeCBob3Jpem9uIF0KICAgICAgICAgICAgY29uZCA6IFsgYmF0Y2hfc2l6ZSB4IGNvbmRfZGltXQoKICAgICAgICAgICAgcmV0dXJuczoKICAgICAgICAgICAgb3V0IDogWyBiYXRjaF9zaXplIHggb3V0X2NoYW5uZWxzIHggaG9yaXpvbiBdCiAgICAgICAgJycnCiAgICAgICAgb3V0ID0gc2VsZi5ibG9ja3NbMF0oeCkKICAgICAgICBlbWJlZCA9IHNlbGYuY29uZF9lbmNvZGVyKGNvbmQpCgogICAgICAgIGVtYmVkID0gZW1iZWQucmVzaGFwZSgKICAgICAgICAgICAgZW1iZWQuc2hhcGVbMF0sIDIsIHNlbGYub3V0X2NoYW5uZWxzLCAxKQogICAgICAgIHNjYWxlID0gZW1iZWRbOiwwLC4uLl0KICAgICAgICBiaWFzID0gZW1iZWRbOiwxLC4uLl0KICAgICAgICBvdXQgPSBzY2FsZSAqIG91dCArIGJpYXMKCiAgICAgICAgb3V0ID0gc2VsZi5ibG9ja3NbMV0ob3V0KQogICAgICAgIG91dCA9IG91dCArIHNlbGYucmVzaWR1YWxfY29udih4KQogICAgICAgIHJldHVybiBvdXQKCgpjbGFzcyBDb25kaXRpb25hbFVuZXQxRChubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsCiAgICAgICAgaW5wdXRfZGltLAogICAgICAgIGdsb2JhbF9jb25kX2RpbSwKICAgICAgICBkaWZmdXNpb25fc3RlcF9lbWJlZF9kaW09MjU2LAogICAgICAgIGRvd25fZGltcz1bMjU2LDUxMiwxMDI0XSwKICAgICAgICBrZXJuZWxfc2l6ZT01LAogICAgICAgIG5fZ3JvdXBzPTgKICAgICAgICApOgogICAgICAgICIiIgogICAgICAgIGlucHV0X2RpbTogRGltIG9mIGFjdGlvbnMuCiAgICAgICAgZ2xvYmFsX2NvbmRfZGltOiBEaW0gb2YgZ2xvYmFsIGNvbmRpdGlvbmluZyBhcHBsaWVkIHdpdGggRmlMTQogICAgICAgICAgaW4gYWRkaXRpb24gdG8gZGlmZnVzaW9uIHN0ZXAgZW1iZWRkaW5nLiBUaGlzIGlzIHVzdWFsbHkgb2JzX2hvcml6b24gKiBvYnNfZGltCiAgICAgICAgZGlmZnVzaW9uX3N0ZXBfZW1iZWRfZGltOiBTaXplIG9mIHBvc2l0aW9uYWwgZW5jb2RpbmcgZm9yIGRpZmZ1c2lvbiBpdGVyYXRpb24gawogICAgICAgIGRvd25fZGltczogQ2hhbm5lbCBzaXplIGZvciBlYWNoIFVOZXQgbGV2ZWwuCiAgICAgICAgICBUaGUgbGVuZ3RoIG9mIHRoaXMgYXJyYXkgZGV0ZXJtaW5lcyBudW1lYnIgb2YgbGV2ZWxzLgogICAgICAgIGtlcm5lbF9zaXplOiBDb252IGtlcm5lbCBzaXplCiAgICAgICAgbl9ncm91cHM6IE51bWJlciBvZiBncm91cHMgZm9yIEdyb3VwTm9ybQogICAgICAgICIiIgoKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBhbGxfZGltcyA9IFtpbnB1dF9kaW1dICsgbGlzdChkb3duX2RpbXMpCiAgICAgICAgc3RhcnRfZGltID0gZG93bl9kaW1zWzBdCgogICAgICAgIGRzZWQgPSBkaWZmdXNpb25fc3RlcF9lbWJlZF9kaW0KICAgICAgICBkaWZmdXNpb25fc3RlcF9lbmNvZGVyID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgU2ludXNvaWRhbFBvc0VtYihkc2VkKSwKICAgICAgICAgICAgbm4uTGluZWFyKGRzZWQsIGRzZWQgKiA0KSwKICAgICAgICAgICAgbm4uTWlzaCgpLAogICAgICAgICAgICBubi5MaW5lYXIoZHNlZCAqIDQsIGRzZWQpLAogICAgICAgICkKICAgICAgICBjb25kX2RpbSA9IGRzZWQgKyBnbG9iYWxfY29uZF9kaW0KCiAgICAgICAgaW5fb3V0ID0gbGlzdCh6aXAoYWxsX2RpbXNbOi0xXSwgYWxsX2RpbXNbMTpdKSkKICAgICAgICBtaWRfZGltID0gYWxsX2RpbXNbLTFdCiAgICAgICAgc2VsZi5taWRfbW9kdWxlcyA9IG5uLk1vZHVsZUxpc3QoWwogICAgICAgICAgICBDb25kaXRpb25hbFJlc2lkdWFsQmxvY2sxRCgKICAgICAgICAgICAgICAgIG1pZF9kaW0sIG1pZF9kaW0sIGNvbmRfZGltPWNvbmRfZGltLAogICAgICAgICAgICAgICAga2VybmVsX3NpemU9a2VybmVsX3NpemUsIG5fZ3JvdXBzPW5fZ3JvdXBzCiAgICAgICAgICAgICksCiAgICAgICAgICAgIENvbmRpdGlvbmFsUmVzaWR1YWxCbG9jazFEKAogICAgICAgICAgICAgICAgbWlkX2RpbSwgbWlkX2RpbSwgY29uZF9kaW09Y29uZF9kaW0sCiAgICAgICAgICAgICAgICBrZXJuZWxfc2l6ZT1rZXJuZWxfc2l6ZSwgbl9ncm91cHM9bl9ncm91cHMKICAgICAgICAgICAgKSwKICAgICAgICBdKQoKICAgICAgICBkb3duX21vZHVsZXMgPSBubi5Nb2R1bGVMaXN0KFtdKQogICAgICAgIGZvciBpbmQsIChkaW1faW4sIGRpbV9vdXQpIGluIGVudW1lcmF0ZShpbl9vdXQpOgogICAgICAgICAgICBpc19sYXN0ID0gaW5kID49IChsZW4oaW5fb3V0KSAtIDEpCiAgICAgICAgICAgIGRvd25fbW9kdWxlcy5hcHBlbmQobm4uTW9kdWxlTGlzdChbCiAgICAgICAgICAgICAgICBDb25kaXRpb25hbFJlc2lkdWFsQmxvY2sxRCgKICAgICAgICAgICAgICAgICAgICBkaW1faW4sIGRpbV9vdXQsIGNvbmRfZGltPWNvbmRfZGltLAogICAgICAgICAgICAgICAgICAgIGtlcm5lbF9zaXplPWtlcm5lbF9zaXplLCBuX2dyb3Vwcz1uX2dyb3VwcyksCiAgICAgICAgICAgICAgICBDb25kaXRpb25hbFJlc2lkdWFsQmxvY2sxRCgKICAgICAgICAgICAgICAgICAgICBkaW1fb3V0LCBkaW1fb3V0LCBjb25kX2RpbT1jb25kX2RpbSwKICAgICAgICAgICAgICAgICAgICBrZXJuZWxfc2l6ZT1rZXJuZWxfc2l6ZSwgbl9ncm91cHM9bl9ncm91cHMpLAogICAgICAgICAgICAgICAgRG93bnNhbXBsZTFkKGRpbV9vdXQpIGlmIG5vdCBpc19sYXN0IGVsc2Ugbm4uSWRlbnRpdHkoKQogICAgICAgICAgICBdKSkKCiAgICAgICAgdXBfbW9kdWxlcyA9IG5uLk1vZHVsZUxpc3QoW10pCiAgICAgICAgZm9yIGluZCwgKGRpbV9pbiwgZGltX291dCkgaW4gZW51bWVyYXRlKHJldmVyc2VkKGluX291dFsxOl0pKToKICAgICAgICAgICAgaXNfbGFzdCA9IGluZCA+PSAobGVuKGluX291dCkgLSAxKQogICAgICAgICAgICB1cF9tb2R1bGVzLmFwcGVuZChubi5Nb2R1bGVMaXN0KFsKICAgICAgICAgICAgICAgIENvbmRpdGlvbmFsUmVzaWR1YWxCbG9jazFEKAogICAgICAgICAgICAgICAgICAgIGRpbV9vdXQqMiwgZGltX2luLCBjb25kX2RpbT1jb25kX2RpbSwKICAgICAgICAgICAgICAgICAgICBrZXJuZWxfc2l6ZT1rZXJuZWxfc2l6ZSwgbl9ncm91cHM9bl9ncm91cHMpLAogICAgICAgICAgICAgICAgQ29uZGl0aW9uYWxSZXNpZHVhbEJsb2NrMUQoCiAgICAgICAgICAgICAgICAgICAgZGltX2luLCBkaW1faW4sIGNvbmRfZGltPWNvbmRfZGltLAogICAgICAgICAgICAgICAgICAgIGtlcm5lbF9zaXplPWtlcm5lbF9zaXplLCBuX2dyb3Vwcz1uX2dyb3VwcyksCiAgICAgICAgICAgICAgICBVcHNhbXBsZTFkKGRpbV9pbikgaWYgbm90IGlzX2xhc3QgZWxzZSBubi5JZGVudGl0eSgpCiAgICAgICAgICAgIF0pKQoKICAgICAgICBmaW5hbF9jb252ID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgQ29udjFkQmxvY2soc3RhcnRfZGltLCBzdGFydF9kaW0sIGtlcm5lbF9zaXplPWtlcm5lbF9zaXplKSwKICAgICAgICAgICAgbm4uQ29udjFkKHN0YXJ0X2RpbSwgaW5wdXRfZGltLCAxKSwKICAgICAgICApCgogICAgICAgIHNlbGYuZGlmZnVzaW9uX3N0ZXBfZW5jb2RlciA9IGRpZmZ1c2lvbl9zdGVwX2VuY29kZXIKICAgICAgICBzZWxmLnVwX21vZHVsZXMgPSB1cF9tb2R1bGVzCiAgICAgICAgc2VsZi5kb3duX21vZHVsZXMgPSBkb3duX21vZHVsZXMKICAgICAgICBzZWxmLmZpbmFsX2NvbnYgPSBmaW5hbF9jb252CgogICAgICAgIG5fcGFyYW1zID0gc3VtKHAubnVtZWwoKSBmb3IgcCBpbiBzZWxmLnBhcmFtZXRlcnMoKSkKICAgICAgICBwcmludChmIm51bWJlciBvZiBwYXJhbWV0ZXJzOiB7bl9wYXJhbXMgLyAxZTY6LjJmfU0iKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsCiAgICAgICAgICAgIHNhbXBsZTogdG9yY2guVGVuc29yLAogICAgICAgICAgICB0aW1lc3RlcDogVW5pb25bdG9yY2guVGVuc29yLCBmbG9hdCwgaW50XSwKICAgICAgICAgICAgZ2xvYmFsX2NvbmQ9Tm9uZSk6CiAgICAgICAgIiIiCiAgICAgICAgeDogKEIsVCxpbnB1dF9kaW0pCiAgICAgICAgdGltZXN0ZXA6IChCLCkgb3IgaW50LCBkaWZmdXNpb24gc3RlcAogICAgICAgIGdsb2JhbF9jb25kOiAoQixnbG9iYWxfY29uZF9kaW0pCiAgICAgICAgb3V0cHV0OiAoQixULGlucHV0X2RpbSkKICAgICAgICAiIiIKICAgICAgICAjIChCLFQsQykKICAgICAgICBzYW1wbGUgPSBzYW1wbGUubW92ZWF4aXMoLTEsLTIpCiAgICAgICAgIyAoQixDLFQpCgogICAgICAgICMgMS4gdGltZQogICAgICAgIHRpbWVzdGVwcyA9IHRpbWVzdGVwCiAgICAgICAgaWYgbm90IHRvcmNoLmlzX3RlbnNvcih0aW1lc3RlcHMpOgogICAgICAgICAgICAjIFRPRE86IHRoaXMgcmVxdWlyZXMgc3luYyBiZXR3ZWVuIENQVSBhbmQgR1BVLiBTbyB0cnkgdG8gcGFzcyB0aW1lc3RlcHMgYXMgdGVuc29ycyBpZiB5b3UgY2FuCiAgICAgICAgICAgIHRpbWVzdGVwcyA9IHRvcmNoLnRlbnNvcihbdGltZXN0ZXBzXSwgZHR5cGU9dG9yY2gubG9uZywgZGV2aWNlPXNhbXBsZS5kZXZpY2UpCiAgICAgICAgZWxpZiB0b3JjaC5pc190ZW5zb3IodGltZXN0ZXBzKSBhbmQgbGVuKHRpbWVzdGVwcy5zaGFwZSkgPT0gMDoKICAgICAgICAgICAgdGltZXN0ZXBzID0gdGltZXN0ZXBzW05vbmVdLnRvKHNhbXBsZS5kZXZpY2UpCiAgICAgICAgIyBicm9hZGNhc3QgdG8gYmF0Y2ggZGltZW5zaW9uIGluIGEgd2F5IHRoYXQncyBjb21wYXRpYmxlIHdpdGggT05OWC9Db3JlIE1MCiAgICAgICAgdGltZXN0ZXBzID0gdGltZXN0ZXBzLmV4cGFuZChzYW1wbGUuc2hhcGVbMF0pCgogICAgICAgIGdsb2JhbF9mZWF0dXJlID0gc2VsZi5kaWZmdXNpb25fc3RlcF9lbmNvZGVyKHRpbWVzdGVwcykKCiAgICAgICAgaWYgZ2xvYmFsX2NvbmQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGdsb2JhbF9mZWF0dXJlID0gdG9yY2guY2F0KFsKICAgICAgICAgICAgICAgIGdsb2JhbF9mZWF0dXJlLCBnbG9iYWxfY29uZAogICAgICAgICAgICBdLCBheGlzPS0xKQoKICAgICAgICB4ID0gc2FtcGxlCiAgICAgICAgaCA9IFtdCiAgICAgICAgZm9yIGlkeCwgKHJlc25ldCwgcmVzbmV0MiwgZG93bnNhbXBsZSkgaW4gZW51bWVyYXRlKHNlbGYuZG93bl9tb2R1bGVzKToKICAgICAgICAgICAgeCA9IHJlc25ldCh4LCBnbG9iYWxfZmVhdHVyZSkKICAgICAgICAgICAgeCA9IHJlc25ldDIoeCwgZ2xvYmFsX2ZlYXR1cmUpCiAgICAgICAgICAgIGguYXBwZW5kKHgpCiAgICAgICAgICAgIHggPSBkb3duc2FtcGxlKHgpCgogICAgICAgIGZvciBtaWRfbW9kdWxlIGluIHNlbGYubWlkX21vZHVsZXM6CiAgICAgICAgICAgIHggPSBtaWRfbW9kdWxlKHgsIGdsb2JhbF9mZWF0dXJlKQoKICAgICAgICBmb3IgaWR4LCAocmVzbmV0LCByZXNuZXQyLCB1cHNhbXBsZSkgaW4gZW51bWVyYXRlKHNlbGYudXBfbW9kdWxlcyk6CiAgICAgICAgICAgIHggPSB0b3JjaC5jYXQoKHgsIGgucG9wKCkpLCBkaW09MSkKICAgICAgICAgICAgeCA9IHJlc25ldCh4LCBnbG9iYWxfZmVhdHVyZSkKICAgICAgICAgICAgeCA9IHJlc25ldDIoeCwgZ2xvYmFsX2ZlYXR1cmUpCiAgICAgICAgICAgIHggPSB1cHNhbXBsZSh4KQoKICAgICAgICB4ID0gc2VsZi5maW5hbF9jb252KHgpCgogICAgICAgICMgKEIsQyxUKQogICAgICAgIHggPSB4Lm1vdmVheGlzKC0xLC0yKQogICAgICAgICMgKEIsVCxDKQogICAgICAgIHJldHVybiB4Cg=='}, 'il/baselines/diffusion_policy/diffusion_policy/evaluate.py': {'base_sha256': 'e6bf3f01e6ea17c97f778c65f6dc916b77fbee86c5d7840ed0247ced8c6c89d9', 'sha256': 'da5dc9eca6bc95211aad68cef5299ab5462f65614813994e12f9aaf4c0149ed5', 'content_b64': 'ZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgZGVmYXVsdGRpY3QKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIHRxZG0gaW1wb3J0IHRxZG0KCgpkZWYgX2VwaXNvZGVfdmVjdG9yKHZhbHVlKToKICAgIGlmIGlzaW5zdGFuY2UodmFsdWUsIHRvcmNoLlRlbnNvcik6CiAgICAgICAgdmFsdWUgPSB2YWx1ZS5kZXRhY2goKS5mbG9hdCgpLmNwdSgpLm51bXB5KCkKICAgIHJldHVybiBucC5hc2FycmF5KHZhbHVlKS5yZXNoYXBlKC0xKQoKCmRlZiBldmFsdWF0ZShuOiBpbnQsIGFnZW50LCBldmFsX2VudnMsIGRldmljZSwgc2ltX2JhY2tlbmQ6IHN0ciwgcHJvZ3Jlc3NfYmFyOiBib29sID0gVHJ1ZSk6CiAgICBmcm9tIG1hbmlfc2tpbGwudXRpbHMgaW1wb3J0IGNvbW1vbiAgIyBsYXp5OiBrZWVwcyB0aGlzIG1vZHVsZSBpbXBvcnRhYmxlIHdpdGhvdXQgdGhlIHNpbXVsYXRvcgoKICAgIHdhc190cmFpbmluZyA9IGdldGF0dHIoYWdlbnQsICJ0cmFpbmluZyIsIE5vbmUpCiAgICBhZ2VudC5ldmFsKCkKICAgIHBiYXIgPSB0cWRtKHRvdGFsPW4pIGlmIHByb2dyZXNzX2JhciBlbHNlIE5vbmUKICAgIHRyeToKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgZXZhbF9tZXRyaWNzID0gZGVmYXVsdGRpY3QobGlzdCkKICAgICAgICAgICAgb2JzLCBpbmZvID0gZXZhbF9lbnZzLnJlc2V0KCkKICAgICAgICAgICAgZXBzX2NvdW50ID0gMAogICAgICAgICAgICB3aGlsZSBlcHNfY291bnQgPCBuOgogICAgICAgICAgICAgICAgb2JzID0gY29tbW9uLnRvX3RlbnNvcihvYnMsIGRldmljZSkKICAgICAgICAgICAgICAgIGFjdGlvbl9zZXEgPSBhZ2VudC5nZXRfYWN0aW9uKG9icykKICAgICAgICAgICAgICAgIGlmIHNpbV9iYWNrZW5kID09ICJwaHlzeF9jcHUiOgogICAgICAgICAgICAgICAgICAgIGFjdGlvbl9zZXEgPSBhY3Rpb25fc2VxLmNwdSgpLm51bXB5KCkKICAgICAgICAgICAgICAgIGZvciBpIGluIHJhbmdlKGFjdGlvbl9zZXEuc2hhcGVbMV0pOgogICAgICAgICAgICAgICAgICAgIG9icywgcmV3LCB0ZXJtaW5hdGVkLCB0cnVuY2F0ZWQsIGluZm8gPSBldmFsX2VudnMuc3RlcChhY3Rpb25fc2VxWzosIGldKQogICAgICAgICAgICAgICAgICAgIGlmIHRydW5jYXRlZC5hbnkoKToKICAgICAgICAgICAgICAgICAgICAgICAgYnJlYWsKCiAgICAgICAgICAgICAgICBpZiB0cnVuY2F0ZWQuYW55KCk6CiAgICAgICAgICAgICAgICAgICAgYXNzZXJ0IHRydW5jYXRlZC5hbGwoKSA9PSB0cnVuY2F0ZWQuYW55KCksICJhbGwgZXBpc29kZXMgc2hvdWxkIHRydW5jYXRlIGF0IHRoZSBzYW1lIHRpbWUgZm9yIGZhaXIgZXZhbHVhdGlvbiB3aXRoIG90aGVyIGFsZ29yaXRobXMiCiAgICAgICAgICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShpbmZvWyJmaW5hbF9pbmZvIl0sIGRpY3QpOgogICAgICAgICAgICAgICAgICAgICAgICBmb3IgaywgdiBpbiBpbmZvWyJmaW5hbF9pbmZvIl1bImVwaXNvZGUiXS5pdGVtcygpOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgZXZhbF9tZXRyaWNzW2tdLmFwcGVuZChfZXBpc29kZV92ZWN0b3IodikpCiAgICAgICAgICAgICAgICAgICAgICAgICMgYWxzbyBsb2cgc29ydF9hY2N1cmFjeSAoZnJhY3Rpb24gb2YgcGFyY2VscyBzb3J0ZWQgYnkgdGhlIHRpbWUgbGltaXQpIC0tIGEKICAgICAgICAgICAgICAgICAgICAgICAgIyBwYXJ0aWFsLWNyZWRpdCB0YXNrLXByb2dyZXNzIG1ldHJpYywgYmV0dGVyIHRoYW4gZnVsbC1lcGlzb2RlIHN1Y2Nlc3MgYWxvbmUuCiAgICAgICAgICAgICAgICAgICAgICAgIGlmICJzb3J0X2FjY3VyYWN5IiBpbiBpbmZvWyJmaW5hbF9pbmZvIl06CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBldmFsX21ldHJpY3NbInNvcnRfYWNjdXJhY3kiXS5hcHBlbmQoX2VwaXNvZGVfdmVjdG9yKGluZm9bImZpbmFsX2luZm8iXVsic29ydF9hY2N1cmFjeSJdKSkKICAgICAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgICAgICBmb3IgZmluYWxfaW5mbyBpbiBpbmZvWyJmaW5hbF9pbmZvIl06CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3IgaywgdiBpbiBmaW5hbF9pbmZvWyJlcGlzb2RlIl0uaXRlbXMoKToKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBldmFsX21ldHJpY3Nba10uYXBwZW5kKF9lcGlzb2RlX3ZlY3Rvcih2KSkKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGlmICJzb3J0X2FjY3VyYWN5IiBpbiBmaW5hbF9pbmZvOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGV2YWxfbWV0cmljc1sic29ydF9hY2N1cmFjeSJdLmFwcGVuZChfZXBpc29kZV92ZWN0b3IoZmluYWxfaW5mb1sic29ydF9hY2N1cmFjeSJdKSkKICAgICAgICAgICAgICAgICAgICB0YWtlID0gbWluKGV2YWxfZW52cy5udW1fZW52cywgbiAtIGVwc19jb3VudCkKICAgICAgICAgICAgICAgICAgICBlcHNfY291bnQgKz0gdGFrZQogICAgICAgICAgICAgICAgICAgIGlmIHBiYXIgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgICAgIHBiYXIudXBkYXRlKHRha2UpCiAgICAgICAgZm9yIGssIHZhbHVlcyBpbiBldmFsX21ldHJpY3MuaXRlbXMoKToKICAgICAgICAgICAgZXZhbF9tZXRyaWNzW2tdID0gbnAuY29uY2F0ZW5hdGUodmFsdWVzLCBheGlzPTApWzpuXQogICAgICAgIHJldHVybiBldmFsX21ldHJpY3MKICAgIGZpbmFsbHk6CiAgICAgICAgaWYgcGJhciBpcyBub3QgTm9uZToKICAgICAgICAgICAgcGJhci5jbG9zZSgpCiAgICAgICAgaWYgd2FzX3RyYWluaW5nIGlzIEZhbHNlOgogICAgICAgICAgICBhZ2VudC5ldmFsKCkKICAgICAgICBlbHNlOgogICAgICAgICAgICBhZ2VudC50cmFpbigpCg=='}, 'il/baselines/diffusion_policy/diffusion_policy/lerobot_encoder.py': {'base_sha256': 'dfab31d866a77d0fa24ce904ec59f713d5d7fbdaf2c69a205ec29bd5ca8629ab', 'sha256': 'dfab31d866a77d0fa24ce904ec59f713d5d7fbdaf2c69a205ec29bd5ca8629ab', 'content_b64': 'IiIiUmVzTmV0MTggKyBTcGF0aWFsU29mdG1heCB2aXN1YWwgZW5jb2RlciAob3JpZ2luYWwgRGlmZnVzaW9uIFBvbGljeSAvIExlUm9ib3Qgc3R5bGUpLgoKRHJvcC1pbiByZXBsYWNlbWVudCBmb3IgYGBQbGFpbkNvbnZgYCBpbiB0aGUgUkdCIERpZmZ1c2lvbiBQb2xpY3kuIFRoZSBwb2ludDogaW5zdGVhZCBvZgpnbG9iYWwtbWF4LXBvb2xpbmcgdGhlIGNvbnYgZmVhdHVyZSBtYXAgdG8gYSBiYWcgb2YgZmVhdHVyZXMgKHdoaWNoIGRpc2NhcmRzICp3aGVyZSogdGhpbmdzCmFyZSksIFNwYXRpYWxTb2Z0bWF4IHR1cm5zIHRoZSBmZWF0dXJlIG1hcCBpbnRvICoqa2V5cG9pbnQgY29vcmRpbmF0ZXMqKiDigJQgZm9yIGVhY2ggb2YgSwpjaGFubmVscyBpdCBjb21wdXRlcyB0aGUgc29mdG1heC13ZWlnaHRlZCBleHBlY3RlZCAoeCwgeSkgbG9jYXRpb24gb2YgYWN0aXZhdGlvbi4gVGhhdCBnaXZlcwp0aGUgcG9saWN5IGV4cGxpY2l0LCBjb250aW51b3VzIG9iamVjdC9ncmlwcGVyIHBvc2l0aW9ucywgd2hpY2ggaXMgd2hhdCBhIHNwYXRpYWwgcGljay1hbmQtcGxhY2UKdGFzayBuZWVkcy4KCkVuY29kZXIgPSBSZXNOZXQxOCB0cnVuayAoQmF0Y2hOb3JtIC0+IEdyb3VwTm9ybSBmb3Igc21hbGwvc3RhY2tlZCBiYXRjaGVzKSB0cnVuY2F0ZWQgdG8gYW4KOHg4IGZlYXR1cmUgbWFwIChmaW5lciBsb2NhbGlzYXRpb24gdGhhbiB0aGUgZmluYWwgNHg0KSwgdGhlbiBTcGF0aWFsU29mdG1heCB3aXRoIGBgbnVtX2twYGAKa2V5cG9pbnRzIC0+IGBgMipudW1fa3BgYCBjb29yZHMgLT4gTGluZWFyIHRvIGBgb3V0X2RpbWBgLgoiIiIKCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKCmNsYXNzIFNwYXRpYWxTb2Z0bWF4KG5uLk1vZHVsZSk6CiAgICAiIiJQZXItY2hhbm5lbCBzb2Z0LWFyZ21heCBvdmVyIGEgKEMsIEgsIFcpIGZlYXR1cmUgbWFwIC0+ICgyKkspIGV4cGVjdGVkIGNvb3Jkcy4KCiAgICBPcHRpb25hbGx5IGEgMXgxIGNvbnYgZmlyc3QgbWFwcyBDIGNoYW5uZWxzIHRvIGBgbnVtX2twYGAga2V5cG9pbnQgY2hhbm5lbHMuCiAgICBSZXR1cm5zIGEgZmxhdCAoQiwgMipudW1fa3ApIHZlY3RvciBvZiAoeCwgeSkgaW4gWy0xLCAxXSBpbWFnZSBjb29yZGluYXRlcy4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbl9jaGFubmVscywgbnVtX2twPTMyKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLm51bV9rcCA9IG51bV9rcAogICAgICAgIHNlbGYua3BfY29udiA9IG5uLkNvbnYyZChpbl9jaGFubmVscywgbnVtX2twLCBrZXJuZWxfc2l6ZT0xKSBpZiBudW1fa3AgZWxzZSBOb25lCiAgICAgICAgc2VsZi5vdXRfY2hhbm5lbHMgPSBudW1fa3AgaWYgbnVtX2twIGVsc2UgaW5fY2hhbm5lbHMKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBmZWF0KTogICAgICAgICAgICAgICAgICAgICAgICMgZmVhdDogKEIsIEMsIEgsIFcpCiAgICAgICAgaWYgc2VsZi5rcF9jb252IGlzIG5vdCBOb25lOgogICAgICAgICAgICBmZWF0ID0gc2VsZi5rcF9jb252KGZlYXQpCiAgICAgICAgYiwgYywgaCwgdyA9IGZlYXQuc2hhcGUKICAgICAgICAjIGNvb3JkaW5hdGUgZ3JpZHMgaW4gWy0xLCAxXQogICAgICAgIHlzLCB4cyA9IHRvcmNoLm1lc2hncmlkKAogICAgICAgICAgICB0b3JjaC5saW5zcGFjZSgtMS4wLCAxLjAsIGgsIGRldmljZT1mZWF0LmRldmljZSwgZHR5cGU9ZmVhdC5kdHlwZSksCiAgICAgICAgICAgIHRvcmNoLmxpbnNwYWNlKC0xLjAsIDEuMCwgdywgZGV2aWNlPWZlYXQuZGV2aWNlLCBkdHlwZT1mZWF0LmR0eXBlKSwKICAgICAgICAgICAgaW5kZXhpbmc9ImlqIiwKICAgICAgICApCiAgICAgICAgeHMgPSB4cy5yZXNoYXBlKDEsIDEsIGggKiB3KQogICAgICAgIHlzID0geXMucmVzaGFwZSgxLCAxLCBoICogdykKICAgICAgICBhdHRuID0gRi5zb2Z0bWF4KGZlYXQucmVzaGFwZShiLCBjLCBoICogdyksIGRpbT0tMSkgICAjIHNwYXRpYWwgc29mdG1heCBwZXIgY2hhbm5lbAogICAgICAgIGV4cF94ID0gKGF0dG4gKiB4cykuc3VtKGRpbT0tMSkgICAgICAgICAgICAjIChCLCBDKQogICAgICAgIGV4cF95ID0gKGF0dG4gKiB5cykuc3VtKGRpbT0tMSkgICAgICAgICAgICAjIChCLCBDKQogICAgICAgIHJldHVybiB0b3JjaC5zdGFjayhbZXhwX3gsIGV4cF95XSwgZGltPS0xKS5yZXNoYXBlKGIsIDIgKiBjKSAgICMgKEIsIDJDKQoKCmRlZiBfYm5fdG9fZ24obW9kdWxlLCBudW1fZ3JvdXBzPTE2KToKICAgICIiIlJlY3Vyc2l2ZWx5IHJlcGxhY2UgQmF0Y2hOb3JtMmQgd2l0aCBHcm91cE5vcm0gKHJvYnVzdCB0byB0aGUgc21hbGwgQipvYnNfaG9yaXpvbgogICAgYmF0Y2hlcyBhbmQgdG8gcnVubmluZy1zdGF0IGRyaWZ0OyBMZVJvYm90IGRvZXMgdGhlIHNhbWUgd2hlbiBub3QgcmVseWluZyBvbiBCTiBzdGF0cykuIiIiCiAgICBmb3IgbmFtZSwgY2hpbGQgaW4gbW9kdWxlLm5hbWVkX2NoaWxkcmVuKCk6CiAgICAgICAgaWYgaXNpbnN0YW5jZShjaGlsZCwgbm4uQmF0Y2hOb3JtMmQpOgogICAgICAgICAgICBnID0gbnVtX2dyb3VwcyBpZiBjaGlsZC5udW1fZmVhdHVyZXMgJSBudW1fZ3JvdXBzID09IDAgZWxzZSAxCiAgICAgICAgICAgIHNldGF0dHIobW9kdWxlLCBuYW1lLCBubi5Hcm91cE5vcm0oZywgY2hpbGQubnVtX2ZlYXR1cmVzKSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBfYm5fdG9fZ24oY2hpbGQsIG51bV9ncm91cHMpCgoKY2xhc3MgUmVzTmV0MThTcGF0aWFsU29mdG1heChubi5Nb2R1bGUpOgogICAgIiIiUmVzTmV0MTggdHJ1bmsgKC0+IDh4OCBmZWF0dXJlIG1hcCkgKyBTcGF0aWFsU29mdG1heCArIExpbmVhciAtPiBvdXRfZGltLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbl9jaGFubmVscz0zLCBvdXRfZGltPTI1NiwgbnVtX2twPTMyLCBwcmV0cmFpbmVkPVRydWUpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIGZyb20gdG9yY2h2aXNpb24ubW9kZWxzIGltcG9ydCByZXNuZXQxOAogICAgICAgIHRyeToKICAgICAgICAgICAgd2VpZ2h0cyA9ICJJTUFHRU5FVDFLX1YxIiBpZiBwcmV0cmFpbmVkIGVsc2UgTm9uZQogICAgICAgICAgICBuZXQgPSByZXNuZXQxOCh3ZWlnaHRzPXdlaWdodHMpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOiAgICAgICAgICAgICAgICAgICAgICMgb2ZmbGluZSAvIG5vIHdlaWdodHMgY2FjaGUgLT4gdHJhaW4gZnJvbSBzY3JhdGNoCiAgICAgICAgICAgIHByaW50KGYiW2xlcm9ib3RfZW5jb2Rlcl0gcHJldHJhaW5lZCByZXNuZXQxOCB1bmF2YWlsYWJsZSAoe2V9KTsgdXNpbmcgcmFuZG9tIGluaXQiLAogICAgICAgICAgICAgICAgICBmbHVzaD1UcnVlKQogICAgICAgICAgICBuZXQgPSByZXNuZXQxOCh3ZWlnaHRzPU5vbmUpCiAgICAgICAgaWYgaW5fY2hhbm5lbHMgIT0gMzogICAgICAgICAgICAgICAgICAgICAgICMgYWRhcHQgZmlyc3QgY29udiBmb3Igbm9uLVJHQiBzdGFja3MKICAgICAgICAgICAgbmV0LmNvbnYxID0gbm4uQ29udjJkKGluX2NoYW5uZWxzLCA2NCwgNywgc3RyaWRlPTIsIHBhZGRpbmc9MywgYmlhcz1GYWxzZSkKICAgICAgICAjIHRydW5rIHVwIHRvIGxheWVyMyAtPiBmb3IgYSAxMjh4MTI4IGlucHV0IHRoaXMgeWllbGRzIGEgKDI1NiwgOCwgOCkgZmVhdHVyZSBtYXAKICAgICAgICAjIChmaW5lciB0aGFuIGxheWVyNCdzIDR4NCksIGdpdmluZyBTcGF0aWFsU29mdG1heCBtb3JlIHNwYXRpYWwgcmVzb2x1dGlvbiB0byBsb2NhbGlzZQogICAgICAgICMgdGhlIHNtYWxsIHBhcmNlbHMgYXMgd2VsbCBhcyB0aGUgbGFyZ2UgYmlucy4KICAgICAgICBzZWxmLnRydW5rID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbmV0LmNvbnYxLCBuZXQuYm4xLCBuZXQucmVsdSwgbmV0Lm1heHBvb2wsCiAgICAgICAgICAgIG5ldC5sYXllcjEsIG5ldC5sYXllcjIsIG5ldC5sYXllcjMsCiAgICAgICAgKQogICAgICAgIF9ibl90b19nbihzZWxmLnRydW5rKQogICAgICAgIGZlYXRfY2hhbm5lbHMgPSAyNTYKICAgICAgICBzZWxmLnNwYXRpYWxfc29mdG1heCA9IFNwYXRpYWxTb2Z0bWF4KGZlYXRfY2hhbm5lbHMsIG51bV9rcD1udW1fa3ApCiAgICAgICAgc2VsZi5mYyA9IG5uLlNlcXVlbnRpYWwobm4uTGluZWFyKDIgKiBudW1fa3AsIG91dF9kaW0pLCBubi5SZUxVKCkpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgaW1hZ2UpOiAgICAgICAgICAgICAgICAgICAgICAjIGltYWdlOiAoQiwgQywgSCwgVyksIGZsb2F0IGluIFswLCAxXQogICAgICAgIGZlYXQgPSBzZWxmLnRydW5rKGltYWdlKQogICAgICAgIGtwID0gc2VsZi5zcGF0aWFsX3NvZnRtYXgoZmVhdCkKICAgICAgICByZXR1cm4gc2VsZi5mYyhrcCkK'}, 'il/baselines/diffusion_policy/diffusion_policy/make_env.py': {'base_sha256': '694b92581e1e476a4a087d5a3d1be9211857fb864a4ee1dfae910fcefe6dc42a', 'sha256': '694b92581e1e476a4a087d5a3d1be9211857fb864a4ee1dfae910fcefe6dc42a', 'content_b64': 'ZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsCmltcG9ydCBneW1uYXNpdW0gYXMgZ3ltCmltcG9ydCBtYW5pX3NraWxsLmVudnMKaW1wb3J0IHdhcmVob3VzZV9zb3J0ICAjIG5vcWE6IEY0MDEgKHJlZ2lzdGVycyBXYXJlaG91c2VTb3J0LXYxKSAtLSB2ZW5kb3Igc2hpbQpmcm9tIG1hbmlfc2tpbGwudXRpbHMgaW1wb3J0IGd5bV91dGlscwpmcm9tIG1hbmlfc2tpbGwudXRpbHMud3JhcHBlcnMgaW1wb3J0IENQVUd5bVdyYXBwZXIsIEZyYW1lU3RhY2ssIFJlY29yZEVwaXNvZGUKZnJvbSBtYW5pX3NraWxsLnZlY3Rvci53cmFwcGVycy5neW1uYXNpdW0gaW1wb3J0IE1hbmlTa2lsbFZlY3RvckVudgoKZGVmIG1ha2VfZXZhbF9lbnZzKAogICAgZW52X2lkLAogICAgbnVtX2VudnM6IGludCwKICAgIHNpbV9iYWNrZW5kOiBzdHIsCiAgICBlbnZfa3dhcmdzOiBkaWN0LAogICAgb3RoZXJfa3dhcmdzOiBkaWN0LAogICAgdmlkZW9fZGlyOiBPcHRpb25hbFtzdHJdID0gTm9uZSwKICAgIHdyYXBwZXJzOiBsaXN0W2d5bS5XcmFwcGVyXSA9IFtdLAopOgogICAgIiIiQ3JlYXRlIHZlY3Rvcml6ZWQgZW52aXJvbm1lbnQgZm9yIGV2YWx1YXRpb24gYW5kL29yIHJlY29yZGluZyB2aWRlb3MuCiAgICBGb3IgQ1BVIHZlY3Rvcml6ZWQgZW52aXJvbm1lbnRzIG9ubHkgdGhlIGZpcnN0IHBhcmFsbGVsIGVudmlyb25tZW50IGlzIHVzZWQgdG8gcmVjb3JkIHZpZGVvcy4KICAgIEZvciBHUFUgdmVjdG9yaXplZCBlbnZpcm9ubWVudHMgYWxsIHBhcmFsbGVsIGVudmlyb25tZW50cyBhcmUgdXNlZCB0byByZWNvcmQgdmlkZW9zLgoKICAgIEFyZ3M6CiAgICAgICAgZW52X2lkOiB0aGUgZW52aXJvbm1lbnQgaWQKICAgICAgICBudW1fZW52czogdGhlIG51bWJlciBvZiBwYXJhbGxlbCBlbnZpcm9ubWVudHMKICAgICAgICBzaW1fYmFja2VuZDogdGhlIHNpbXVsYXRpb24gYmFja2VuZCB0byB1c2UuIGNhbiBiZSAiY3B1IiBvciAiZ3B1CiAgICAgICAgZW52X2t3YXJnczogdGhlIGVudmlyb25tZW50IGt3YXJncy4gWW91IGNhbiBhbHNvIHBhc3MgaW4gbWF4X2VwaXNvZGVfc3RlcHMgaW4gZW52X2t3YXJncyB0byBvdmVycmlkZSB0aGUgZGVmYXVsdCBtYXggZXBpc29kZSBzdGVwcyBmb3IgdGhlIGVudmlyb25tZW50LgogICAgICAgIHZpZGVvX2RpcjogdGhlIGRpcmVjdG9yeSB0byBzYXZlIHRoZSB2aWRlb3MuIElmIE5vbmUgbm8gdmlkZW9zIGFyZSByZWNvcmRlZC4KICAgICAgICB3cmFwcGVyczogdGhlIGxpc3Qgb2Ygd3JhcHBlcnMgdG8gYXBwbHkgdG8gdGhlIGVudmlyb25tZW50LgogICAgIiIiCiAgICBpZiBzaW1fYmFja2VuZCA9PSAicGh5c3hfY3B1IjoKCiAgICAgICAgZGVmIGNwdV9tYWtlX2VudigKICAgICAgICAgICAgZW52X2lkLCBzZWVkLCB2aWRlb19kaXI9Tm9uZSwgZW52X2t3YXJncz1kaWN0KCksIG90aGVyX2t3YXJncz1kaWN0KCkKICAgICAgICApOgogICAgICAgICAgICBkZWYgdGh1bmsoKToKICAgICAgICAgICAgICAgIGVudiA9IGd5bS5tYWtlKGVudl9pZCwgcmVjb25maWd1cmF0aW9uX2ZyZXE9MSwgKiplbnZfa3dhcmdzKQogICAgICAgICAgICAgICAgZm9yIHdyYXBwZXIgaW4gd3JhcHBlcnM6CiAgICAgICAgICAgICAgICAgICAgZW52ID0gd3JhcHBlcihlbnYpCiAgICAgICAgICAgICAgICBlbnYgPSBGcmFtZVN0YWNrKGVudiwgbnVtX3N0YWNrPW90aGVyX2t3YXJnc1sib2JzX2hvcml6b24iXSkKICAgICAgICAgICAgICAgIGVudiA9IENQVUd5bVdyYXBwZXIoZW52LCBpZ25vcmVfdGVybWluYXRpb25zPVRydWUsIHJlY29yZF9tZXRyaWNzPVRydWUpCiAgICAgICAgICAgICAgICBpZiB2aWRlb19kaXI6CiAgICAgICAgICAgICAgICAgICAgZW52ID0gUmVjb3JkRXBpc29kZSgKICAgICAgICAgICAgICAgICAgICAgICAgZW52LAogICAgICAgICAgICAgICAgICAgICAgICBvdXRwdXRfZGlyPXZpZGVvX2RpciwKICAgICAgICAgICAgICAgICAgICAgICAgc2F2ZV90cmFqZWN0b3J5PUZhbHNlLAogICAgICAgICAgICAgICAgICAgICAgICBpbmZvX29uX3ZpZGVvPVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgIHNvdXJjZV90eXBlPSJkaWZmdXNpb25fcG9saWN5IiwKICAgICAgICAgICAgICAgICAgICAgICAgc291cmNlX2Rlc2M9ImRpZmZ1c2lvbl9wb2xpY3kgZXZhbHVhdGlvbiByb2xsb3V0IiwKICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICBlbnYuYWN0aW9uX3NwYWNlLnNlZWQoc2VlZCkKICAgICAgICAgICAgICAgIGVudi5vYnNlcnZhdGlvbl9zcGFjZS5zZWVkKHNlZWQpCiAgICAgICAgICAgICAgICByZXR1cm4gZW52CgogICAgICAgICAgICByZXR1cm4gdGh1bmsKCiAgICAgICAgdmVjdG9yX2NscyA9ICgKICAgICAgICAgICAgZ3ltLnZlY3Rvci5TeW5jVmVjdG9yRW52CiAgICAgICAgICAgIGlmIG51bV9lbnZzID09IDEKICAgICAgICAgICAgZWxzZSBsYW1iZGEgeDogZ3ltLnZlY3Rvci5Bc3luY1ZlY3RvckVudih4LCBjb250ZXh0PSJmb3Jrc2VydmVyIikKICAgICAgICApCiAgICAgICAgZW52ID0gdmVjdG9yX2NscygKICAgICAgICAgICAgWwogICAgICAgICAgICAgICAgY3B1X21ha2VfZW52KAogICAgICAgICAgICAgICAgICAgIGVudl9pZCwKICAgICAgICAgICAgICAgICAgICBzZWVkLAogICAgICAgICAgICAgICAgICAgIHZpZGVvX2RpciBpZiBzZWVkID09IDAgZWxzZSBOb25lLAogICAgICAgICAgICAgICAgICAgIGVudl9rd2FyZ3MsCiAgICAgICAgICAgICAgICAgICAgb3RoZXJfa3dhcmdzLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgZm9yIHNlZWQgaW4gcmFuZ2UobnVtX2VudnMpCiAgICAgICAgICAgIF0KICAgICAgICApCiAgICBlbHNlOgogICAgICAgIGVudiA9IGd5bS5tYWtlKAogICAgICAgICAgICBlbnZfaWQsCiAgICAgICAgICAgIG51bV9lbnZzPW51bV9lbnZzLAogICAgICAgICAgICBzaW1fYmFja2VuZD1zaW1fYmFja2VuZCwKICAgICAgICAgICAgcmVjb25maWd1cmF0aW9uX2ZyZXE9MSwKICAgICAgICAgICAgKiplbnZfa3dhcmdzCiAgICAgICAgKQogICAgICAgICMgaG9ub3VyIGFuIGV4cGxpY2l0IGVudl9rd2FyZ3MgbWF4X2VwaXNvZGVfc3RlcHM7IGZpbmRfbWF4X2VwaXNvZGVfc3RlcHNfdmFsdWUgcmVhZHMgdGhlCiAgICAgICAgIyAqcmVnaXN0ZXJlZCogdmFsdWUgKDEwMCBmb3IgV2FyZWhvdXNlU29ydCksIHdoaWNoIHdvdWxkIGNsaXAgZXZhbCB2aWRlb3MgbWlkLWVwaXNvZGUuCiAgICAgICAgbWF4X2VwaXNvZGVfc3RlcHMgPSBlbnZfa3dhcmdzLmdldCgKICAgICAgICAgICAgIm1heF9lcGlzb2RlX3N0ZXBzIiwgZ3ltX3V0aWxzLmZpbmRfbWF4X2VwaXNvZGVfc3RlcHNfdmFsdWUoZW52KSkKICAgICAgICBmb3Igd3JhcHBlciBpbiB3cmFwcGVyczoKICAgICAgICAgICAgZW52ID0gd3JhcHBlcihlbnYpCiAgICAgICAgZW52ID0gRnJhbWVTdGFjayhlbnYsIG51bV9zdGFjaz1vdGhlcl9rd2FyZ3NbIm9ic19ob3Jpem9uIl0pCiAgICAgICAgaWYgdmlkZW9fZGlyOgogICAgICAgICAgICBlbnYgPSBSZWNvcmRFcGlzb2RlKAogICAgICAgICAgICAgICAgZW52LAogICAgICAgICAgICAgICAgb3V0cHV0X2Rpcj12aWRlb19kaXIsCiAgICAgICAgICAgICAgICBzYXZlX3RyYWplY3Rvcnk9RmFsc2UsCiAgICAgICAgICAgICAgICBzYXZlX3ZpZGVvPVRydWUsCiAgICAgICAgICAgICAgICBzb3VyY2VfdHlwZT0iZGlmZnVzaW9uX3BvbGljeSIsCiAgICAgICAgICAgICAgICBzb3VyY2VfZGVzYz0iZGlmZnVzaW9uX3BvbGljeSBldmFsdWF0aW9uIHJvbGxvdXQiLAogICAgICAgICAgICAgICAgbWF4X3N0ZXBzX3Blcl92aWRlbz1tYXhfZXBpc29kZV9zdGVwcywKICAgICAgICAgICAgKQogICAgICAgIGVudiA9IE1hbmlTa2lsbFZlY3RvckVudihlbnYsIGlnbm9yZV90ZXJtaW5hdGlvbnM9VHJ1ZSwgcmVjb3JkX21ldHJpY3M9VHJ1ZSkKICAgIHJldHVybiBlbnYK'}, 'il/baselines/diffusion_policy/diffusion_policy/plain_conv.py': {'base_sha256': 'b53d5db2019fe57c9f70f697b19d4392a485c6eed1f12eacd7d6814d27e879f3', 'sha256': 'b53d5db2019fe57c9f70f697b19d4392a485c6eed1f12eacd7d6814d27e879f3', 'content_b64': 'aW1wb3J0IHRvcmNoLm5uIGFzIG5uCgoKZGVmIG1ha2VfbWxwKGluX2NoYW5uZWxzLCBtbHBfY2hhbm5lbHMsIGFjdF9idWlsZGVyPW5uLlJlTFUsIGxhc3RfYWN0PVRydWUpOgogICAgY19pbiA9IGluX2NoYW5uZWxzCiAgICBtb2R1bGVfbGlzdCA9IFtdCiAgICBmb3IgaWR4LCBjX291dCBpbiBlbnVtZXJhdGUobWxwX2NoYW5uZWxzKToKICAgICAgICBtb2R1bGVfbGlzdC5hcHBlbmQobm4uTGluZWFyKGNfaW4sIGNfb3V0KSkKICAgICAgICBpZiBsYXN0X2FjdCBvciBpZHggPCBsZW4obWxwX2NoYW5uZWxzKSAtIDE6CiAgICAgICAgICAgIG1vZHVsZV9saXN0LmFwcGVuZChhY3RfYnVpbGRlcigpKQogICAgICAgIGNfaW4gPSBjX291dAogICAgcmV0dXJuIG5uLlNlcXVlbnRpYWwoKm1vZHVsZV9saXN0KQoKCmNsYXNzIFBsYWluQ29udihubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgaW5fY2hhbm5lbHM9MywKICAgICAgICBvdXRfZGltPTI1NiwKICAgICAgICBwb29sX2ZlYXR1cmVfbWFwPUZhbHNlLAogICAgICAgIGxhc3RfYWN0PVRydWUsICAjIFRydWUgZm9yIENvbnZCb2R5LCBGYWxzZSBmb3IgQ05OCiAgICApOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgICMgYXNzdW1lIGlucHV0IGltYWdlIHNpemUgaXMgMTI4eDEyOAoKICAgICAgICBzZWxmLm91dF9kaW0gPSBvdXRfZGltCiAgICAgICAgc2VsZi5jbm4gPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5Db252MmQoaW5fY2hhbm5lbHMsIDE2LCAzLCBwYWRkaW5nPTEsIGJpYXM9VHJ1ZSksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICAgICAgbm4uTWF4UG9vbDJkKDIsIDIpLCAgIyBbMzIsIDMyXQogICAgICAgICAgICBubi5Db252MmQoMTYsIDMyLCAzLCBwYWRkaW5nPTEsIGJpYXM9VHJ1ZSksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICAgICAgbm4uTWF4UG9vbDJkKDIsIDIpLCAgIyBbMTYsIDE2XQogICAgICAgICAgICBubi5Db252MmQoMzIsIDY0LCAzLCBwYWRkaW5nPTEsIGJpYXM9VHJ1ZSksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICAgICAgbm4uTWF4UG9vbDJkKDIsIDIpLCAgIyBbOCwgOF0KICAgICAgICAgICAgbm4uQ29udjJkKDY0LCAxMjgsIDMsIHBhZGRpbmc9MSwgYmlhcz1UcnVlKSwKICAgICAgICAgICAgbm4uUmVMVShpbnBsYWNlPVRydWUpLAogICAgICAgICAgICBubi5NYXhQb29sMmQoMiwgMiksICAjIFs0LCA0XQogICAgICAgICAgICBubi5Db252MmQoMTI4LCAxMjgsIDEsIHBhZGRpbmc9MCwgYmlhcz1UcnVlKSwKICAgICAgICAgICAgbm4uUmVMVShpbnBsYWNlPVRydWUpLAogICAgICAgICkKCiAgICAgICAgaWYgcG9vbF9mZWF0dXJlX21hcDoKICAgICAgICAgICAgc2VsZi5wb29sID0gbm4uQWRhcHRpdmVNYXhQb29sMmQoKDEsIDEpKQogICAgICAgICAgICBzZWxmLmZjID0gbWFrZV9tbHAoMTI4LCBbb3V0X2RpbV0sIGxhc3RfYWN0PWxhc3RfYWN0KQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYucG9vbCA9IE5vbmUKICAgICAgICAgICAgc2VsZi5mYyA9IG1ha2VfbWxwKDEyOCAqIDQgKiA0ICogNCwgW291dF9kaW1dLCBsYXN0X2FjdD1sYXN0X2FjdCkKCiAgICAgICAgc2VsZi5yZXNldF9wYXJhbWV0ZXJzKCkKCiAgICBkZWYgcmVzZXRfcGFyYW1ldGVycyhzZWxmKToKICAgICAgICBmb3IgbmFtZSwgbW9kdWxlIGluIHNlbGYubmFtZWRfbW9kdWxlcygpOgogICAgICAgICAgICBpZiBpc2luc3RhbmNlKG1vZHVsZSwgKG5uLkxpbmVhciwgbm4uQ29udjFkLCBubi5Db252MmQpKToKICAgICAgICAgICAgICAgIGlmIG1vZHVsZS5iaWFzIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAgIG5uLmluaXQuemVyb3NfKG1vZHVsZS5iaWFzKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGltYWdlKToKICAgICAgICB4ID0gc2VsZi5jbm4oaW1hZ2UpCiAgICAgICAgaWYgc2VsZi5wb29sIGlzIG5vdCBOb25lOgogICAgICAgICAgICB4ID0gc2VsZi5wb29sKHgpCiAgICAgICAgeCA9IHguZmxhdHRlbigxKQogICAgICAgIHggPSBzZWxmLmZjKHgpCiAgICAgICAgcmV0dXJuIHgK'}, 'il/baselines/diffusion_policy/diffusion_policy/streaming_dataset.py': {'base_sha256': '5560000cdf54b7be71372802570eb84157389da220bd15743faa8ed56ec0b8fd', 'sha256': '5560000cdf54b7be71372802570eb84157389da220bd15743faa8ed56ec0b8fd', 'content_b64': 'IiIiTWVtb3J5LWxlYW4gZGVtbyBsb2FkZXIgZm9yIHRoZSBSR0IgRGlmZnVzaW9uIFBvbGljeS4KClRoZSB2ZW5kb3JlZCBgbG9hZF9kZW1vX2RhdGFzZXRgIG1hdGVyaWFsaXNlcyBFVkVSWSBrZXkgb2YgdGhlIGg1IChyZ2IgZnJhbWVzLCBlbnYgc3RhdGVzLCBjYW1lcmEKcGFyYW1zLCAuLi4pIGZvciBhbGwgdHJhamVjdG9yaWVzIGluIHN5c3RlbSBSQU0gYmVmb3JlIGFueXRoaW5nIHJlYWNoZXMgdGhlIEdQVS4gT24gQ29sYWIKKDEyLjcgR0IgUkFNKSB0aGF0IGlzIH43IEdCIGZvciB0aGUgaGFyZCBzZXQgYW5kIE9PTSBmb3IgbWl4ZWQtbGV2ZWwgdHJhaW5pbmcuIFRoaXMgbG9hZGVyIHJlYWRzCm9uZSB0cmFqZWN0b3J5IGF0IGEgdGltZSwga2VlcHMgb25seSB3aGF0IHRoZSBwb2xpY3kgY29uc3VtZXMgYW5kIG1vdmVzIGl0IHRvIGBkZXZpY2VgCmltbWVkaWF0ZWx5IChyZ2IgYXMgdWludDgsIHNvIHRoZSB3aG9sZSBoYXJkIHNldCBpcyB+My44IEdCIG9mIFZSQU0pLgoKSXQgYWxzbyBhcHBsaWVzIHR3byBkYXRhIGZpeGVzOgogICogYGNsaXBfYWN0aW9uc2A6IHRoZSBzY3JpcHRlZCBkZW1vcyBzdG9yZSB1bmNsaXBwZWQgZW5kLWVmZmVjdG9yIGRlbHRhcyAofGF8IHVwIHRvIH4zLjcpIHdoaWxlCiAgICB0aGUgY29udHJvbGxlciBleGVjdXRlcyBjbGlwKGEsIC0xLCAxKS4gVHJhaW5pbmcgb24gdGhlIGV4ZWN1dGVkIGFjdGlvbiBrZWVwcyB0aGUgZGlmZnVzaW9uCiAgICB0YXJnZXRzIGluc2lkZSB0aGUgY2xpcF9zYW1wbGUgcmFuZ2UuCiAgKiBleHBsaWNpdCBwcm9wcmlvY2VwdGlvbiBrZXkgb3JkZXIgKHFwb3MsIHF2ZWwsIHRjcF9wb3NlLCBpc19ncmFzcGVkKSBtYXRjaGluZwogICAgRmxhdHRlblJHQkRPYnNlcnZhdGlvbldyYXBwZXIsIGluZGVwZW5kZW50IG9mIGg1cHkncyBhbHBoYWJldGljYWwga2V5IG9yZGVyLgoKTXVsdGlwbGUgZGVtbyBmaWxlcyBtYXkgYmUgZ2l2ZW4gKG1peGVkLWxldmVsIHRyYWluaW5nKTsgdHJhamVjdG9yaWVzIGFyZSBzaW1wbHkgY29uY2F0ZW5hdGVkLgoiIiIKCmltcG9ydCBqc29uCmltcG9ydCBvcwoKaW1wb3J0IGg1cHkKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLnV0aWxzLmRhdGEuZGF0YXNldCBpbXBvcnQgRGF0YXNldAoKUkdCX0tFWSA9ICJvYnMvc2Vuc29yX2RhdGEvc2NlbmVfY2FtZXJhL3JnYiIKU1RBVEVfS0VZUyA9ICgib2JzL2FnZW50L3Fwb3MiLCAib2JzL2FnZW50L3F2ZWwiLCAib2JzL2V4dHJhL3RjcF9wb3NlIiwgIm9icy9leHRyYS9pc19ncmFzcGVkIikKCgpkZWYgX3RyYWpfa2V5cyhmKToKICAgIHJldHVybiBzb3J0ZWQoKGsgZm9yIGsgaW4gZi5rZXlzKCkgaWYgay5zdGFydHN3aXRoKCJ0cmFqXyIpKSwga2V5PWxhbWJkYSB4OiBpbnQoeC5zcGxpdCgiXyIpWy0xXSkpCgoKZGVmIHJlYWRfdHJhamVjdG9yeSh0cmFqLCBkZXZpY2UsIGNsaXBfYWN0aW9ucz1UcnVlKToKICAgICIiImg1IGdyb3VwIC0+IGRpY3QocmdiIHVpbnQ4IChUKzEsMyxILFcpLCBzdGF0ZSBmMzIgKFQrMSwyNiksIGFjdGlvbnMgZjMyIChULDQpKSBvbiBkZXZpY2UuIiIiCiAgICByZ2IgPSB0b3JjaC5mcm9tX251bXB5KHRyYWpbUkdCX0tFWV1bKCldKSAgICAgICAgICAgICAgICAgIyAoVCsxLCBILCBXLCAzKSB1aW50OAogICAgcmdiID0gcmdiLnBlcm11dGUoMCwgMywgMSwgMikuY29udGlndW91cygpLnRvKGRldmljZSkgICAgICMgKFQrMSwgMywgSCwgVykKICAgIHBhcnRzID0gW10KICAgIGZvciBrIGluIFNUQVRFX0tFWVM6CiAgICAgICAgdiA9IHRyYWpba11bKCldCiAgICAgICAgaWYgdi5uZGltID09IDE6CiAgICAgICAgICAgIHYgPSB2WzosIE5vbmVdCiAgICAgICAgcGFydHMuYXBwZW5kKHYuYXN0eXBlKG5wLmZsb2F0MzIpKQogICAgc3RhdGUgPSB0b3JjaC5mcm9tX251bXB5KG5wLmNvbmNhdGVuYXRlKHBhcnRzLCBheGlzPTEpKS50byhkZXZpY2UpICAgIyAoVCsxLCAyNikKICAgIGFjdGlvbnMgPSB0b3JjaC5mcm9tX251bXB5KHRyYWpbImFjdGlvbnMiXVsoKV0uYXN0eXBlKG5wLmZsb2F0MzIpKQogICAgaWYgY2xpcF9hY3Rpb25zOgogICAgICAgIGFjdGlvbnMgPSBhY3Rpb25zLmNsYW1wXygtMS4wLCAxLjApCiAgICBhY3Rpb25zID0gYWN0aW9ucy50byhkZXZpY2UpCiAgICBhc3NlcnQgc3RhdGUuc2hhcGVbMF0gPT0gYWN0aW9ucy5zaGFwZVswXSArIDEgYW5kIHJnYi5zaGFwZVswXSA9PSBzdGF0ZS5zaGFwZVswXSwgXAogICAgICAgIChyZ2Iuc2hhcGUsIHN0YXRlLnNoYXBlLCBhY3Rpb25zLnNoYXBlKQogICAgcmV0dXJuIHsicmdiIjogcmdiLCAic3RhdGUiOiBzdGF0ZSwgImFjdGlvbnMiOiBhY3Rpb25zfQoKCmRlZiBkZW1vX2Vudl9pbmZvKGRlbW9fcGF0aCk6CiAgICB3aXRoIG9wZW4oZGVtb19wYXRoWzotMl0gKyAianNvbiIpIGFzIGY6CiAgICAgICAgcmV0dXJuIGpzb24ubG9hZChmKVsiZW52X2luZm8iXQoKCmNsYXNzIFN0cmVhbWluZ1JHQkRlbW9EYXRhc2V0KERhdGFzZXQpOgogICAgIiIiU2xpY2VzIG9mIChvYnNfaG9yaXpvbiBvYnNlcnZhdGlvbnMsIHByZWRfaG9yaXpvbiBhY3Rpb25zKSwgYWxsIHRlbnNvcnMgcmVzaWRlbnQgb24gYGRldmljZWAuCgogICAgU2FtZSBwYWRkaW5nIGNvbnZlbnRpb24gYXMgdGhlIHZlbmRvcmVkIFNtYWxsRGVtb0RhdGFzZXRfRGlmZnVzaW9uUG9saWN5OgogICAgICB8b3xvfCAgICAgICAgICAgICAgICAgICAgICAgICAgICAgb2JzZXJ2YXRpb25zOiBvYnNfaG9yaXpvbgogICAgICB8IHxhfGF8YXxhfGF8YXxhfGF8ICAgICAgICAgICAgICAgYWN0aW9ucyBleGVjdXRlZDogYWN0X2hvcml6b24KICAgICAgfHB8cHxwfHB8cHxwfHB8cHxwfHB8cHxwfHB8cHxwfHB8IGFjdGlvbnMgcHJlZGljdGVkOiBwcmVkX2hvcml6b24KICAgIHBhZF9iZWZvcmUgPSBvYnNfaG9yaXpvbiAtIDEgKGZpcnN0IG9icyByZXBlYXRlZCksIHBhZF9hZnRlciA9IHByZWRfaG9yaXpvbiAtIG9ic19ob3Jpem9uCiAgICAoYXJtIHN0YXlzIHN0aWxsOiB6ZXJvIHh5eiBkZWx0YSwgZ3JpcHBlciBjb3BpZWQgZnJvbSB0aGUgbGFzdCBhY3Rpb24pLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGRlbW9fcGF0aHMsIG9ic19ob3Jpem9uLCBwcmVkX2hvcml6b24sIGRldmljZSwgbnVtX2RlbW9zPU5vbmUsIGNsaXBfYWN0aW9ucz1UcnVlLAogICAgICAgICAgICAgICAgIHZlcmJvc2U9VHJ1ZSk6CiAgICAgICAgaWYgaXNpbnN0YW5jZShkZW1vX3BhdGhzLCBzdHIpOgogICAgICAgICAgICBkZW1vX3BhdGhzID0gW2RlbW9fcGF0aHNdCiAgICAgICAgc2VsZi5vYnNfaG9yaXpvbiwgc2VsZi5wcmVkX2hvcml6b24gPSBpbnQob2JzX2hvcml6b24pLCBpbnQocHJlZF9ob3Jpem9uKQogICAgICAgIHNlbGYuZGV2aWNlID0gdG9yY2guZGV2aWNlKGRldmljZSkKICAgICAgICBzZWxmLnRyYWplY3RvcmllcyA9IFtdCiAgICAgICAgc2VsZi5zb3VyY2VzID0gW10KICAgICAgICBmb3IgcGF0aCBpbiBkZW1vX3BhdGhzOgogICAgICAgICAgICBhc3NlcnQgb3MucGF0aC5leGlzdHMocGF0aCksIGYiZGVtbyBkYXRhc2V0IG5vdCBmb3VuZDoge3BhdGh9IgogICAgICAgICAgICB3aXRoIGg1cHkuRmlsZShwYXRoLCAiciIpIGFzIGY6CiAgICAgICAgICAgICAgICBrZXlzID0gX3RyYWpfa2V5cyhmKQogICAgICAgICAgICAgICAgaWYgbnVtX2RlbW9zIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAgIGtleXMgPSBrZXlzWzogaW50KG51bV9kZW1vcyldCiAgICAgICAgICAgICAgICBmb3IgaSwgayBpbiBlbnVtZXJhdGUoa2V5cyk6CiAgICAgICAgICAgICAgICAgICAgc2VsZi50cmFqZWN0b3JpZXMuYXBwZW5kKHJlYWRfdHJhamVjdG9yeShmW2tdLCBzZWxmLmRldmljZSwgY2xpcF9hY3Rpb25zKSkKICAgICAgICAgICAgICAgICAgICBzZWxmLnNvdXJjZXMuYXBwZW5kKChwYXRoLCBrKSkKICAgICAgICAgICAgICAgICAgICBpZiB2ZXJib3NlIGFuZCAoaSArIDEpICUgNTAgPT0gMDoKICAgICAgICAgICAgICAgICAgICAgICAgcHJpbnQoZiJbc3RyZWFtaW5nX2RhdGFzZXRdIHtvcy5wYXRoLmJhc2VuYW1lKG9zLnBhdGguZGlybmFtZShwYXRoKSl9OiAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGYie2kgKyAxfS97bGVuKGtleXMpfSB0cmFqZWN0b3JpZXMgbG9hZGVkIiwgZmx1c2g9VHJ1ZSkKICAgICAgICBhc3NlcnQgc2VsZi50cmFqZWN0b3JpZXMsICJubyB0cmFqZWN0b3JpZXMgbG9hZGVkIgogICAgICAgIGFjdF9kaW0gPSBzZWxmLnRyYWplY3Rvcmllc1swXVsiYWN0aW9ucyJdLnNoYXBlWzFdCiAgICAgICAgc2VsZi5wYWRfYWN0aW9uX2FybSA9IHRvcmNoLnplcm9zKChhY3RfZGltIC0gMSwpLCBkZXZpY2U9c2VsZi5kZXZpY2UpCgogICAgICAgIHNlbGYuc2xpY2VzID0gW10KICAgICAgICB0b3RhbCA9IDAKICAgICAgICBwYWRfYmVmb3JlID0gc2VsZi5vYnNfaG9yaXpvbiAtIDEKICAgICAgICBwYWRfYWZ0ZXIgPSBzZWxmLnByZWRfaG9yaXpvbiAtIHNlbGYub2JzX2hvcml6b24KICAgICAgICBmb3IgdGksIHRyIGluIGVudW1lcmF0ZShzZWxmLnRyYWplY3Rvcmllcyk6CiAgICAgICAgICAgIEwgPSB0clsiYWN0aW9ucyJdLnNoYXBlWzBdCiAgICAgICAgICAgIHRvdGFsICs9IEwKICAgICAgICAgICAgc2VsZi5zbGljZXMgKz0gWyh0aSwgcywgcyArIHNlbGYucHJlZF9ob3Jpem9uKSBmb3IgcyBpbiByYW5nZSgtcGFkX2JlZm9yZSwgTCAtIHNlbGYucHJlZF9ob3Jpem9uICsgcGFkX2FmdGVyKV0KICAgICAgICBuX2ZyYW1lcyA9IHN1bSh0WyJyZ2IiXS5zaGFwZVswXSBmb3IgdCBpbiBzZWxmLnRyYWplY3RvcmllcykKICAgICAgICBnYiA9IG5fZnJhbWVzICogaW50KG5wLnByb2Qoc2VsZi50cmFqZWN0b3JpZXNbMF1bInJnYiJdLnNoYXBlWzE6XSkpIC8gMWU5CiAgICAgICAgaWYgdmVyYm9zZToKICAgICAgICAgICAgcHJpbnQoZiJbc3RyZWFtaW5nX2RhdGFzZXRdIHtsZW4oc2VsZi50cmFqZWN0b3JpZXMpfSB0cmFqZWN0b3JpZXMsIHt0b3RhbH0gdHJhbnNpdGlvbnMsICIKICAgICAgICAgICAgICAgICAgZiJ7bGVuKHNlbGYuc2xpY2VzKX0gb2JzIHNlcXVlbmNlcywgcmdiIHtnYjouMmZ9IEdCIHVpbnQ4IG9uIHtzZWxmLmRldmljZX0iLCBmbHVzaD1UcnVlKQoKICAgIGRlZiBfX2xlbl9fKHNlbGYpOgogICAgICAgIHJldHVybiBsZW4oc2VsZi5zbGljZXMpCgogICAgZGVmIF9fZ2V0aXRlbV9fKHNlbGYsIGluZGV4KToKICAgICAgICB0aSwgc3RhcnQsIGVuZCA9IHNlbGYuc2xpY2VzW2luZGV4XQogICAgICAgIHRyID0gc2VsZi50cmFqZWN0b3JpZXNbdGldCiAgICAgICAgTCA9IHRyWyJhY3Rpb25zIl0uc2hhcGVbMF0KICAgICAgICBvYnNfc2VxID0ge30KICAgICAgICBmb3IgayBpbiAoInJnYiIsICJzdGF0ZSIpOgogICAgICAgICAgICB2ID0gdHJba11bbWF4KDAsIHN0YXJ0KTogc3RhcnQgKyBzZWxmLm9ic19ob3Jpem9uXQogICAgICAgICAgICBpZiBzdGFydCA8IDA6CiAgICAgICAgICAgICAgICB2ID0gdG9yY2guY2F0KFt2WzoxXS5leHBhbmQoLXN0YXJ0LCAqdi5zaGFwZVsxOl0pLCB2XSwgZGltPTApCiAgICAgICAgICAgIG9ic19zZXFba10gPSB2CiAgICAgICAgYWN0ID0gdHJbImFjdGlvbnMiXVttYXgoMCwgc3RhcnQpOiBlbmRdCiAgICAgICAgaWYgc3RhcnQgPCAwOgogICAgICAgICAgICBhY3QgPSB0b3JjaC5jYXQoW2FjdFs6MV0uZXhwYW5kKC1zdGFydCwgYWN0LnNoYXBlWzFdKSwgYWN0XSwgZGltPTApCiAgICAgICAgaWYgZW5kID4gTDoKICAgICAgICAgICAgcGFkID0gdG9yY2guY2F0KChzZWxmLnBhZF9hY3Rpb25fYXJtLCBhY3RbLTEsIC0xOl0pLCBkaW09MCkKICAgICAgICAgICAgYWN0ID0gdG9yY2guY2F0KFthY3QsIHBhZFtOb25lXS5leHBhbmQoZW5kIC0gTCwgLTEpXSwgZGltPTApCiAgICAgICAgYXNzZXJ0IG9ic19zZXFbInN0YXRlIl0uc2hhcGVbMF0gPT0gc2VsZi5vYnNfaG9yaXpvbiBhbmQgYWN0LnNoYXBlWzBdID09IHNlbGYucHJlZF9ob3Jpem9uCiAgICAgICAgcmV0dXJuIHsib2JzZXJ2YXRpb25zIjogb2JzX3NlcSwgImFjdGlvbnMiOiBhY3R9Cg=='}, 'il/baselines/diffusion_policy/diffusion_policy/utils.py': {'base_sha256': '7a6888af501744f5ee6fbce808899637a066c84bd154bc6ed2884db053fdebbe', 'sha256': '7a6888af501744f5ee6fbce808899637a066c84bd154bc6ed2884db053fdebbe', 'content_b64': 'aW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgpmcm9tIGd5bW5hc2l1bSBpbXBvcnQgc3BhY2VzCmZyb20gaDVweSBpbXBvcnQgRGF0YXNldCwgRmlsZSwgR3JvdXAKZnJvbSB0b3JjaC51dGlscy5kYXRhLnNhbXBsZXIgaW1wb3J0IFNhbXBsZXIKCgpjbGFzcyBJdGVyYXRpb25CYXNlZEJhdGNoU2FtcGxlcihTYW1wbGVyKToKICAgICIiIldyYXBzIGEgQmF0Y2hTYW1wbGVyLgogICAgUmVzYW1wbGluZyBmcm9tIGl0IHVudGlsIGEgc3BlY2lmaWVkIG51bWJlciBvZiBpdGVyYXRpb25zIGhhdmUgYmVlbiBzYW1wbGVkCiAgICBSZWZlcmVuY2VzOgogICAgICAgIGh0dHBzOi8vZ2l0aHViLmNvbS9mYWNlYm9va3Jlc2VhcmNoL21hc2tyY25uLWJlbmNobWFyay9ibG9iL21hc3Rlci9tYXNrcmNubl9iZW5jaG1hcmsvZGF0YS9zYW1wbGVycy9pdGVyYXRpb25fYmFzZWRfYmF0Y2hfc2FtcGxlci5weQogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGJhdGNoX3NhbXBsZXIsIG51bV9pdGVyYXRpb25zLCBzdGFydF9pdGVyPTApOgogICAgICAgIHNlbGYuYmF0Y2hfc2FtcGxlciA9IGJhdGNoX3NhbXBsZXIKICAgICAgICBzZWxmLm51bV9pdGVyYXRpb25zID0gbnVtX2l0ZXJhdGlvbnMKICAgICAgICBzZWxmLnN0YXJ0X2l0ZXIgPSBzdGFydF9pdGVyCgogICAgZGVmIF9faXRlcl9fKHNlbGYpOgogICAgICAgIGl0ZXJhdGlvbiA9IHNlbGYuc3RhcnRfaXRlcgogICAgICAgIHdoaWxlIGl0ZXJhdGlvbiA8IHNlbGYubnVtX2l0ZXJhdGlvbnM6CiAgICAgICAgICAgICMgaWYgdGhlIHVuZGVybHlpbmcgc2FtcGxlciBoYXMgYSBzZXRfZXBvY2ggbWV0aG9kLCBsaWtlCiAgICAgICAgICAgICMgRGlzdHJpYnV0ZWRTYW1wbGVyLCB1c2VkIGZvciBtYWtpbmcgZWFjaCBwcm9jZXNzIHNlZQogICAgICAgICAgICAjIGEgZGlmZmVyZW50IHNwbGl0IG9mIHRoZSBkYXRhc2V0LCB0aGVuIHNldCBpdAogICAgICAgICAgICBpZiBoYXNhdHRyKHNlbGYuYmF0Y2hfc2FtcGxlci5zYW1wbGVyLCAic2V0X2Vwb2NoIik6CiAgICAgICAgICAgICAgICBzZWxmLmJhdGNoX3NhbXBsZXIuc2FtcGxlci5zZXRfZXBvY2goaXRlcmF0aW9uKQogICAgICAgICAgICBmb3IgYmF0Y2ggaW4gc2VsZi5iYXRjaF9zYW1wbGVyOgogICAgICAgICAgICAgICAgeWllbGQgYmF0Y2gKICAgICAgICAgICAgICAgIGl0ZXJhdGlvbiArPSAxCiAgICAgICAgICAgICAgICBpZiBpdGVyYXRpb24gPj0gc2VsZi5udW1faXRlcmF0aW9uczoKICAgICAgICAgICAgICAgICAgICBicmVhawoKICAgIGRlZiBfX2xlbl9fKHNlbGYpOgogICAgICAgIHJldHVybiBzZWxmLm51bV9pdGVyYXRpb25zIC0gc2VsZi5zdGFydF9pdGVyCgoKZGVmIHdvcmtlcl9pbml0X2ZuKHdvcmtlcl9pZCwgYmFzZV9zZWVkPU5vbmUpOgogICAgIiIiVGhlIGZ1bmN0aW9uIGlzIGRlc2lnbmVkIGZvciBweXRvcmNoIG11bHRpLXByb2Nlc3MgZGF0YWxvYWRlci4KICAgIE5vdGUgdGhhdCB3ZSB1c2UgdGhlIHB5dG9yY2ggcmFuZG9tIGdlbmVyYXRvciB0byBnZW5lcmF0ZSBhIGJhc2Vfc2VlZC4KICAgIFBsZWFzZSB0cnkgdG8gYmUgY29uc2lzdGVudC4KICAgIFJlZmVyZW5jZXM6CiAgICAgICAgaHR0cHM6Ly9weXRvcmNoLm9yZy9kb2NzL3N0YWJsZS9ub3Rlcy9mYXEuaHRtbCNkYXRhbG9hZGVyLXdvcmtlcnMtcmFuZG9tLXNlZWQKICAgICIiIgogICAgaWYgYmFzZV9zZWVkIGlzIE5vbmU6CiAgICAgICAgYmFzZV9zZWVkID0gdG9yY2guSW50VGVuc29yKDEpLnJhbmRvbV8oKS5pdGVtKCkKICAgICMgcHJpbnQod29ya2VyX2lkLCBiYXNlX3NlZWQpCiAgICBucC5yYW5kb20uc2VlZChiYXNlX3NlZWQgKyB3b3JrZXJfaWQpCgoKVEFSR0VUX0tFWV9UT19TT1VSQ0VfS0VZID0gewogICAgInN0YXRlcyI6ICJlbnZfc3RhdGVzIiwKICAgICJvYnNlcnZhdGlvbnMiOiAib2JzIiwKICAgICJzdWNjZXNzIjogInN1Y2Nlc3MiLAogICAgIm5leHRfb2JzZXJ2YXRpb25zIjogIm9icyIsCiAgICAjICdkb25lcyc6ICdkb25lcycsCiAgICAjICdyZXdhcmRzJzogJ3Jld2FyZHMnLAogICAgImFjdGlvbnMiOiAiYWN0aW9ucyIsCn0KCgpkZWYgbG9hZF9jb250ZW50X2Zyb21faDVfZmlsZShmaWxlKToKICAgIGlmIGlzaW5zdGFuY2UoZmlsZSwgKEZpbGUsIEdyb3VwKSk6CiAgICAgICAgcmV0dXJuIHtrZXk6IGxvYWRfY29udGVudF9mcm9tX2g1X2ZpbGUoZmlsZVtrZXldKSBmb3Iga2V5IGluIGxpc3QoZmlsZS5rZXlzKCkpfQogICAgZWxpZiBpc2luc3RhbmNlKGZpbGUsIERhdGFzZXQpOgogICAgICAgIHJldHVybiBmaWxlWygpXQogICAgZWxzZToKICAgICAgICByYWlzZSBOb3RJbXBsZW1lbnRlZEVycm9yKGYiVW5zcHBvcnRlZCBoNSBmaWxlIHR5cGU6IHt0eXBlKGZpbGUpfSIpCgoKZGVmIGxvYWRfaGRmNSgKICAgIHBhdGgsCik6CiAgICBwcmludCgiTG9hZGluZyBIREY1IGZpbGUiLCBwYXRoKQogICAgZmlsZSA9IEZpbGUocGF0aCwgInIiKQogICAgcmV0ID0gbG9hZF9jb250ZW50X2Zyb21faDVfZmlsZShmaWxlKQogICAgZmlsZS5jbG9zZSgpCiAgICBwcmludCgiTG9hZGVkIikKICAgIHJldHVybiByZXQKCgpkZWYgbG9hZF90cmFqX2hkZjUocGF0aCwgbnVtX3RyYWo9Tm9uZSk6CiAgICBwcmludCgiTG9hZGluZyBIREY1IGZpbGUiLCBwYXRoKQogICAgZmlsZSA9IEZpbGUocGF0aCwgInIiKQogICAga2V5cyA9IGxpc3QoZmlsZS5rZXlzKCkpCiAgICBpZiBudW1fdHJhaiBpcyBub3QgTm9uZToKICAgICAgICBhc3NlcnQgbnVtX3RyYWogPD0gbGVuKGtleXMpLCBmIm51bV90cmFqOiB7bnVtX3RyYWp9ID4gbGVuKGtleXMpOiB7bGVuKGtleXMpfSIKICAgICAgICBrZXlzID0gc29ydGVkKGtleXMsIGtleT1sYW1iZGEgeDogaW50KHguc3BsaXQoIl8iKVstMV0pKQogICAgICAgIGtleXMgPSBrZXlzWzpudW1fdHJhal0KICAgIHJldCA9IHtrZXk6IGxvYWRfY29udGVudF9mcm9tX2g1X2ZpbGUoZmlsZVtrZXldKSBmb3Iga2V5IGluIGtleXN9CiAgICBmaWxlLmNsb3NlKCkKICAgIHByaW50KCJMb2FkZWQiKQogICAgcmV0dXJuIHJldAoKCmRlZiBsb2FkX2RlbW9fZGF0YXNldCgKICAgIHBhdGgsIGtleXM9WyJvYnNlcnZhdGlvbnMiLCAiYWN0aW9ucyJdLCBudW1fdHJhaj1Ob25lLCBjb25jYXQ9VHJ1ZQopOgogICAgIyBhc3NlcnQgbnVtX3RyYWogaXMgTm9uZQogICAgcmF3X2RhdGEgPSBsb2FkX3RyYWpfaGRmNShwYXRoLCBudW1fdHJhaikKICAgICMgcmF3X2RhdGEgaGFzIGtleXMgbGlrZTogWyd0cmFqXzAnLCAndHJhal8xJywgLi4uXQogICAgIyByYXdfZGF0YVsndHJhal8wJ10gaGFzIGtleXMgbGlrZTogWydhY3Rpb25zJywgJ2RvbmVzJywgJ2Vudl9zdGF0ZXMnLCAnaW5mb3MnLCAuLi5dCiAgICBfdHJhaiA9IHJhd19kYXRhWyJ0cmFqXzAiXQogICAgZm9yIGtleSBpbiBrZXlzOgogICAgICAgIHNvdXJjZV9rZXkgPSBUQVJHRVRfS0VZX1RPX1NPVVJDRV9LRVlba2V5XQogICAgICAgIGFzc2VydCBzb3VyY2Vfa2V5IGluIF90cmFqLCBmImtleToge3NvdXJjZV9rZXl9IG5vdCBpbiB0cmFqXzA6IHtfdHJhai5rZXlzKCl9IgogICAgZGF0YXNldCA9IHt9CiAgICBmb3IgdGFyZ2V0X2tleSBpbiBrZXlzOgogICAgICAgICMgaWYgJ25leHQnIGluIHRhcmdldF9rZXk6CiAgICAgICAgIyAgICAgcmFpc2UgTm90SW1wbGVtZW50ZWRFcnJvcignUGxlYXNlIGNhcmVmdWxseSBkZWFsIHdpdGggdGhlIGxlbmd0aCBvZiB0cmFqZWN0b3J5JykKICAgICAgICBzb3VyY2Vfa2V5ID0gVEFSR0VUX0tFWV9UT19TT1VSQ0VfS0VZW3RhcmdldF9rZXldCiAgICAgICAgZGF0YXNldFt0YXJnZXRfa2V5XSA9IFtyYXdfZGF0YVtpZHhdW3NvdXJjZV9rZXldIGZvciBpZHggaW4gcmF3X2RhdGFdCiAgICAgICAgaWYgaXNpbnN0YW5jZShkYXRhc2V0W3RhcmdldF9rZXldWzBdLCBucC5uZGFycmF5KSBhbmQgY29uY2F0OgogICAgICAgICAgICBpZiB0YXJnZXRfa2V5IGluIFsib2JzZXJ2YXRpb25zIiwgInN0YXRlcyJdIGFuZCBsZW4oCiAgICAgICAgICAgICAgICBkYXRhc2V0W3RhcmdldF9rZXldWzBdCiAgICAgICAgICAgICkgPiBsZW4ocmF3X2RhdGFbInRyYWpfMCJdWyJhY3Rpb25zIl0pOgogICAgICAgICAgICAgICAgZGF0YXNldFt0YXJnZXRfa2V5XSA9IG5wLmNvbmNhdGVuYXRlKAogICAgICAgICAgICAgICAgICAgIFt0WzotMV0gZm9yIHQgaW4gZGF0YXNldFt0YXJnZXRfa2V5XV0sIGF4aXM9MAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBlbGlmIHRhcmdldF9rZXkgaW4gWyJuZXh0X29ic2VydmF0aW9ucyIsICJuZXh0X3N0YXRlcyJdIGFuZCBsZW4oCiAgICAgICAgICAgICAgICBkYXRhc2V0W3RhcmdldF9rZXldWzBdCiAgICAgICAgICAgICkgPiBsZW4ocmF3X2RhdGFbInRyYWpfMCJdWyJhY3Rpb25zIl0pOgogICAgICAgICAgICAgICAgZGF0YXNldFt0YXJnZXRfa2V5XSA9IG5wLmNvbmNhdGVuYXRlKAogICAgICAgICAgICAgICAgICAgIFt0WzE6XSBmb3IgdCBpbiBkYXRhc2V0W3RhcmdldF9rZXldXSwgYXhpcz0wCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBkYXRhc2V0W3RhcmdldF9rZXldID0gbnAuY29uY2F0ZW5hdGUoZGF0YXNldFt0YXJnZXRfa2V5XSwgYXhpcz0wKQoKICAgICAgICAgICAgcHJpbnQoIkxvYWQiLCB0YXJnZXRfa2V5LCBkYXRhc2V0W3RhcmdldF9rZXldLnNoYXBlKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHByaW50KAogICAgICAgICAgICAgICAgIkxvYWQiLAogICAgICAgICAgICAgICAgdGFyZ2V0X2tleSwKICAgICAgICAgICAgICAgIGxlbihkYXRhc2V0W3RhcmdldF9rZXldKSwKICAgICAgICAgICAgICAgIHR5cGUoZGF0YXNldFt0YXJnZXRfa2V5XVswXSksCiAgICAgICAgICAgICkKICAgIHJldHVybiBkYXRhc2V0CgoKZGVmIGNvbnZlcnRfb2JzKG9icywgY29uY2F0X2ZuLCB0cmFuc3Bvc2VfZm4sIHN0YXRlX29ic19leHRyYWN0b3IsIGRlcHRoID0gVHJ1ZSk6CiAgICBpbWdfZGljdCA9IG9ic1sic2Vuc29yX2RhdGEiXQogICAgbHMgPSBbInJnYiJdCiAgICBpZiBkZXB0aDoKICAgICAgICBscyA9IFsicmdiIiwgImRlcHRoIl0KCiAgICBuZXdfaW1nX2RpY3QgPSB7CiAgICAgICAga2V5OiB0cmFuc3Bvc2VfZm4oCiAgICAgICAgICAgIGNvbmNhdF9mbihbdltrZXldIGZvciB2IGluIGltZ19kaWN0LnZhbHVlcygpXSkKICAgICAgICApICAjIChDLCBILCBXKSBvciAoQiwgQywgSCwgVykKICAgICAgICBmb3Iga2V5IGluIGxzCiAgICB9CiAgICBpZiAiZGVwdGgiIGluIG5ld19pbWdfZGljdCBhbmQgaXNpbnN0YW5jZShuZXdfaW1nX2RpY3RbJ2RlcHRoJ10sIHRvcmNoLlRlbnNvcik6ICMgTVMyIHZlYyBlbnYgdXNlcyBmbG9hdDE2LCBidXQgZ3ltIEFzeW5jVmVjRW52IHVzZXMgZmxvYXQzMgogICAgICAgIG5ld19pbWdfZGljdFsnZGVwdGgnXSA9IG5ld19pbWdfZGljdFsnZGVwdGgnXS50byh0b3JjaC5mbG9hdDE2KQoKICAgICMgVW5pZmllZCB2ZXJzaW9uCiAgICBzdGF0ZXNfdG9fc3RhY2sgPSBzdGF0ZV9vYnNfZXh0cmFjdG9yKG9icykKICAgIGZvciBqIGluIHJhbmdlKGxlbihzdGF0ZXNfdG9fc3RhY2spKToKICAgICAgICBpZiBzdGF0ZXNfdG9fc3RhY2tbal0uZHR5cGUgPT0gbnAuZmxvYXQ2NDoKICAgICAgICAgICAgc3RhdGVzX3RvX3N0YWNrW2pdID0gc3RhdGVzX3RvX3N0YWNrW2pdLmFzdHlwZShucC5mbG9hdDMyKQogICAgdHJ5OgogICAgICAgIHN0YXRlID0gbnAuaHN0YWNrKHN0YXRlc190b19zdGFjaykKICAgIGV4Y2VwdDogICMgZGlydHkgZml4IGZvciBjb25jYXQgdHJhamVjdG9yeSBvZiBzdGF0ZXMKICAgICAgICBzdGF0ZSA9IG5wLmNvbHVtbl9zdGFjayhzdGF0ZXNfdG9fc3RhY2spCiAgICBpZiBzdGF0ZS5kdHlwZSA9PSBucC5mbG9hdDY0OgogICAgICAgIGZvciB4IGluIHN0YXRlc190b19zdGFjazoKICAgICAgICAgICAgcHJpbnQoeC5zaGFwZSwgeC5kdHlwZSkKICAgICAgICBpbXBvcnQgcGRiCgogICAgICAgIHBkYi5zZXRfdHJhY2UoKQoKICAgIG91dF9kaWN0ID0gewogICAgICAgICJzdGF0ZSI6IHN0YXRlLAogICAgICAgICJyZ2IiOiBuZXdfaW1nX2RpY3RbInJnYiJdLAogICAgfQoKICAgIGlmICJkZXB0aCIgaW4gbmV3X2ltZ19kaWN0OgogICAgICAgIG91dF9kaWN0WyJkZXB0aCJdID0gbmV3X2ltZ19kaWN0WyJkZXB0aCJdCgoKICAgIHJldHVybiBvdXRfZGljdAoKCmRlZiBidWlsZF9vYnNfc3BhY2UoZW52LCBkZXB0aF9kdHlwZSwgc3RhdGVfb2JzX2V4dHJhY3Rvcik6CiAgICAjIE5PVEU6IFdlIGhhdmUgdG8gdXNlIGZsb2F0MzIgZm9yIGd5bSBBc3luY1ZlY0VudiBzaW5jZSBpdCBkb2VzIG5vdCBzdXBwb3J0IGZsb2F0MTYsIGJ1dCB3ZSBjYW4gdXNlIGZsb2F0MTYgZm9yIE1TMiB2ZWMgZW52CiAgICBvYnNfc3BhY2UgPSBlbnYub2JzZXJ2YXRpb25fc3BhY2UKCiAgICAjIFVuaWZpZWQgdmVyc2lvbgogICAgc3RhdGVfZGltID0gc3VtKFt2LnNoYXBlWzBdIGZvciB2IGluIHN0YXRlX29ic19leHRyYWN0b3Iob2JzX3NwYWNlKV0pCgogICAgc2luZ2xlX2ltZ19zcGFjZSA9IG5leHQoaXRlcihlbnYub2JzZXJ2YXRpb25fc3BhY2VbImltYWdlIl0udmFsdWVzKCkpKQogICAgaCwgdywgXyA9IHNpbmdsZV9pbWdfc3BhY2VbInJnYiJdLnNoYXBlCiAgICBuX2ltYWdlcyA9IGxlbihlbnYub2JzZXJ2YXRpb25fc3BhY2VbImltYWdlIl0pCgogICAgcmV0dXJuIHNwYWNlcy5EaWN0KAogICAgICAgIHsKICAgICAgICAgICAgInN0YXRlIjogc3BhY2VzLkJveCgKICAgICAgICAgICAgICAgIC1mbG9hdCgiaW5mIiksIGZsb2F0KCJpbmYiKSwgc2hhcGU9KHN0YXRlX2RpbSwpLCBkdHlwZT1ucC5mbG9hdDMyCiAgICAgICAgICAgICksCiAgICAgICAgICAgICJyZ2IiOiBzcGFjZXMuQm94KDAsIDI1NSwgc2hhcGU9KG5faW1hZ2VzICogMywgaCwgdyksIGR0eXBlPW5wLnVpbnQ4KSwKICAgICAgICAgICAgImRlcHRoIjogc3BhY2VzLkJveCgKICAgICAgICAgICAgICAgIC1mbG9hdCgiaW5mIiksIGZsb2F0KCJpbmYiKSwgc2hhcGU9KG5faW1hZ2VzLCBoLCB3KSwgZHR5cGU9ZGVwdGhfZHR5cGUKICAgICAgICAgICAgKSwKICAgICAgICB9CiAgICApCgoKZGVmIGJ1aWxkX3N0YXRlX29ic19leHRyYWN0b3IoZW52X2lkKToKICAgICMgTk9URTogWW91IGNhbiB0dW5lL21vZGlmeSBzdGF0ZSBvYnNlcnZhdGlvbnMgc3BlY2lmaWMgdG8gZWFjaCBlbnZpcm9ubWVudCBoZXJlIGFzIHlvdSB3aXNoLiBCeSBkZWZhdWx0IHdlIGluY2x1ZGUgYWxsIGRhdGEKICAgICMgYnV0IGluIHNvbWUgdXNlIGNhc2VzIHlvdSBtaWdodCB3YW50IHRvIGV4Y2x1ZGUgZS5nLiBvYnNbImFnZW50Il1bInF2ZWwiXSBhcyBxdmVsIGlzIG5vdCBhbHdheXMgc29tZXRoaW5nIHlvdSBxdWVyeSBpbiB0aGUgcmVhbCB3b3JsZC4KICAgIHJldHVybiBsYW1iZGEgb2JzOiBsaXN0KG9ic1siYWdlbnQiXS52YWx1ZXMoKSkgKyBsaXN0KG9ic1siZXh0cmEiXS52YWx1ZXMoKSkK'}, 'il/baselines/diffusion_policy/record_dp.py': {'base_sha256': 'c6dcd5b10d13b4f22658b8bc8cbb86318a64f75acfb0aeeab7cbafae3f310176', 'sha256': 'c6dcd5b10d13b4f22658b8bc8cbb86318a64f75acfb0aeeab7cbafae3f310176', 'content_b64': 'IiIiUmVjb3JkIE9ORSBjbGVhbiBEUC1zdGF0ZSByb2xsb3V0IHZpZGVvIChyZW5kZXIgKyBzY2VuZSBjYW0pIGZyb20gYSB0cmFpbmVkIGNoZWNrcG9pbnQuCgpCdWlsZHMgdGhlIGV2YWwgZW52IGRpcmVjdGx5IChyYXRoZXIgdGhhbiB2aWEgbWFrZV9ldmFsX2VudnMpIHNvIHdlIGNhbiBzZXQKYGBtYXhfc3RlcHNfcGVyX3ZpZGVvYGAgdG8gdGhlIEZVTEwgaG9yaXpvbiDigJQgbWFrZV9ldmFsX2VudnMgZGVyaXZlcyBpdCBmcm9tIHRoZSAqcmVnaXN0ZXJlZCoKbWF4X2VwaXNvZGVfc3RlcHMgKDEwMCksIHdoaWNoIHdvdWxkIGNsaXAgdGhlIHZpZGVvIGJlZm9yZSB0aGUgcmV0dXJuLWhvbWUgYW5kIHNwbGl0IGl0IGluIHR3by4KIiIiCmltcG9ydCBzeXMsIGdsb2IsIG9zCmZyb20gdHlwZXMgaW1wb3J0IFNpbXBsZU5hbWVzcGFjZQppbXBvcnQgdG9yY2gKaW1wb3J0IGd5bW5hc2l1bSBhcyBneW0KaW1wb3J0IHdhcmVob3VzZV9zb3J0ICAjIG5vcWEKZnJvbSB0cmFpbiBpbXBvcnQgQWdlbnQgICMgdmVuZG9yZWQgRFAgQWdlbnQgKGRpZmZ1c2lvbiBVLU5ldCkKZnJvbSBtYW5pX3NraWxsLnV0aWxzIGltcG9ydCBjb21tb24KZnJvbSBtYW5pX3NraWxsLnV0aWxzLndyYXBwZXJzIGltcG9ydCBGcmFtZVN0YWNrLCBSZWNvcmRFcGlzb2RlCmZyb20gbWFuaV9za2lsbC52ZWN0b3Iud3JhcHBlcnMuZ3ltbmFzaXVtIGltcG9ydCBNYW5pU2tpbGxWZWN0b3JFbnYKCkNLUFQgPSBzeXMuYXJndlsxXSBpZiBsZW4oc3lzLmFyZ3YpID4gMSBlbHNlICJydW5zL3dhcmVob3VzZV9zdGF0ZV9kcF92Mi9jaGVja3BvaW50cy9iZXN0X2V2YWxfc3VjY2Vzc19hdF9lbmQucHQiClZJRERJUiA9IHN5cy5hcmd2WzJdIGlmIGxlbihzeXMuYXJndikgPiAyIGVsc2UgIi9ob21lL2RhdmlkL2NvZGUvbWFyc29faGFja2F0aG9uL2lsL3ZpZGVvcy9zdGF0ZV9kcCIKSE9SSVpPTiA9IDIwMCAgICMgZnVsbCB0d28tcGFyY2VsIHNvcnQgKyBjbGVhbiByZXR1cm4taG9tZSBpbiBPTkUgY2xpcAphcmdzID0gU2ltcGxlTmFtZXNwYWNlKG9ic19ob3Jpem9uPTIsIGFjdF9ob3Jpem9uPTgsIHByZWRfaG9yaXpvbj0xNiwKICAgICAgICAgICAgICAgICAgICAgICBkaWZmdXNpb25fc3RlcF9lbWJlZF9kaW09NjQsIHVuZXRfZGltcz1bNjQsIDEyOCwgMjU2XSwgbl9ncm91cHM9OCkKZGV2aWNlID0gImN1ZGEiCgpiYXNlID0gZ3ltLm1ha2UoIldhcmVob3VzZVNvcnQtdjEiLCBudW1fZW52cz0xLCBvYnNfbW9kZT0ic3RhdGUiLAogICAgICAgICAgICAgICAgY29udHJvbF9tb2RlPSJwZF9lZV9kZWx0YV9wb3MiLCBzaW1fYmFja2VuZD0iZ3B1IiwgcmVuZGVyX21vZGU9ImFsbCIsCiAgICAgICAgICAgICAgICBtYXhfZXBpc29kZV9zdGVwcz1IT1JJWk9OLCBkaWZmaWN1bHR5PSJlYXN5IiwgbnVtX3BhcmNlbHM9MiwgZml4ZWRfcG9zZXM9VHJ1ZSwKICAgICAgICAgICAgICAgIGh1bWFuX3JlbmRlcl9jYW1lcmFfY29uZmlncz1kaWN0KHNoYWRlcl9wYWNrPSJkZWZhdWx0IikpCmJhc2UgPSBGcmFtZVN0YWNrKGJhc2UsIG51bV9zdGFjaz1hcmdzLm9ic19ob3Jpem9uKQpiYXNlID0gUmVjb3JkRXBpc29kZShiYXNlLCBvdXRwdXRfZGlyPVZJRERJUiwgc2F2ZV90cmFqZWN0b3J5PUZhbHNlLCBzYXZlX3ZpZGVvPVRydWUsCiAgICAgICAgICAgICAgICAgICAgIHZpZGVvX2Zwcz0yMCwgbWF4X3N0ZXBzX3Blcl92aWRlbz1IT1JJWk9OKQplbnZzID0gTWFuaVNraWxsVmVjdG9yRW52KGJhc2UsIGlnbm9yZV90ZXJtaW5hdGlvbnM9VHJ1ZSwgcmVjb3JkX21ldHJpY3M9VHJ1ZSkKCmFnZW50ID0gQWdlbnQoZW52cywgYXJncykudG8oZGV2aWNlKQpjayA9IHRvcmNoLmxvYWQoQ0tQVCwgbWFwX2xvY2F0aW9uPWRldmljZSwgd2VpZ2h0c19vbmx5PUZhbHNlKQphZ2VudC5sb2FkX3N0YXRlX2RpY3QoY2tbImVtYV9hZ2VudCJdKTsgYWdlbnQuZXZhbCgpCgpvYnMsIF8gPSBlbnZzLnJlc2V0KHNlZWQ9NTAwMCkKc3RlcHMgPSAwCndoaWxlIHN0ZXBzIDwgSE9SSVpPTjoKICAgIG9icyA9IGNvbW1vbi50b190ZW5zb3Iob2JzLCBkZXZpY2UpCiAgICBhc2VxID0gYWdlbnQuZ2V0X2FjdGlvbihvYnMpCiAgICBzdG9wID0gRmFsc2UKICAgIGZvciBpIGluIHJhbmdlKGFzZXEuc2hhcGVbMV0pOgogICAgICAgIG9icywgciwgdGUsIHRyLCBpbmZvID0gZW52cy5zdGVwKGFzZXFbOiwgaV0pOyBzdGVwcyArPSAxCiAgICAgICAgaWYgdHIuYW55KCkgb3Igc3RlcHMgPj0gSE9SSVpPTjoKICAgICAgICAgICAgc3RvcCA9IFRydWU7IGJyZWFrCiAgICBpZiBzdG9wOgogICAgICAgIGJyZWFrCnNjID0gaW5mb1siZmluYWxfaW5mbyJdWyJlcGlzb2RlIl1bInN1Y2Nlc3NfYXRfZW5kIl0uZmxvYXQoKS5tZWFuKCkuaXRlbSgpIGlmICJmaW5hbF9pbmZvIiBpbiBpbmZvIGVsc2UgIm4vYSIKcHJpbnQoInN0ZXBzOiIsIHN0ZXBzLCAiZmluYWwgc3VjY2Vzc19hdF9lbmQ6Iiwgc2MpCmVudnMuY2xvc2UoKQpmb3IgZXh0cmEgaW4gc29ydGVkKGdsb2IuZ2xvYihvcy5wYXRoLmpvaW4oVklERElSLCAiKi5tcDQiKSkpWzE6XToKICAgIG9zLnJlbW92ZShleHRyYSkKcHJpbnQoInZpZGVvIC0+Iiwgc29ydGVkKGdsb2IuZ2xvYihvcy5wYXRoLmpvaW4oVklERElSLCAiKi5tcDQiKSkpKQo='}, 'il/baselines/diffusion_policy/setup.py': {'base_sha256': '3a5a5c31fbf5736a807e6f8d048e64bd87336d639dd2e9da33ae85456fc0e423', 'sha256': '3a5a5c31fbf5736a807e6f8d048e64bd87336d639dd2e9da33ae85456fc0e423', 'content_b64': 'ZnJvbSBzZXR1cHRvb2xzIGltcG9ydCBzZXR1cCwgZmluZF9wYWNrYWdlcwoKc2V0dXAoCiAgICBuYW1lPSJkaWZmdXNpb25fcG9saWN5IiwKICAgIHZlcnNpb249IjAuMS4wIiwKICAgIHBhY2thZ2VzPWZpbmRfcGFja2FnZXMoKSwKICAgIGluc3RhbGxfcmVxdWlyZXM9WwogICAgICAgICJkaWZmdXNlcnMiLAogICAgICAgICJ0ZW5zb3Jib2FyZCIsCiAgICAgICAgIndhbmRiIiwKICAgICAgICAibWFuaV9za2lsbCIKICAgIF0sCiAgICBkZXNjcmlwdGlvbj0iQSBtaW5pbWFsIHNldHVwIGZvciBEaWZmdXNpb24gUG9saWN5IGZvciBNYW5pU2tpbGwiLAogICAgbG9uZ19kZXNjcmlwdGlvbj1vcGVuKCJSRUFETUUubWQiKS5yZWFkKCksCiAgICBsb25nX2Rlc2NyaXB0aW9uX2NvbnRlbnRfdHlwZT0idGV4dC9tYXJrZG93biIsCikK'}, 'il/baselines/diffusion_policy/train.py': {'base_sha256': '7348088687b5739c6a13c7349304d5068ef373100eb5a4f3136e9fb36c3cce28', 'sha256': '7348088687b5739c6a13c7349304d5068ef373100eb5a4f3136e9fb36c3cce28', 'content_b64': '...'}, 'il/baselines/diffusion_policy/train_rgbd.py': {'base_sha256': 'a6548a00c63cf25fd3dc7568d6d6670ac3d2deda56840d0a3d438eaf122e9e45', 'sha256': '2a776d7d268f53372bde102be90f9a54c6df71fb6040f7b4de73c0399e65b564', 'content_b64': '...'}, 'il/conf/method/act_rgb.yaml': {'base_sha256': 'c42c2c154b58cb86bf0397fa8861a38e3d3fb4f2a2e0b252568267979015ed2d', 'sha256': 'c42c2c154b58cb86bf0397fa8861a38e3d3fb4f2a2e0b252568267979015ed2d', 'content_b64': 'IyBAcGFja2FnZSBfZ2xvYmFsXwojIEhFREdFIExJTkUgKDFzdC1wbGFjZSBhcHByb2FjaCwgdmVuZG9yZWQgZnJvbSBzYWhpbHJhanB1cmthcjAzL2hhY2thdGhvbi1kZXYtYWN0LW1lcmdlZCkuCiMgUnVuIHdpdGggYW4gZXhwbGljaXQgZXBpc29kZSBidWRnZXQ6IHB5dGhvbiBpbC90cmFpbi5weSBtZXRob2Q9YWN0X3JnYiBkZW1vX2Rpcj1lYXN5IG1heF9lcGlzb2RlX3N0ZXBzPTI1MAojIFRPRE8oUzMpOiBpdHMgZGF0YXNldCBsb2FkZXIgcmVzaXplcyBldmVyeSBmcmFtZSB0byAyMjR4MjI0IHVpbnQ4IGluIHN5c3RlbSBSQU0gKGVhc3kgfjMuNSBHQiwgaGFyZCB+MTIgR0IpCiMgLT4gcG9ydCB0byB0aGUgc3RyZWFtaW5nIGxvYWRlciArIG9uLXRoZS1mbHkgcmVzaXplIGJlZm9yZSB1c2luZyBpdCBvbiBtZWRpdW0vaGFyZC4KIyBSR0IgQUNUIChBY3Rpb24gQ2h1bmtpbmcgVHJhbnNmb3JtZXIsIENWQUUgKyBERVRSIGJhY2tib25lKTogaW1hZ2UgKyByb2JvdCBwcm9wcmlvY2VwdGlvbiBvbmx5CiMgKE5PIHByaXZpbGVnZWQgc3RhdGUsIE5PIGRlcHRoIC0tIFdhcmVob3VzZVNvcnQncyBvYnMgY29udHJhY3QgaGFzIG5vICdkZXB0aCcga2V5KS4gUmVzaXplcyB0aGUKIyAxMjh4MTI4IHNjZW5lIGNhbWVyYSB1cCB0byAyMjR4MjI0IGludGVybmFsbHkgZm9yIHRoZSBJbWFnZU5ldC1wcmV0cmFpbmVkIFJlc05ldDE4IGJhY2tib25lLgpiYXNlbGluZV9kaXI6IGFjdApzY3JpcHQ6IHRyYWluX3JnYmQucHkKZGVtb19raW5kOiByZ2IKZmxhZ3M6CiAgaW5jbHVkZV9kZXB0aDogZmFsc2UKICBvYnNfY2FtZXJhOiBzY2VuZQogIGJhY2tib25lOiByZXNuZXQxOAogIG51bV9xdWVyaWVzOiAzMAogIHRlbXBvcmFsX2FnZzogdHJ1ZQogIHRvdGFsX2l0ZXJzOiAzMDAwMAogIGJhdGNoX3NpemU6IDMyICAjIERFVFIrQ1ZBRSBvbiAyMjR4MjI0IGltYWdlcyBpcyBmYXIgaGVhdmllciBwZXItc2FtcGxlIHRoYW4gRFAvU0FDJ3MgQ05OcwogIGxyOiAxZS00CiAga2xfd2VpZ2h0OiAxMAogIG51bV9ldmFsX2VudnM6IDgKICBudW1fZXZhbF9lcGlzb2RlczogMTYKICBldmFsX2ZyZXE6IDUwMDAKICBsb2dfZnJlcTogMTAwMAogIGNhcHR1cmVfdmlkZW86IGZhbHNlICAgIyBSZWNvcmRFcGlzb2RlIGJ1ZmZlcnMgd2hvbGUgZXBpc29kZXMgaW4gUkFNIC0+IE9PTSBvbiBDb2xhYgogIGV4cF9uYW1lOiB3YXJlaG91c2VfcmdiX2FjdAo='}, 'il/conf/method/dp.yaml': {'base_sha256': 'd03ae9e41c13dc32824c081a433bb4d4c8d62610f3e36678c8d28a55503b5204', 'sha256': 'd03ae9e41c13dc32824c081a433bb4d4c8d62610f3e36678c8d28a55503b5204', 'content_b64': 'IyBAcGFja2FnZSBfZ2xvYmFsXwojIFN0YXRlIERpZmZ1c2lvbiBQb2xpY3kgKE1BSU4gdHJhY2spOiBwcml2aWxlZ2VkIGxvdy1kaW0gc3RhdGUgb2JzLiBUaGUgZW5kLXRvLWVuZCBiYXNlbGluZQojIHdpcmVkIGludG8gZXZhbC5weS4gU3RhdGUgaW5wdXQgaXMgcGFyY2VsLWNvdW50LXNwZWNpZmljIC0+IHRyYWluIG9uZSBjaGVja3BvaW50IFBFUiBsZXZlbC4KYmFzZWxpbmVfZGlyOiBkaWZmdXNpb25fcG9saWN5CnNjcmlwdDogdHJhaW4ucHkKZGVtb19raW5kOiBzdGF0ZQpmbGFnczoKICB0b3RhbF9pdGVyczogMzAwMDAKICBiYXRjaF9zaXplOiAyNTYKICBvYnNfaG9yaXpvbjogMgogIGFjdF9ob3Jpem9uOiA4CiAgcHJlZF9ob3Jpem9uOiAxNgogIG51bV9ldmFsX2VudnM6IDgKICBudW1fZXZhbF9lcGlzb2RlczogMTYKICBldmFsX2ZyZXE6IDUwMDAKICBsb2dfZnJlcTogMTAwMAogIHNhdmVfZnJlcTogMTAwMDAKICBjYXB0dXJlX3ZpZGVvOiBmYWxzZSAgICMgUkFNOiBSZWNvcmRFcGlzb2RlIGJ1ZmZlcnMgd2hvbGUgZXBpc29kZXMKICBjbGlwX2FjdGlvbnM6IHRydWUKICBldmFsX2luZmVyZW5jZV9zdGVwczogMTYKICBleHBfbmFtZTogd2FyZWhvdXNlX3N0YXRlX2RwX2Vhc3kK'}, 'il/conf/method/dp_rgb.yaml': {'base_sha256': 'ca9cc91123dda7195645206f3323218b38fd822bed46a242b2b48a5f8f301249', 'sha256': 'ca9cc91123dda7195645206f3323218b38fd822bed46a242b2b48a5f8f301249', 'content_b64': 'IyBAcGFja2FnZSBfZ2xvYmFsXwojIFJHQiAoc2NlbmUtY2FtKSBEaWZmdXNpb24gUG9saWN5OiBpbWFnZSArIHJvYm90IHByb3ByaW9jZXB0aW9uIG9ubHkgKE5PIHByaXZpbGVnZWQgc3RhdGUpLgojIFJlc05ldDE4ICsgU3BhdGlhbFNvZnRtYXggZW5jb2RlciwgRHJRIHJhbmRvbS1zaGlmdCBhdWcsIGNsaXBwZWQgZGVtbyBhY3Rpb25zLCBmcDE2IGVuY29kZXIuCiMgR2VuZXJpYyBjb25maWcgLS0gcHJlZmVyIGRwX3JnYl9lYXN5IC8gZHBfcmdiX21lZGl1bSAvIGRwX3JnYl9oYXJkICh0aGV5IHNldCB0aGUgZXBpc29kZSBidWRnZXQpLgojICAgcHl0aG9uIGlsL3RyYWluLnB5IG1ldGhvZD1kcF9yZ2IgZGVtb19kaXI9ZWFzeSBtYXhfZXBpc29kZV9zdGVwcz0yNTAgZmxhZ3MuZXhwX25hbWU9cmdiX2RwX2Vhc3kKYmFzZWxpbmVfZGlyOiBkaWZmdXNpb25fcG9saWN5CnNjcmlwdDogdHJhaW5fcmdiZC5weQpkZW1vX2tpbmQ6IHJnYgpmbGFnczoKICBvYnNfbW9kZTogcmdiCiAgb2JzX2NhbWVyYTogc2NlbmUKICB2aXN1YWxfZW5jb2RlcjogcmVzbmV0MTggICAjIHJlc25ldDE4IHwgcGxhaW5fY29udgogIG51bV9rcDogMzIKICB0b3RhbF9pdGVyczogMzAwMDAKICBiYXRjaF9zaXplOiAxMjgKICBscjogMS4wZS00CiAgb2JzX2hvcml6b246IDIKICBhY3RfaG9yaXpvbjogOAogIHByZWRfaG9yaXpvbjogMTYKICBudW1fZGlmZnVzaW9uX2l0ZXJzOiAxMDAKICBldmFsX2luZmVyZW5jZV9zdGVwczogMTYgICAjIGFsc28gdGhlIGRlcGxveW1lbnQgZGVmYXVsdCBzdG9yZWQgaW4gdGhlIGNoZWNrcG9pbnQgY29uZmlnCiAgY2xpcF9hY3Rpb25zOiB0cnVlCiAgaW1hZ2VfYXVnX3BhZDogNAogIHByb3ByaW9fbm9pc2Vfc3RkOiAwLjAKICBhbXA6IHRydWUKICB0b3JjaF9kZXRlcm1pbmlzdGljOiBmYWxzZQogIG51bV9ldmFsX2VudnM6IDgKICBudW1fZXZhbF9lcGlzb2RlczogMzIKICBldmFsX2ZyZXE6IDUwMDAKICBza2lwX2luaXRpYWxfZXZhbDogdHJ1ZQogIGxvZ19mcmVxOiA1MDAKICBzYXZlX2ZyZXE6IDI1MDAKICBjYXB0dXJlX3ZpZGVvOiBmYWxzZSAgICAgICAjIFJlY29yZEVwaXNvZGUgYnVmZmVycyB3aG9sZSBlcGlzb2RlcyBpbiBSQU0gLT4gT09NIG9uIENvbGFiCiAgbnVtX2RlbW9zOiBudWxsICAgICAgICAgICAgIyBwZXIgZmlsZTsgbnVsbCA9IGFsbCAyMDAKICBja3B0X2RpcjogbnVsbCAgICAgICAgICAgICAjIGUuZy4gL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS9tYXJzby9ja3B0cy88ZXhwPjsgbnVsbCA9IHJ1bnMvPGV4cD4vY2hlY2twb2ludHMKICByZXN1bWU6IG51bGwgICAgICAgICAgICAgICAjIHBhdGggdG8gbGF0ZXN0LnB0IHRvIGNvbnRpbnVlIGEgcnVuIChzYW1lIGh5cGVycGFyYW1ldGVycykKICBleHBfbmFtZV90aW1lc3RhbXA6IGZhbHNlICAjIGFwcGVuZCBfTU1ERC1ISE1NIHRvIGV4cF9uYW1lCiAgZXhwX25hbWU6IHdhcmVob3VzZV9yZ2JfZHAK'}, 'il/conf/method/dp_rgb_easy.yaml': {'base_sha256': '7363d549918d217754291ce5e524c4c79e4ac317d85604e48d9944c03858ea36', 'sha256': '7363d549918d217754291ce5e524c4c79e4ac317d85604e48d9944c03858ea36', 'content_b64': 'IyBAcGFja2FnZSBfZ2xvYmFsXwojIFJHQiAoc2NlbmUtY2FtKSBEaWZmdXNpb24gUG9saWN5OiBpbWFnZSArIHJvYm90IHByb3ByaW9jZXB0aW9uIG9ubHkgKE5PIHByaXZpbGVnZWQgc3RhdGUpLgojIFJlc05ldDE4ICsgU3BhdGlhbFNvZnRtYXggZW5jb2RlciwgRHJRIHJhbmRvbS1zaGlmdCBhdWcsIGNsaXBwZWQgZGVtbyBhY3Rpb25zLCBmcDE2IGVuY29kZXIuCiMgTGV2ZWwgcHJlc2V0OiBlYXN5IGRlbW9zLCAyNTAtc3RlcCBlcGlzb2RlcyBmb3IgdGhlIHRyYWluaW5nLXRpbWUgZXZhbC4KIyAgIHB5dGhvbiBpbC90cmFpbi5weSBtZXRob2Q9ZHBfcmdiX2Vhc3kgICAgICAgICAgICAgICAgICAgICAgIyAoKyBmbGFncy5ja3B0X2Rpcj0vY29udGVudC9kcml2ZS8uLi4gb24gQ29sYWIpCmJhc2VsaW5lX2RpcjogZGlmZnVzaW9uX3BvbGljeQpzY3JpcHQ6IHRyYWluX3JnYmQucHkKZGVtb19raW5kOiByZ2IKZGVtb19kaXI6IGVhc3kKbWF4X2VwaXNvZGVfc3RlcHM6IDI1MApmbGFnczoKICBvYnNfbW9kZTogcmdiCiAgb2JzX2NhbWVyYTogc2NlbmUKICB2aXN1YWxfZW5jb2RlcjogcmVzbmV0MTggICAjIHJlc25ldDE4IHwgcGxhaW5fY29udgogIG51bV9rcDogMzIKICB0b3RhbF9pdGVyczogMzAwMDAKICBiYXRjaF9zaXplOiAxMjgKICBscjogMS4wZS00CiAgb2JzX2hvcml6b246IDIKICBhY3RfaG9yaXpvbjogOAogIHByZWRfaG9yaXpvbjogMTYKICBudW1fZGlmZnVzaW9uX2l0ZXJzOiAxMDAKICBldmFsX2luZmVyZW5jZV9zdGVwczogMTYgICAjIGFsc28gdGhlIGRlcGxveW1lbnQgZGVmYXVsdCBzdG9yZWQgaW4gdGhlIGNoZWNrcG9pbnQgY29uZmlnCiAgY2xpcF9hY3Rpb25zOiB0cnVlCiAgaW1hZ2VfYXVnX3BhZDogNAogIHByb3ByaW9fbm9pc2Vfc3RkOiAwLjAKICBhbXA6IHRydWUKICB0b3JjaF9kZXRlcm1pbmlzdGljOiBmYWxzZQogIG51bV9ldmFsX2VudnM6IDgKICBudW1fZXZhbF9lcGlzb2RlczogMzIKICBldmFsX2ZyZXE6IDUwMDAKICBza2lwX2luaXRpYWxfZXZhbDogdHJ1ZQogIGxvZ19mcmVxOiA1MDAKICBzYXZlX2ZyZXE6IDI1MDAKICBjYXB0dXJlX3ZpZGVvOiBmYWxzZSAgICAgICAjIFJlY29yZEVwaXNvZGUgYnVmZmVycyB3aG9sZSBlcGlzb2RlcyBpbiBSQU0gLT4gT09NIG9uIENvbGFiCiAgbnVtX2RlbW9zOiBudWxsICAgICAgICAgICAgIyBwZXIgZmlsZTsgbnVsbCA9IGFsbCAyMDAKICBja3B0X2RpcjogbnVsbCAgICAgICAgICAgICAjIGUuZy4gL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS9tYXJzby9ja3B0cy88ZXhwPjsgbnVsbCA9IHJ1bnMvPGV4cD4vY2hlY2twb2ludHMKICByZXN1bWU6IG51bGwgICAgICAgICAgICAgICAjIHBhdGggdG8gbGF0ZXN0LnB0IHRvIGNvbnRpbnVlIGEgcnVuIChzYW1lIGh5cGVycGFyYW1ldGVycykKICBleHBfbmFtZV90aW1lc3RhbXA6IGZhbHNlICAjIGFwcGVuZCBfTU1ERC1ISE1NIHRvIGV4cF9uYW1lCiAgZXhwX25hbWU6IHJnYl9kcF9lYXN5Cg=='}, 'il/conf/method/dp_rgb_hard.yaml': {'base_sha256': '257a155cbfd710c68be47c78856b7777220041cb498b2f5b13347f9579838e8f', 'sha256': '257a155cbfd710c68be47c78856b7777220041cb498b2f5b13347f9579838e8f', 'content_b64': 'IyBAcGFja2FnZSBfZ2xvYmFsXwojIFJHQiAoc2NlbmUtY2FtKSBEaWZmdXNpb24gUG9saWN5OiBpbWFnZSArIHJvYm90IHByb3ByaW9jZXB0aW9uIG9ubHkgKE5PIHByaXZpbGVnZWQgc3RhdGUpLgojIFJlc05ldDE4ICsgU3BhdGlhbFNvZnRtYXggZW5jb2RlciwgRHJRIHJhbmRvbS1zaGlmdCBhdWcsIGNsaXBwZWQgZGVtbyBhY3Rpb25zLCBmcDE2IGVuY29kZXIuCiMgTGV2ZWwgcHJlc2V0OiBoYXJkIGRlbW9zLCA4MDAtc3RlcCBlcGlzb2RlcyBmb3IgdGhlIHRyYWluaW5nLXRpbWUgZXZhbC4KIyAgIHB5dGhvbiBpbC90cmFpbi5weSBtZXRob2Q9ZHBfcmdiX2hhcmQgICAgICAgICAgICAgICAgICAgICAgIyAoKyBmbGFncy5ja3B0X2Rpcj0vY29udGVudC9kcml2ZS8uLi4gb24gQ29sYWIpCmJhc2VsaW5lX2RpcjogZGlmZnVzaW9uX3BvbGljeQpzY3JpcHQ6IHRyYWluX3JnYmQucHkKZGVtb19raW5kOiByZ2IKZGVtb19kaXI6IGhhcmQKbWF4X2VwaXNvZGVfc3RlcHM6IDgwMApmbGFnczoKICBvYnNfbW9kZTogcmdiCiAgb2JzX2NhbWVyYTogc2NlbmUKICB2aXN1YWxfZW5jb2RlcjogcmVzbmV0MTggICAjIHJlc25ldDE4IHwgcGxhaW5fY29udgogIG51bV9rcDogMzIKICB0b3RhbF9pdGVyczogNDAwMDAKICBiYXRjaF9zaXplOiAxMjgKICBscjogMS4wZS00CiAgb2JzX2hvcml6b246IDIKICBhY3RfaG9yaXpvbjogOAogIHByZWRfaG9yaXpvbjogMTYKICBudW1fZGlmZnVzaW9uX2l0ZXJzOiAxMDAKICBldmFsX2luZmVyZW5jZV9zdGVwczogMTYgICAjIGFsc28gdGhlIGRlcGxveW1lbnQgZGVmYXVsdCBzdG9yZWQgaW4gdGhlIGNoZWNrcG9pbnQgY29uZmlnCiAgY2xpcF9hY3Rpb25zOiB0cnVlCiAgaW1hZ2VfYXVnX3BhZDogNAogIHByb3ByaW9fbm9pc2Vfc3RkOiAwLjAKICBhbXA6IHRydWUKICB0b3JjaF9kZXRlcm1pbmlzdGljOiBmYWxzZQogIG51bV9ldmFsX2VudnM6IDgKICBudW1fZXZhbF9lcGlzb2RlczogMzIKICBldmFsX2ZyZXE6IDUwMDAKICBza2lwX2luaXRpYWxfZXZhbDogdHJ1ZQogIGxvZ19mcmVxOiA1MDAKICBzYXZlX2ZyZXE6IDI1MDAKICBjYXB0dXJlX3ZpZGVvOiBmYWxzZSAgICAgICAjIFJlY29yZEVwaXNvZGUgYnVmZmVycyB3aG9sZSBlcGlzb2RlcyBpbiBSQU0gLT4gT09NIG9uIENvbGFiCiAgbnVtX2RlbW9zOiBudWxsICAgICAgICAgICAgIyBwZXIgZmlsZTsgbnVsbCA9IGFsbCAyMDAKICBja3B0X2RpcjogbnVsbCAgICAgICAgICAgICAjIGUuZy4gL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS9tYXJzby9ja3B0cy88ZXhwPjsgbnVsbCA9IHJ1bnMvPGV4cD4vY2hlY2twb2ludHMKICByZXN1bWU6IG51bGwgICAgICAgICAgICAgICAjIHBhdGggdG8gbGF0ZXN0LnB0IHRvIGNvbnRpbnVlIGEgcnVuIChzYW1lIGh5cGVycGFyYW1ldGVycykKICBleHBfbmFtZV90aW1lc3RhbXA6IGZhbHNlICAjIGFwcGVuZCBfTU1ERC1ISE1NIHRvIGV4cF9uYW1lCiAgZXhwX25hbWU6IHJnYl9kcF9oYXJkCg=='}, 'il/conf/method/dp_rgb_medium.yaml': {'base_sha256': '293b0136544665a3e6b5da7497d781aaeb63b4b75b926bf4b9211d10108774be', 'sha256': '293b0136544665a3e6b5da7497d781aaeb63b4b75b926bf4b9211d10108774be', 'content_b64': 'IyBAcGFja2FnZSBfZ2xvYmFsXwojIFJHQiAoc2NlbmUtY2FtKSBEaWZmdXNpb24gUG9saWN5OiBpbWFnZSArIHJvYm90IHByb3ByaW9jZXB0aW9uIG9ubHkgKE5PIHByaXZpbGVnZWQgc3RhdGUpLgojIFJlc05ldDE4ICsgU3BhdGlhbFNvZnRtYXggZW5jb2RlciwgRHJRIHJhbmRvbS1zaGlmdCBhdWcsIGNsaXBwZWQgZGVtbyBhY3Rpb25zLCBmcDE2IGVuY29kZXIuCiMgTGV2ZWwgcHJlc2V0OiBtZWRpdW0gZGVtb3MsIDUwMC1zdGVwIGVwaXNvZGVzIGZvciB0aGUgdHJhaW5pbmctdGltZSBldmFsLgojICAgcHl0aG9uIGlsL3RyYWluLnB5IG1ldGhvZD1kcF9yZ2JfbWVkaXVtICAgICAgICAgICAgICAgICAgICAgICMgKCsgZmxhZ3MuY2twdF9kaXI9L2NvbnRlbnQvZHJpdmUvLi4uIG9uIENvbGFiKQpiYXNlbGluZV9kaXI6IGRpZmZ1c2lvbl9wb2xpY3kKc2NyaXB0OiB0cmFpbl9yZ2JkLnB5CmRlbW9fa2luZDogcmdiCmRlbW9fZGlyOiBtZWRpdW0KbWF4X2VwaXNvZGVfc3RlcHM6IDUwMApmbGFnczoKICBvYnNfbW9kZTogcmdiCiAgb2JzX2NhbWVyYTogc2NlbmUKICB2aXN1YWxfZW5jb2RlcjogcmVzbmV0MTggICAjIHJlc25ldDE4IHwgcGxhaW5fY29udgogIG51bV9rcDogMzIKICB0b3RhbF9pdGVyczogMzAwMDAKICBiYXRjaF9zaXplOiAxMjgKICBscjogMS4wZS00CiAgb2JzX2hvcml6b246IDIKICBhY3RfaG9yaXpvbjogOAogIHByZWRfaG9yaXpvbjogMTYKICBudW1fZGlmZnVzaW9uX2l0ZXJzOiAxMDAKICBldmFsX2luZmVyZW5jZV9zdGVwczogMTYgICAjIGFsc28gdGhlIGRlcGxveW1lbnQgZGVmYXVsdCBzdG9yZWQgaW4gdGhlIGNoZWNrcG9pbnQgY29uZmlnCiAgY2xpcF9hY3Rpb25zOiB0cnVlCiAgaW1hZ2VfYXVnX3BhZDogNAogIHByb3ByaW9fbm9pc2Vfc3RkOiAwLjAKICBhbXA6IHRydWUKICB0b3JjaF9kZXRlcm1pbmlzdGljOiBmYWxzZQogIG51bV9ldmFsX2VudnM6IDgKICBudW1fZXZhbF9lcGlzb2RlczogMzIKICBldmFsX2ZyZXE6IDUwMDAKICBza2lwX2luaXRpYWxfZXZhbDogdHJ1ZQogIGxvZ19mcmVxOiA1MDAKICBzYXZlX2ZyZXE6IDI1MDAKICBjYXB0dXJlX3ZpZGVvOiBmYWxzZSAgICAgICAjIFJlY29yZEVwaXNvZGUgYnVmZmVycyB3aG9sZSBlcGlzb2RlcyBpbiBSQU0gLT4gT09NIG9uIENvbGFiCiAgbnVtX2RlbW9zOiBudWxsICAgICAgICAgICAgIyBwZXIgZmlsZTsgbnVsbCA9IGFsbCAyMDAKICBja3B0X2RpcjogbnVsbCAgICAgICAgICAgICAjIGUuZy4gL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS9tYXJzby9ja3B0cy88ZXhwPjsgbnVsbCA9IHJ1bnMvPGV4cD4vY2hlY2twb2ludHMKICByZXN1bWU6IG51bGwgICAgICAgICAgICAgICAjIHBhdGggdG8gbGF0ZXN0LnB0IHRvIGNvbnRpbnVlIGEgcnVuIChzYW1lIGh5cGVycGFyYW1ldGVycykKICBleHBfbmFtZV90aW1lc3RhbXA6IGZhbHNlICAjIGFwcGVuZCBfTU1ERC1ISE1NIHRvIGV4cF9uYW1lCiAgZXhwX25hbWU6IHJnYl9kcF9tZWRpdW0K'}, 'il/conf/train.yaml': {'base_sha256': '49c6db7cdb077f1a02f399dface2d8cbecc7a32e2d7404ac2233e68341b92568', 'sha256': '49c6db7cdb077f1a02f399dface2d8cbecc7a32e2d7404ac2233e68341b92568', 'content_b64': 'IyBIeWRyYSBjb25maWcgZm9yIHRoZSBJTCB0cmFpbmluZyBkaXNwYXRjaGVyIChpbC90cmFpbi5weSkuCiMgTWFpbiB0cmFjayBpcyBTVEFURSAoZGVmYXVsdCBtZXRob2Q9ZHApOyBkcF9yZ2IgaXMgdGhlIG9wdGlvbmFsIGltYWdlIHRyYWNrLgojIFBpY2sgYSBtZXRob2Qgd2l0aCBtZXRob2Q9Li4uIGFuZCBvdmVycmlkZSBhbnkgaHlwZXJwYXJhbWV0ZXIgb24gdGhlIENMSSwgZS5nLgojICAgcGl4aSBydW4gcHl0aG9uIGlsL3RyYWluLnB5ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBkZWZhdWx0OiBkcCAoc3RhdGUgRFAsIG1haW4gdHJhY2spCiMgICBwaXhpIHJ1biBweXRob24gaWwvdHJhaW4ucHkgbWV0aG9kPWRwX3JnYiAgICAgICAgICAgICAgICMgb3B0aW9uYWwgaW1hZ2UgdHJhY2sKIyAgIHBpeGkgcnVuIHB5dGhvbiBpbC90cmFpbi5weSBtZXRob2Q9ZHAgZmxhZ3MudG90YWxfaXRlcnM9ODAwMCBmbGFncy5wcmVkX2hvcml6b249MzIKZGVmYXVsdHM6CiAgLSBfc2VsZl8KICAtIG1ldGhvZDogZHAgICAjIGRlZmF1bHQ6IHN0YXRlIERpZmZ1c2lvbiBQb2xpY3kgKG1haW4gdHJhY2spCgojIHNoYXJlZCBhY3Jvc3MgYWxsIG1ldGhvZHMKZW52X2lkOiBXYXJlaG91c2VTb3J0LXYxCmNvbnRyb2xfbW9kZTogcGRfZWVfZGVsdGFfcG9zCnNpbV9iYWNrZW5kOiBncHUKbWF4X2VwaXNvZGVfc3RlcHM6IDIwMCAgICAgICAgICAgICMgcGVyLWxldmVsIG1ldGhvZCB5YW1scyAoZHBfcmdiX2Vhc3kvbWVkaXVtL2hhcmQpIG92ZXJyaWRlIHRoaXMKZGVtb19kaXI6IGVhc3kgICAgICAgICAgICAgICAgICAjIGlsL2RlbW9zLzxkaXI+IGRhdGFzZXQocyk7IGEgbGlzdCA9IG1peGVkLWxldmVsIHRyYWluaW5nCmRlbW9fcGF0aDogbnVsbCAgICAgICAgICAgICAgICAgIyBleHBsaWNpdCAuaDUgcGF0aChzKSAoc3RyIG9yIGxpc3QpOyBudWxsIC0+IGZyb20gZGVtb19kaXIgKyBtZXRob2QuZGVtb19raW5kCgpoeWRyYToKICBvdXRwdXRfc3ViZGlyOiBudWxsCiAgcnVuOgogICAgZGlyOiAuCg=='}, 'il/train.py': {'base_sha256': 'ff2379125b28bc19b12fc4c1f9067232af78da731a65755616005de7f5f82f2e', 'sha256': 'ff2379125b28bc19b12fc4c1f9067232af78da731a65755616005de7f5f82f2e', 'content_b64': 'IiIiSHlkcmEgZGlzcGF0Y2hlciBmb3IgdGhlIElMIGJhc2VsaW5lcyAoY29uc2lzdGVudCB3aXRoIHRoZSByZXBvJ3MgSHlkcmEgc3R5bGUpLgoKVGhpcyBkb2VzIE5PVCByZWltcGxlbWVudCBhbnkgbGVhcm5pbmcgbG9naWMuIEl0IGxvYWRzIGEgbWV0aG9kIGNvbmZpZyAoaWwvY29uZi9tZXRob2QvKi55YW1sKSwKY29udmVydHMgaXRzIGBmbGFnc2AgaW50byB0aGUgdmVuZG9yZWQgYmFzZWxpbmUgc2NyaXB0J3MgQ0xJICh1bmRlcnNjb3JlIGtleXMgLT4gLS1oeXBoZW4tZmxhZ3M7CmJvb2xlYW5zIC0+IC0tZmxhZyAvIC0tbm8tZmxhZyksIGFuZCBydW5zIHRoYXQgcmVhbCBzY3JpcHQgd2l0aCB0aGUgcmlnaHQgZGVtbyBwYXRoICsgc2hhcmVkIGFyZ3MuCgogIHB5dGhvbiBpbC90cmFpbi5weSBtZXRob2Q9ZHBfcmdiX2Vhc3kgICAgICAgICAgICAgICAgICAgICAjIHJnYiBEUCwgZWFzeSBkZW1vcywgMjUwLXN0ZXAgZXZhbAogIHB5dGhvbiBpbC90cmFpbi5weSBtZXRob2Q9ZHBfcmdiIGRlbW9fZGlyPVtoYXJkLG1lZGl1bV0gICAjIG1peGVkLWxldmVsIHRyYWluaW5nIChmaXJzdCA9IGV2YWwgc2NlbmUpCiAgcGl4aSBydW4gcHl0aG9uIGlsL3RyYWluLnB5IG1ldGhvZD1kcF9yZ2IgZmxhZ3MudG90YWxfaXRlcnM9ODAwMCBmbGFncy5ldmFsX2ZyZXE9NDAwMAoKT3V0cHV0cyAoY2hlY2twb2ludHMgKyB0ZW5zb3Jib2FyZCArIGV2YWwgdmlkZW9zKSBsYW5kIHVuZGVyCiAgaWwvYmFzZWxpbmVzLzxiYXNlbGluZV9kaXI+L3J1bnMvPGZsYWdzLmV4cF9uYW1lPi8KRXZhbHVhdGUgdmlhOiBwaXhpIHJ1biBweXRob24gZXZhbC5weSAuLi4gcG9saWN5PXdhcmVob3VzZV9zb3J0LmlsX3BvbGljeTpsb2FkX2RwX3JnYiAuLi4KIiIiCgppbXBvcnQgb3MKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwoKaW1wb3J0IGh5ZHJhCmZyb20gb21lZ2Fjb25mIGltcG9ydCBPbWVnYUNvbmYKCkhFUkUgPSBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkKCgpkZWYgX2RlbW9fcGF0aChkZW1vX2Rpciwga2luZCk6CiAgICBkID0gb3MucGF0aC5qb2luKEhFUkUsICJkZW1vcyIsIGRlbW9fZGlyKQogICAgcmV0dXJuIG9zLnBhdGguam9pbihkLCBmInRyYWplY3Rvcnkue2tpbmR9LnBkX2VlX2RlbHRhX3Bvcy5waHlzeF9jdWRhLmg1IikKCgpkZWYgX2FzX2xpc3QoeCk6CiAgICBpZiB4IGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIFtdCiAgICBpZiBpc2luc3RhbmNlKHgsIHN0cik6CiAgICAgICAgcmV0dXJuIFt4XQogICAgcmV0dXJuIGxpc3QoeCkKCgpkZWYgX2RlbW9fcGF0aHMoY2ZnKToKICAgICIiImRlbW9fcGF0aCAoc3RyIHwgbGlzdCkgd2luczsgZWxzZSBvbmUgZmlsZSBwZXIgZGVtb19kaXIgZW50cnkgKHN0ciB8IGxpc3QsIGUuZy4gZGVtb19kaXI9W2hhcmQsbWVkaXVtXSkuIiIiCiAgICBwYXRocyA9IF9hc19saXN0KGNmZy5nZXQoImRlbW9fcGF0aCIpKQogICAgaWYgbm90IHBhdGhzOgogICAgICAgIHBhdGhzID0gW19kZW1vX3BhdGgoZCwgY2ZnLmRlbW9fa2luZCkgZm9yIGQgaW4gX2FzX2xpc3QoY2ZnLmdldCgiZGVtb19kaXIiLCAiZWFzeSIpKV0KICAgIHJldHVybiBwYXRocwoKCmRlZiBfZmxhZ3NfdG9fY2xpKGZsYWdzOiBkaWN0KToKICAgICIiIm1ldGhvZC5mbGFncyAodW5kZXJzY29yZSBrZXlzKSAtPiB2ZW5kb3JlZCB0eXJvIENMSSBhcmdzLgogICAgYm9vbCBUcnVlIC0+IC0tZmxhZywgYm9vbCBGYWxzZSAtPiAtLW5vLWZsYWc7IGV2ZXJ5dGhpbmcgZWxzZSAtPiAtLWZsYWcgdmFsdWUuIiIiCiAgICBjbGkgPSBbXQogICAgZm9yIGtleSwgdmFsIGluIGZsYWdzLml0ZW1zKCk6CiAgICAgICAgZmxhZyA9ICItLSIgKyBrZXkucmVwbGFjZSgiXyIsICItIikKICAgICAgICBpZiBpc2luc3RhbmNlKHZhbCwgYm9vbCk6CiAgICAgICAgICAgIGNsaS5hcHBlbmQoZmxhZyBpZiB2YWwgZWxzZSAiLS1uby0iICsga2V5LnJlcGxhY2UoIl8iLCAiLSIpKQogICAgICAgIGVsaWYgaXNpbnN0YW5jZSh2YWwsIChsaXN0LCB0dXBsZSkpOgogICAgICAgICAgICBjbGkgKz0gW2ZsYWddICsgW3N0cih2KSBmb3IgdiBpbiB2YWxdCiAgICAgICAgZWxpZiB2YWwgaXMgTm9uZToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBlbHNlOgogICAgICAgICAgICBjbGkgKz0gW2ZsYWcsIHN0cih2YWwpXQogICAgcmV0dXJuIGNsaQoKCkBoeWRyYS5tYWluKHZlcnNpb25fYmFzZT1Ob25lLCBjb25maWdfcGF0aD0iY29uZiIsIGNvbmZpZ19uYW1lPSJ0cmFpbiIpCmRlZiBtYWluKGNmZyk6CiAgICBkZW1vcyA9IF9kZW1vX3BhdGhzKGNmZykKICAgIGZvciBkZW1vIGluIGRlbW9zOgogICAgICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhkZW1vKToKICAgICAgICAgICAgc3lzLmV4aXQoZiJkZW1vIGRhdGFzZXQgbm90IGZvdW5kOiB7ZGVtb31cbiAgcnVuOiBweXRob24gaWwvZG93bmxvYWRfZGVtb3MucHkgIChvciBpbC9nZW5fZGVtb3MucHkpIikKICAgIGNvbW1vbiA9IFsiLS1lbnYtaWQiLCBjZmcuZW52X2lkLCAiLS1jb250cm9sLW1vZGUiLCBjZmcuY29udHJvbF9tb2RlLAogICAgICAgICAgICAgICItLXNpbS1iYWNrZW5kIiwgY2ZnLnNpbV9iYWNrZW5kLCAiLS1tYXgtZXBpc29kZS1zdGVwcyIsIHN0cihjZmcubWF4X2VwaXNvZGVfc3RlcHMpXQogICAgZmxhZ3MgPSBPbWVnYUNvbmYudG9fY29udGFpbmVyKGNmZy5mbGFncywgcmVzb2x2ZT1UcnVlKQogICAgY21kID0gW3N5cy5leGVjdXRhYmxlLCAiLXUiLCBjZmcuc2NyaXB0LCAiLS1kZW1vLXBhdGgiLCAqZGVtb3NdICsgY29tbW9uICsgX2ZsYWdzX3RvX2NsaShmbGFncykKICAgIGN3ZCA9IG9zLnBhdGguam9pbihIRVJFLCAiYmFzZWxpbmVzIiwgY2ZnLmJhc2VsaW5lX2RpcikKICAgIG1ldGhvZCA9IGh5ZHJhLmNvcmUuaHlkcmFfY29uZmlnLkh5ZHJhQ29uZmlnLmdldCgpLnJ1bnRpbWUuY2hvaWNlcy5nZXQoIm1ldGhvZCIsIGNmZy5zY3JpcHQpCiAgICBwcmludChmIltpbC90cmFpbl0gbWV0aG9kPXttZXRob2R9XG4iCiAgICAgICAgICBmIltpbC90cmFpbl0gY3dkPXtjd2R9XG5baWwvdHJhaW5dIHsnICcuam9pbihjbWQpfSIsIGZsdXNoPVRydWUpCiAgICBzeXMuZXhpdChzdWJwcm9jZXNzLnJ1bihjbWQsIGN3ZD1jd2QpLnJldHVybmNvZGUpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='}, 'tools/hard_gpu_worker.py': {'base_sha256': None, 'sha256': 'b56c8c5f9a1123acb983fa4e20f58a082f2d767b248e66d1b478435ea43d813b', 'content_b64': 'IiIiSGFyZC1vbmx5IGluLXByb2Nlc3MgR1BVIGVudHJ5IHBvaW50LiBWZXJpZnkgaGVsZCBGRHMgYmVmb3JlIGFueSBDVURBLWNhcGFibGUgaW1wb3J0LiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKZnJvbSBjb250ZXh0bGliIGltcG9ydCBudWxsY29udGV4dAppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBydW5weQppbXBvcnQgc3lzCgpmcm9tIHRvb2xzIGltcG9ydCBydW5fY29sYWJfaGFyZCBhcyBoYXJkCgoKY2xhc3MgU21va2VBdWRpdDoKICAgICIiIk9ic2VydmUgYWN0dWFsIHNjYWxlciBiYXRjaGVzIGFuZCBvcHRpbWl6ZXIgY2FsbHMgd2l0aG91dCBjaGFuZ2luZyBiYXRjaC9zY2hlZHVsZS9FTUEgYmVoYXZpb3IuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYpOgogICAgICAgIHNlbGYubG9zc2VzID0gW10KICAgICAgICBzZWxmLnVwZGF0ZV9iYXRjaGVzID0gW10KICAgICAgICBzZWxmLmRldmljZXMgPSBzZXQoKQoKICAgIGRlZiBfX2VudGVyX18oc2VsZik6CiAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgZnJvbSB0b3JjaC5vcHRpbS5vcHRpbWl6ZXIgaW1wb3J0IHJlZ2lzdGVyX29wdGltaXplcl9zdGVwX3ByZV9ob29rLCByZWdpc3Rlcl9vcHRpbWl6ZXJfc3RlcF9wb3N0X2hvb2sKICAgICAgICBzZWxmLm9yaWdpbmFsX3NjYWxlID0gdG9yY2guYW1wLkdyYWRTY2FsZXIuc2NhbGUKCiAgICAgICAgZGVmIHNjYWxlKHNjYWxlciwgbG9zcywgKmFyZ3MsICoqa3dhcmdzKToKICAgICAgICAgICAgaGFyZC5yZXF1aXJlKHRvcmNoLmlzX3RlbnNvcihsb3NzKSBhbmQgbG9zcy5udW1lbCgpID09IDEgYW5kIGhhcmQuZmluaXRlX3RyZWUobG9zcyksCiAgICAgICAgICAgICAgICAgICAgICAgICAnU21va2Ugbm9uZmluaXRlIG9yIG5vbnNjYWxhciBsb3NzJykKICAgICAgICAgICAgc2VsZi5sb3NzZXMuYXBwZW5kKGZsb2F0KGxvc3MuZGV0YWNoKCkpKQogICAgICAgICAgICBzZWxmLmRldmljZXMuYWRkKGxvc3MuZGV2aWNlLnR5cGUpCiAgICAgICAgICAgIHJldHVybiBzZWxmLm9yaWdpbmFsX3NjYWxlKHNjYWxlciwgbG9zcywgKmFyZ3MsICoqa3dhcmdzKQoKICAgICAgICBkZWYgYmVmb3JlKG9wdGltaXplciwgYXJncywga3dhcmdzKToKICAgICAgICAgICAgZ3JhZHMgPSBbcC5ncmFkIGZvciBncm91cCBpbiBvcHRpbWl6ZXIucGFyYW1fZ3JvdXBzIGZvciBwIGluIGdyb3VwWydwYXJhbXMnXSBpZiBwLmdyYWQgaXMgbm90IE5vbmVdCiAgICAgICAgICAgIGhhcmQucmVxdWlyZShpc2luc3RhbmNlKG9wdGltaXplciwgdG9yY2gub3B0aW0uQWRhbVcpIGFuZCBncmFkcyBhbmQgaGFyZC5maW5pdGVfdHJlZShncmFkcyksCiAgICAgICAgICAgICAgICAgICAgICAgICAnU21va2Ugc3VjY2Vzc2Z1bCBzdGVwIHJlcXVpcmVzIGZpbml0ZSB1bnNjYWxlZCBncmFkaWVudHMnKQoKICAgICAgICBkZWYgYWZ0ZXIob3B0aW1pemVyLCBhcmdzLCBrd2FyZ3MpOgogICAgICAgICAgICBoYXJkLnJlcXVpcmUoaGFyZC5maW5pdGVfdHJlZShvcHRpbWl6ZXIuc3RhdGVfZGljdCgpKSBhbmQKICAgICAgICAgICAgICAgICAgICAgICAgIGhhcmQuZmluaXRlX3RyZWUoW3AgZm9yIGcgaW4gb3B0aW1pemVyLnBhcmFtX2dyb3VwcyBmb3IgcCBpbiBnWydwYXJhbXMnXV0pLAogICAgICAgICAgICAgICAgICAgICAgICAgJ1Ntb2tlIG5vbmZpbml0ZSBvcHRpbWl6ZXIgdXBkYXRlJykKICAgICAgICAgICAgc2VsZi51cGRhdGVfYmF0Y2hlcy5hcHBlbmQobGVuKHNlbGYubG9zc2VzKSkKCiAgICAgICAgc2VsZi5wcmVfaG9vayA9IHJlZ2lzdGVyX29wdGltaXplcl9zdGVwX3ByZV9ob29rKGJlZm9yZSkKICAgICAgICBzZWxmLnBvc3RfaG9vayA9IHJlZ2lzdGVyX29wdGltaXplcl9zdGVwX3Bvc3RfaG9vayhhZnRlcikKICAgICAgICB0b3JjaC5hbXAuR3JhZFNjYWxlci5zY2FsZSA9IHNjYWxlCiAgICAgICAgcmV0dXJuIHNlbGYKCiAgICBkZWYgX19leGl0X18oc2VsZiwgKmV4Yyk6CiAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgdG9yY2guYW1wLkdyYWRTY2FsZXIuc2NhbGUgPSBzZWxmLm9yaWdpbmFsX3NjYWxlCiAgICAgICAgc2VsZi5wcmVfaG9vay5yZW1vdmUoKQogICAgICAgIHNlbGYucG9zdF9ob29rLnJlbW92ZSgpCgogICAgZGVmIHJlcG9ydChzZWxmKToKICAgICAgICByZXR1cm4gZGljdChwcm9jZXNzZWRfYmF0Y2hlcz1sZW4oc2VsZi5sb3NzZXMpLCBzdWNjZXNzZnVsX29wdGltaXplcl91cGRhdGVzPWxlbihzZWxmLnVwZGF0ZV9iYXRjaGVzKSwKICAgICAgICAgICAgICAgICAgICBza2lwcGVkX29wdGltaXplcl91cGRhdGVzPWxlbihzZWxmLmxvc3NlcykgLSBsZW4oc2VsZi51cGRhdGVfYmF0Y2hlcyksCiAgICAgICAgICAgICAgICAgICAgc3VjY2Vzc2Z1bF91cGRhdGVfYmF0Y2hlcz1zZWxmLnVwZGF0ZV9iYXRjaGVzLCBsb3NzZXM9c2VsZi5sb3NzZXMsCiAgICAgICAgICAgICAgICAgICAgZGV2aWNlcz1zb3J0ZWQoc2VsZi5kZXZpY2VzKSwgZmluaXRlX3N1Y2Nlc3NmdWxfZ3JhZGllbnRzX2FuZF9zdGF0ZT1UcnVlKQoKCmRlZiB0cmFpbmluZ19jb21tYW5kKGNmZywgcmVwbyk6CiAgICBmcm9tIGlsLnRyYWluIGltcG9ydCBfZGVtb19wYXRocywgX2ZsYWdzX3RvX2NsaQogICAgZnJvbSBvbWVnYWNvbmYgaW1wb3J0IE9tZWdhQ29uZgogICAgaGFyZC5yZXF1aXJlKGNmZy5iYXNlbGluZV9kaXIgPT0gJ2RpZmZ1c2lvbl9wb2xpY3knIGFuZCBjZmcuc2NyaXB0ID09ICd0cmFpbl9yZ2JkLnB5JwogICAgICAgICAgICAgICAgIGFuZCBjZmcuZGVtb19kaXIgPT0gJ2hhcmQnIGFuZCBjZmcubWF4X2VwaXNvZGVfc3RlcHMgPT0gODAwLCAnSGFyZCB0cmFpbmVyIGNvbnRyYWN0IGRpZmZlcnMnKQogICAgZGVtb3MgPSBfZGVtb19wYXRocyhjZmcpCiAgICBoYXJkLnJlcXVpcmUoZGVtb3MgPT0gW3N0cihQYXRoKHJlcG8pIC8gaGFyZC5ERU1PKV0sICdIYXJkIGRlbW8gcGF0aCBkaWZmZXJzJykKICAgIGZsYWdzID0gT21lZ2FDb25mLnRvX2NvbnRhaW5lcihjZmcuZmxhZ3MsIHJlc29sdmU9VHJ1ZSkKICAgIGNvbW1vbiA9IFsnLS1lbnYtaWQnLCBjZmcuZW52X2lkLCAnLS1jb250cm9sLW1vZGUnLCBjZmcuY29udHJvbF9tb2RlLAogICAgICAgICAgICAgICctLXNpbS1iYWNrZW5kJywgY2ZnLnNpbV9iYWNrZW5kLCAnLS1tYXgtZXBpc29kZS1zdGVwcycsIHN0cihjZmcubWF4X2VwaXNvZGVfc3RlcHMpXQogICAgcmV0dXJuIFtzeXMuZXhlY3V0YWJsZSwgJy11JywgY2ZnLnNjcmlwdCwgJy0tZGVtby1wYXRoJywgKmRlbW9zXSArIGNvbW1vbiArIF9mbGFnc190b19jbGkoZmxhZ3MpCgoKZGVmIHJ1bl90cmFpbmluZyhjZmcsIHJlcG8sIGxvY2FsLCBmZHMpOgogICAgZnJvbSBoeWRyYS5jb3JlLmh5ZHJhX2NvbmZpZyBpbXBvcnQgSHlkcmFDb25maWcKICAgIGZyb20gb21lZ2Fjb25mIGltcG9ydCBPbWVnYUNvbmYKICAgIGhhcmQucmVxdWlyZShIeWRyYUNvbmZpZy5nZXQoKS5ydW50aW1lLmNob2ljZXNbJ21ldGhvZCddID09ICdkcF9yZ2JfaGFyZCcsICdIYXJkIG1ldGhvZCByZXF1aXJlZCcpCiAgICBjbWQgPSB0cmFpbmluZ19jb21tYW5kKGNmZywgcmVwbykKICAgIGhhcmQucmVxdWlyZShhbGwoUGF0aChwKS5pc19maWxlKCkgZm9yIHAgaW4gW3N0cihQYXRoKHJlcG8pIC8gaGFyZC5ERU1PKV0pLCAnU3RhZ2VkIGRlbW8gbWlzc2luZycpCiAgICBvdXRwdXQgPSBQYXRoKEh5ZHJhQ29uZmlnLmdldCgpLnJ1bnRpbWUub3V0cHV0X2RpcikKICAgIG91dHB1dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBPbWVnYUNvbmYuc2F2ZShjZmcsIG91dHB1dCAvICdyZXNvbHZlZC1jb25maWcueWFtbCcsIHJlc29sdmU9VHJ1ZSkKICAgIGhhcmQud3JpdGVfanNvbihvdXRwdXQgLyAnbGF1bmNoZXIuanNvbicsIGRpY3QoY29tbWFuZD1jbWQsIGh5ZHJhX292ZXJyaWRlcz1saXN0KEh5ZHJhQ29uZmlnLmdldCgpLm92ZXJyaWRlcy50YXNrKSwKICAgICAgICAgICAgICAgICAgICBwaWQ9b3MuZ2V0cGlkKCksIGxvY2tfcGF0aHM9W3N0cihwKSBmb3IgcCBpbiBoYXJkLmxvY2tfcGF0aHMocmVwbywgbG9jYWwpXSkpCiAgICBjd2QgPSBQYXRoKHJlcG8pIC8gJ2lsL2Jhc2VsaW5lcy9kaWZmdXNpb25fcG9saWN5JwogICAgcHJpbnQoZidbaWwvdHJhaW5dIG1ldGhvZD1kcF9yZ2JfaGFyZFxuW2lsL3RyYWluXSBjd2Q9e2N3ZH1cbltpbC90cmFpbl0geyIgIi5qb2luKGNtZCl9JywgZmx1c2g9VHJ1ZSkKICAgIHByaW50KCdbaGFyZF0gZXhlY3V0aW5nIHRyYWluZXIgaW4gdGhpcyBsb2NrLW93bmluZyBwcm9jZXNzJywgZmx1c2g9VHJ1ZSkKICAgIHNtb2tlID0gY2ZnLmZsYWdzLnRvdGFsX2l0ZXJzID09IDEwMAogICAgaWYgc21va2U6CiAgICAgICAgaGFyZC5yZXF1aXJlKGNmZy5mbGFncy5ldmFsX2ZyZXEgPT0gMCBhbmQgY2ZnLmZsYWdzLmxvZ19mcmVxID09IDEsICdTbW9rZSBsb2dnaW5nL2V2YWwgY29udHJhY3QgZGlmZmVycycpCiAgICBhdWRpdCA9IFNtb2tlQXVkaXQoKSBpZiBzbW9rZSBlbHNlIG51bGxjb250ZXh0KCkKICAgICMgUmVwZWF0IGF0IHRoZSBib3VuZGFyeTogbm8gc3VicHJvY2Vzcy9leGVjL2Nsb3NlX2ZkcyBob3AgYmV0d2VlbiB0aGlzIGNoZWNrIGFuZCBDVURBLgogICAgaGFyZC52ZXJpZnlfaW5oZXJpdGVkX2xvY2tzKGZkcywgcmVwbywgbG9jYWwpCiAgICB3aXRoIGF1ZGl0OgogICAgICAgIGV4ZWN1dGVfc2NyaXB0KGN3ZCAvIGNmZy5zY3JpcHQsIGNtZFszOl0sIGN3ZCkKICAgIGlmIHNtb2tlOgogICAgICAgIGhhcmQud3JpdGVfanNvbihQYXRoKGxvY2FsKSAvICdzbW9rZS9iYXRjaC1hdWRpdC5qc29uJywgYXVkaXQucmVwb3J0KCkpCgoKZGVmIGV4ZWN1dGVfc2NyaXB0KHBhdGgsIGFyZ3MsIGN3ZCk6CiAgICBvbGRfY3dkLCBvbGRfYXJndiwgb2xkX3BhdGggPSBQYXRoLmN3ZCgpLCBzeXMuYXJndiwgc3lzLnBhdGhbOl0KICAgIHRyeToKICAgICAgICBvcy5jaGRpcihjd2QpCiAgICAgICAgc3lzLmFyZ3YgPSBbc3RyKHBhdGgpLCAqYXJnc10KICAgICAgICBzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBhdGgocGF0aCkucGFyZW50KSkKICAgICAgICBydW5weS5ydW5fcGF0aChzdHIocGF0aCksIHJ1bl9uYW1lPSdfX21haW5fXycpCiAgICBmaW5hbGx5OgogICAgICAgIG9zLmNoZGlyKG9sZF9jd2QpCiAgICAgICAgc3lzLmFyZ3YsIHN5cy5wYXRoWzpdID0gb2xkX2FyZ3YsIG9sZF9wYXRoCgoKZGVmIG1haW4oKToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLXJlcG8nLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1sb2NhbCcsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLWxvY2stZmRzJywgcmVxdWlyZWQ9VHJ1ZSwgbmFyZ3M9MywgdHlwZT1pbnQpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCdlbnRyeScsIGNob2ljZXM9WydpbC90cmFpbi5weScsICdldmFsLnB5JywgJ3Rvb2xzL3JvbGxvdXRfbG9nZ2VyLnB5JywgJ3Rvb2xzL3J1bl9jb2xhYl9oYXJkLnB5J10pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCdhcmd1bWVudHMnLCBuYXJncz1hcmdwYXJzZS5SRU1BSU5ERVIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgaGFyZC52ZXJpZnlfaW5oZXJpdGVkX2xvY2tzKGFyZ3MubG9ja19mZHMsIGFyZ3MucmVwbywgYXJncy5sb2NhbCkKICAgIGlmIGFyZ3MuZW50cnkgPT0gJ2lsL3RyYWluLnB5JzoKICAgICAgICBpbXBvcnQgaHlkcmEKICAgICAgICBzeXMuYXJndiA9IFtzeXMuYXJndlswXSwgKmFyZ3MuYXJndW1lbnRzXQogICAgICAgIEBoeWRyYS5tYWluKHZlcnNpb25fYmFzZT1Ob25lLCBjb25maWdfcGF0aD1zdHIoUGF0aChhcmdzLnJlcG8pIC8gJ2lsL2NvbmYnKSwgY29uZmlnX25hbWU9J3RyYWluJykKICAgICAgICBkZWYgbGF1bmNoKGNmZyk6CiAgICAgICAgICAgIHJ1bl90cmFpbmluZyhjZmcsIGFyZ3MucmVwbywgYXJncy5sb2NhbCwgYXJncy5sb2NrX2ZkcykKICAgICAgICBsYXVuY2goKQogICAgZWxzZToKICAgICAgICBleGVjdXRlX3NjcmlwdChQYXRoKGFyZ3MucmVwbykgLyBhcmdzLmVudHJ5LCBhcmdzLmFyZ3VtZW50cywgYXJncy5yZXBvKQoKCmlmIF9fbmFtZV9fID09ICdfX21haW5fXyc6CiAgICBtYWluKCkK'}, 'tools/make_colab_hard_notebook.py': {'base_sha256': None, 'sha256': 'a8c3db1911bf73f4f3c87cdefc3b0a1b726e64b27e06286fbca5548d100d908b', 'content_b64': 'IiIiR2VuZXJhdGUgYW4gdW5leGVjdXRlZCwgc3RhZ2VkIEhhcmQgbm90ZWJvb2sgZnJvbSByZXZpZXdlZCB3b3JrdHJlZSBieXRlcy4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFzdAppbXBvcnQgYmFzZTY0CmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgc3VicHJvY2Vzcwpmcm9tIHRleHR3cmFwIGltcG9ydCBkZWRlbnQKCmZyb20gdG9vbHMgaW1wb3J0IHJ1bl9jb2xhYl9oYXJkIGFzIGhhcmQKClJFUE8gPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1sxXQpPVVRQVVQgPSBSRVBPIC8gJ01BUlNPX0NPTEFCX0hBUkQuaXB5bmInCgpNT1VOVCA9ICcnJwppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gZ29vZ2xlLmNvbGFiIGltcG9ydCBkcml2ZQppZiBub3Qgb3MucGF0aC5pc21vdW50KCcvY29udGVudC9kcml2ZScpOgogICAgZHJpdmUubW91bnQoJy9jb250ZW50L2RyaXZlJykKYXNzZXJ0IG9zLnBhdGguaXNtb3VudCgnL2NvbnRlbnQvZHJpdmUnKSwgJ0EgcmVhbCBEcml2ZSBtb3VudCBpcyByZXF1aXJlZCcKYXNzZXJ0IFBhdGgoJy9jb250ZW50L2RyaXZlL015RHJpdmUvbWFyc28nKS5pc19kaXIoKSwgJ0V4aXN0aW5nIG1hcnNvIERyaXZlIHJvb3QgaXMgcmVxdWlyZWQnCnByaW50KCdEcml2ZSBtb3VudGVkLiBQYXJlbnQgc3RhZ2VzIEhhcmQgZGVtb3Mgc2VwYXJhdGVseTsgYSBEcml2ZSBIYXJkIG1hcmtlciBpcyBub3QgcmVxdWlyZWQuJykKJycnCgpTRVRVUCA9ICcnJwppbXBvcnQgYmFzZTY0LCBoYXNobGliLCBqc29uLCBvcywgc3VicHJvY2VzcywgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKQVBQTFlfSEFSRF9TRVRVUCA9IEZhbHNlCmlmIEFQUExZX0hBUkRfU0VUVVA6CiAgICAjIFRoaXMgbmFtZXNwYWNlIGV4ZWN1dGVzIG9ubHkgdGhlIGhhc2gtY2hlY2tlZCwgZW1iZWRkZWQgc3RkbGliIEhhcmQgYm9vdHN0cmFwLgogICAgZW50cnkgPSBFTUJFRERFRF9TT1VSQ0VTWyd0b29scy9ydW5fY29sYWJfaGFyZC5weSddCiAgICBwYXlsb2FkID0gYmFzZTY0LmI2NGRlY29kZShlbnRyeVsnY29udGVudF9iNjQnXSwgdmFsaWRhdGU9VHJ1ZSkKICAgIGFzc2VydCBoYXNobGliLnNoYTI1NihwYXlsb2FkKS5oZXhkaWdlc3QoKSA9PSBlbnRyeVsnc2hhMjU2J10KICAgIGhhcmRfc2V0dXAgPSB7J19fbmFtZV9fJzogJ2hhcmRfYm9vdHN0cmFwJ30KICAgIGV4ZWMoY29tcGlsZShwYXlsb2FkLCAnZW1iZWRkZWQvdG9vbHMvcnVuX2NvbGFiX2hhcmQucHknLCAnZXhlYycpLCBoYXJkX3NldHVwKQogICAgRVhQID0gJ2hhcmRfJyArIHRpbWUuc3RyZnRpbWUoJyVZJW0lZF8lSCVNJVMnKQogICAgUExBTiA9IGhhcmRfc2V0dXBbJ2Jvb3RzdHJhcCddKEVNQkVEREVEX1NPVVJDRVMsIEVYUCkKICAgIHByaW50KCdQcmVwYXJlZCBuZXcgcnVuOicsIFBMQU4pCiAgICBwcmludCgnUmV2aWV3ZWQgb3JpZ2luYWxzIGJhY2tlZCB1cCB1bmRlciBzb3VyY2UtYmFja3Vwczsgc25hcHNob3RzIHVuZGVyIHNvdXJjZXMuJykKZWxzZToKICAgIHByaW50KCdTZXR1cCBza2lwcGVkLiBSZXZpZXcgdGhpcyBjZWxsLCB0aGVuIHNldCBBUFBMWV9IQVJEX1NFVFVQPVRydWUgYW5kIGV4ZWN1dGUgb25seSB0aGlzIGNlbGwuJykKJycnCgpIRUxQRVIgPSAnJycKIyBObyBrZXJuZWwgcmVzdGFydCwgY2xvbmUsIHBhY2thZ2UgaW5zdGFsbGF0aW9uLCBlbnZpcm9ubWVudCByZXBsYWNlbWVudCwgb3IgZGF0YSBkb3dubG9hZC4KUkVQTyA9IFBhdGgoJy9jb250ZW50L2Jlcmxpbi1tYXJzby1oYWNrYXRob24nKQpQWSA9ICcvY29udGVudC9tYXJzby1weTMxMi9iaW4vcHl0aG9uJwpFTlYgPSBkaWN0KG9zLmVudmlyb24sIERJU1BMQVk9JycsIFBZT1BFTkdMX1BMQVRGT1JNPSdlZ2wnLCBIREY1X1VTRV9GSUxFX0xPQ0tJTkc9J0ZBTFNFJywKICAgICAgICAgICBQWVRIT05VTkJVRkZFUkVEPScxJywgUFlUSE9OUEFUSD1zdHIoUkVQTyksIFBZVE9SQ0hfQ1VEQV9BTExPQ19DT05GPSdleHBhbmRhYmxlX3NlZ21lbnRzOlRydWUnKQpkZWYgaGFyZF9waGFzZShwaGFzZSk6CiAgICBhc3NlcnQgJ1BMQU4nIGluIGdsb2JhbHMoKSwgJ1J1biB0aGUgcmV2aWV3ZWQgc2V0dXAgY2VsbCBmaXJzdCAob3IgcmVzdG9yZSB0aGUgZXhhY3Qgc2F2ZWQgUExBTiBwYXRoKScKICAgIHN1YnByb2Nlc3MucnVuKFtQWSwgJy11Jywgc3RyKFJFUE8vJ3Rvb2xzL3J1bl9jb2xhYl9oYXJkLnB5JyksIHN0cihQTEFOKSwgJy0tcGhhc2UnLCBwaGFzZV0sCiAgICAgICAgICAgICAgICAgICBjd2Q9UkVQTywgZW52PUVOViwgY2hlY2s9VHJ1ZSkKJycnCgpDSEVDS1MgPSAnJycKUlVOX0NIRUNLUyA9IEZhbHNlCmlmIFJVTl9DSEVDS1M6CiAgICBoYXJkX3BoYXNlKCdjaGVja3MnKQplbHNlOgogICAgcHJpbnQoJ0NoZWNrcyBza2lwcGVkLiBBZnRlciBwYXJlbnQgc3RhZ2VzIGxvY2FsIGg1L2pzb24sIGVuYWJsZSBvbmx5IHRoaXMgY2VsbC4nKQonJycKU01PS0UgPSAnJycKUlVOX1NNT0tFID0gRmFsc2UKaWYgUlVOX1NNT0tFOgogICAgaGFyZF9waGFzZSgnc21va2UnKQplbHNlOgogICAgcHJpbnQoJ1Ntb2tlIHNraXBwZWQuIEVuYWJsZSBhZnRlciBjaGVja3NfcGFzc2VkOiAxMDAgaXRlcmF0aW9ucywgYWxsIDIwMCBkZW1vcywgc2VwYXJhdGUgd2VpZ2h0cy4nKQonJycKTEFVTkNIID0gJycnClJVTl9IQVJEID0gRmFsc2UKaWYgUlVOX0hBUkQ6CiAgICBoYXJkX3BoYXNlKCdydW4nKQplbHNlOgogICAgcHJpbnQoJ0Z1bGwgdHJhaW5pbmcgc2tpcHBlZC4gUmV2aWV3IHNtb2tlX3Bhc3NlZCwgdGhlbiBlbmFibGUgb25seSB0aGlzIGNlbGwuJykKICAgIHByaW50KCc0MCwwMDAgaXRlcmF0aW9ucyBmcm9tIGZpeGVkIHNlZWQgMTsgbm8gc21va2UgY2hlY2twb2ludCByZXN1bWUuIFNpeC1ob3VyIHRyYWluIGRlYWRsaW5lLicpCicnJwpNT05JVE9SID0gJycnCiMgUmVhZC1vbmx5LiBTZXQgdGhlIGV4YWN0IFBMQU4gcGF0aCBhZnRlciByZWNvbm5lY3Q7IHRoaXMgY2VsbCBuZXZlciBjaG9vc2VzIG9yIGxhdW5jaGVzIGEgcnVuLgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IGpzb24KUExBTl9QQVRIID0gc3RyKFBMQU4pIGlmICdQTEFOJyBpbiBnbG9iYWxzKCkgZWxzZSAnJwppZiBQTEFOX1BBVEg6CiAgICByb290ID0gUGF0aChQTEFOX1BBVEgpLnBhcmVudC5wYXJlbnQKICAgIGZvciByZWwgaW4gKCdydW4vc3RhdHVzLmpzb24nLCAncnVuL3N1bW1hcnkuanNvbicsICdydW4vZHVyYWJpbGl0eS5qc29uJywgJ3J1bi90cmFpbi5sb2cnKToKICAgICAgICBwYXRoID0gcm9vdCAvIHJlbAogICAgICAgIGlmIHBhdGguaXNfZmlsZSgpOgogICAgICAgICAgICB3aXRoIHBhdGgub3BlbigncmInKSBhcyBzdHJlYW06CiAgICAgICAgICAgICAgICBzdHJlYW0uc2VlayhtYXgoMCwgcGF0aC5zdGF0KCkuc3Rfc2l6ZSAtIDUwMDApKQogICAgICAgICAgICAgICAgcHJpbnQocmVsLCBzdHJlYW0ucmVhZCgpLmRlY29kZShlcnJvcnM9J3JlcGxhY2UnKSkKZWxzZToKICAgIHByaW50KCdTZXQgUExBTl9QQVRIIHRvIC9jb250ZW50L21hcnNvLWhhcmQvaGFyZF88dGltZXN0YW1wPi9ydW4vcGxhbi5qc29uLiBEbyBub3QgcmVsYXVuY2ggYmxpbmRseS4nKQonJycKRklOQUxJWkUgPSAnJycKUkVUUllfRklOQUxfU1lOQyA9IEZhbHNlCmlmIFJFVFJZX0ZJTkFMX1NZTkM6CiAgICBoYXJkX3BoYXNlKCdmaW5hbGl6ZScpCmVsc2U6CiAgICBwcmludCgnT3B0aW9uYWw6IGFmdGVyIGNvbXB1dGVfc3RhdHVzPWNvbXBsZXRlZCBhbmQgcmVtb3VudGluZyBEcml2ZSwgcmV0cnkgZHVyYWJpbGl0eSBvbmx5LicpCicnJwoKCmRlZiBzb3VyY2VfYnVuZGxlKHJlcG89UkVQTyk6CiAgICBidW5kbGUgPSB7fQogICAgZm9yIHJlbCBpbiBzb3J0ZWQoaGFyZC5TT1VSQ0VfRklMRVMpOgogICAgICAgIHBhdGggPSBoYXJkLmNoaWxkKFBhdGgocmVwbyksIHJlbCkKICAgICAgICBkYXRhID0gcGF0aC5yZWFkX2J5dGVzKCkKICAgICAgICBpZiBwYXRoLnN1ZmZpeCA9PSAnLnB5JzoKICAgICAgICAgICAgYXN0LnBhcnNlKGRhdGEsIGZpbGVuYW1lPXJlbCkKICAgICAgICBvbGQgPSBzdWJwcm9jZXNzLnJ1bihbJ2dpdCcsICdzaG93JywgZidIRUFEOntyZWx9J10sIGN3ZD1yZXBvLCBjYXB0dXJlX291dHB1dD1UcnVlLCBjaGVjaz1GYWxzZSkKICAgICAgICBidW5kbGVbcmVsXSA9IGRpY3QoYmFzZV9zaGEyNTY9aGFzaGxpYi5zaGEyNTYob2xkLnN0ZG91dCkuaGV4ZGlnZXN0KCkgaWYgb2xkLnJldHVybmNvZGUgPT0gMCBlbHNlIE5vbmUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgIHNoYTI1Nj1oYXNobGliLnNoYTI1NihkYXRhKS5oZXhkaWdlc3QoKSwgY29udGVudF9iNjQ9YmFzZTY0LmI2NGVuY29kZShkYXRhKS5kZWNvZGUoJ2FzY2lpJykpCiAgICBoYXJkLnZhbGlkYXRlX3NvdXJjZV9zZXQoYnVuZGxlKQogICAgcmV0dXJuIGJ1bmRsZQoKCmRlZiBjZWxsKHNvdXJjZSwgbmFtZSwgKiwgbWFya2Rvd249RmFsc2UpOgogICAgdGV4dCA9IGRlZGVudChzb3VyY2UpLnN0cmlwKCkgKyAnXG4nCiAgICByZXN1bHQgPSBkaWN0KGNlbGxfdHlwZT0nbWFya2Rvd24nIGlmIG1hcmtkb3duIGVsc2UgJ2NvZGUnLCBpZD1uYW1lLCBtZXRhZGF0YT17fSwgc291cmNlPXRleHQuc3BsaXRsaW5lcyhUcnVlKSkKICAgIGlmIG5vdCBtYXJrZG93bjoKICAgICAgICBhc3QucGFyc2UodGV4dCwgZmlsZW5hbWU9bmFtZSkKICAgICAgICByZXN1bHQudXBkYXRlKGV4ZWN1dGlvbl9jb3VudD1Ob25lLCBvdXRwdXRzPVtdKQogICAgcmV0dXJuIHJlc3VsdAoKCmRlZiBidWlsZF9ub3RlYm9vaygpOgogICAgYnVuZGxlID0gc291cmNlX2J1bmRsZSgpCiAgICBuYiA9IGRpY3QobmJmb3JtYXQ9NCwgbmJmb3JtYXRfbWlub3I9NSwKICAgICAgICAgICAgICBtZXRhZGF0YT17J2tlcm5lbHNwZWMnOiB7J2Rpc3BsYXlfbmFtZSc6ICdQeXRob24gMycsICdsYW5ndWFnZSc6ICdweXRob24nLCAnbmFtZSc6ICdweXRob24zJ319LAogICAgICAgICAgICAgIGNlbGxzPVtjZWxsKCcnJwojIEhhcmQgUkdCIERpZmZ1c2lvbiBQb2xpY3kg4oCUIHN0YWdlZCBsb2NhbC1maXJzdCBydW4KRXhlY3V0ZSBlYWNoIHN0YWdlIHNlcGFyYXRlbHk7ICoqZG8gbm90IHVzZSBSdW4gQWxsKiouIEFsbCBhY3Rpb24gc3dpdGNoZXMgc3RhcnQgRmFsc2UuClJldXNlIGAvY29udGVudC9iZXJsaW4tbWFyc28taGFja2F0aG9uYCBhbmQgYC9jb250ZW50L21hcnNvLXB5MzEyL2Jpbi9weXRob25gLgpQYXJlbnQgc3RhZ2VzIGBpbC9kZW1vcy9oYXJkL3RyYWplY3RvcnkucmdiLnBkX2VlX2RlbHRhX3Bvcy5waHlzeF9jdWRhLntoNSxqc29ufWAuClRoaXMgbm90ZWJvb2sgZG9lcyBub3QgaW5zdGFsbCBwYWNrYWdlcyBvciByZXN0YXJ0IHRoZSBrZXJuZWwuCgoxLiBNb3VudCBjaGVjazsgaW5zcGVjdCBleHBlY3RlZCBEcml2ZSByb290LgoyLiBFbmFibGUgcmV2aWV3ZWQgc291cmNlIHNldHVwIG9uY2UuIEV2ZXJ5IGRlc3RpbmF0aW9uIG11c3QgbWF0Y2ggaXRzIGJhc2UvY3VycmVudCBoYXNoOwogICB1bmtub3duIHNvdXJjZXMgYW5kIGV4aXN0aW5nIG91dHB1dCBkaXJlY3RvcmllcyBhcmUgcmVqZWN0ZWQuIE9yaWdpbmFscyBhcmUgYmFja2VkIHVwLgozLiBFbmFibGUgZGF0YXNldC9DVURBL3JlZmVyZW5jZSBjaGVja3MuIFRoZSByZWZlcmVuY2UgcmVjb3JkcyB2YWxpZCBIYXJkIGZhaWx1cmVzIGhvbmVzdGx5Lgo0LiBFbmFibGUgMTAwLXN0ZXAgQ1VEQSBzbW9rZS4gSW5zcGVjdCBmaW5pdGUgdXBkYXRlcyBhbmQgYHNtb2tlX3Bhc3NlZGAuCjUuIEVuYWJsZSBgUlVOX0hBUkRgIG9ubHkgYWZ0ZXIgcmV2aWV3aW5nIHNtb2tlLiBGcmVzaCBzZWVkIDEsIDQwLDAwMCBpdGVyYXRpb25zLCBBTVAsCiAgIGJhdGNoIDEyOCwgbHIgMWUtNCwgUmVzTmV0MTggc2NlbmUgUkdCLCBob3Jpem9ucyAyLzgvMTY7IGxhdGVzdCBzYXZlZCBldmVyeSAxLDAwMC4KICAgVHJhaW5pbmcgZXZhbDogMzIgZXBpc29kZXMgLyA4IGVudnMgZXZlcnkgNSwwMDAuIFNpeC1ob3VyIHRyYWluaW5nIGxpbWl0Lgo2LiBNb25pdG9yIGJ5IGV4YWN0IHNhdmVkIHBsYW4gcGF0aC4gUmUtZXhlY3V0aW5nIGEgdHJhaW5pbmcgcGhhc2UgaXMgcmVqZWN0ZWQuCgpPdXRwdXRzOiBgL2NvbnRlbnQvbWFyc28taGFyZC9oYXJkXzx0aW1lc3RhbXA+YCBhbmQKYE15RHJpdmUvbWFyc28vcmVjb3Zlcmllcy9oYXJkL2hhcmRfPHRpbWVzdGFtcD5gLiBMb2NhbCBjb21wbGV0aW9uIGFuZCByZW1vdGUgcGVuZGluZyBhcmUgc2VwYXJhdGUuClRoZSBiZXN0IHNvcnQtYWNjdXJhY3kgY2hlY2twb2ludCBpcyBldmFsdWF0ZWQgb24gZm91ciBsb2NhbCBzZWVkLTUwMDAgY29uZmlndXJhdGlvbnMsIHRoZW4KcmVjb3JkZWQgd2l0aCBvbmUgdmlkZW8gZW52IGFuZCBlaWdodCBkaWFnbm9zdGljIGVwaXNvZGVzLiBTY29yZXMgYXJlIGxvY2FsIHZhbGlkYXRpb24sCm5vdCBvZmZpY2lhbCBvciBoZWxkb3V0LiBTaGFyZWQgZXZhbC9kaWFnbm9zdGljIGxvb3BzIGV4ZWN1dGUgNzk5IHN0ZXBzIHdpdGggYW4gODAwLXN0ZXAgbGltaXQuCkEgbWlzc2luZyBiZXN0IGNoZWNrcG9pbnQgKGFsbCB0cmFja2VkIHNvcnQgbWV0cmljcyB6ZXJvKSB1c2VzIGZpbmFsIHdpdGggYW4gZXhwbGljaXQgZmFsbGJhY2sgcmVjb3JkLgonJycsICdpbnN0cnVjdGlvbnMnLCBtYXJrZG93bj1UcnVlKSwKICAgICAgICAgICAgICAgICAgICAgY2VsbChNT1VOVCwgJ21vdW50LWNoZWNrJyksCiAgICAgICAgICAgICAgICAgICAgIGNlbGwoJ0VNQkVEREVEX1NPVVJDRVMgPSAnICsgcmVwcihidW5kbGUpICsgJ1xuJyArIGRlZGVudChTRVRVUCksICdyZXZpZXdlZC1zZXR1cCcpLAogICAgICAgICAgICAgICAgICAgICBjZWxsKEhFTFBFUiArIENIRUNLUywgJ2RhdGFzZXQtZ3B1LXJlZmVyZW5jZScpLAogICAgICAgICAgICAgICAgICAgICBjZWxsKFNNT0tFLCAnc21va2UtZ2F0ZScpLCBjZWxsKExBVU5DSCwgJ2Z1bGwtbGF1bmNoJyksCiAgICAgICAgICAgICAgICAgICAgIGNlbGwoTU9OSVRPUiwgJ21vbml0b3InKSwgY2VsbChGSU5BTElaRSwgJ3JldHJ5LWR1cmFiaWxpdHknKV0pCiAgICB2YWxpZGF0ZV9ub3RlYm9vayhuYikKICAgIHJldHVybiBuYgoKCmRlZiB2YWxpZGF0ZV9ub3RlYm9vayhuYik6CiAgICBjb2RlID0ge2NbJ2lkJ106ICcnLmpvaW4oY1snc291cmNlJ10pIGZvciBjIGluIG5iWydjZWxscyddIGlmIGNbJ2NlbGxfdHlwZSddID09ICdjb2RlJ30KICAgIGZvciBuYW1lLCBzb3VyY2UgaW4gY29kZS5pdGVtcygpOgogICAgICAgIGFzdC5wYXJzZShzb3VyY2UsIGZpbGVuYW1lPW5hbWUpCiAgICBmb3IgbmFtZSwgZmxhZyBpbiBbKCdyZXZpZXdlZC1zZXR1cCcsICdBUFBMWV9IQVJEX1NFVFVQJyksICgnZGF0YXNldC1ncHUtcmVmZXJlbmNlJywgJ1JVTl9DSEVDS1MnKSwKICAgICAgICAgICAgICAgICAgICAgICAoJ3Ntb2tlLWdhdGUnLCAnUlVOX1NNT0tFJyksICgnZnVsbC1sYXVuY2gnLCAnUlVOX0hBUkQnKSwgKCdyZXRyeS1kdXJhYmlsaXR5JywgJ1JFVFJZX0ZJTkFMX1NZTkMnKV06CiAgICAgICAgdHJlZSA9IGFzdC5wYXJzZShjb2RlW25hbWVdKQogICAgICAgIGFzc2lnbm1lbnRzID0gW24gZm9yIG4gaW4gdHJlZS5ib2R5IGlmIGlzaW5zdGFuY2UobiwgYXN0LkFzc2lnbikgYW5kIGFueShpc2luc3RhbmNlKHQsIGFzdC5OYW1lKSBhbmQgdC5pZCA9PSBmbGFnIGZvciB0IGluIG4udGFyZ2V0cyldCiAgICAgICAgaGFyZC5yZXF1aXJlKGxlbihhc3NpZ25tZW50cykgPT0gMSBhbmQgaXNpbnN0YW5jZShhc3NpZ25tZW50c1swXS52YWx1ZSwgYXN0LkNvbnN0YW50KQogICAgICAgICAgICAgICAgICAgICBhbmQgYXNzaWdubWVudHNbMF0udmFsdWUudmFsdWUgaXMgRmFsc2UsIGYnVW5zYWZlIGxhdW5jaCBmbGFnOiB7ZmxhZ30nKQogICAgdHJlZSA9IGFzdC5wYXJzZShjb2RlWydyZXZpZXdlZC1zZXR1cCddKQogICAgYnVuZGxlID0gYXN0LmxpdGVyYWxfZXZhbCh0cmVlLmJvZHlbMF0udmFsdWUpCiAgICBoYXJkLnZhbGlkYXRlX3NvdXJjZV9zZXQoYnVuZGxlKQogICAgY3VycmVudCA9IHNvdXJjZV9idW5kbGUoKQogICAgaGFyZC5yZXF1aXJlKGJ1bmRsZSA9PSBjdXJyZW50LCAnU3RhbGUgZW1iZWRkZWQgc291cmNlczsgcmVnZW5lcmF0ZSBmcm9tIHRoaXMgd29ya3RyZWUnKQogICAgZm9yIHJlbCwgZW50cnkgaW4gYnVuZGxlLml0ZW1zKCk6CiAgICAgICAgcGF5bG9hZCA9IGJhc2U2NC5iNjRkZWNvZGUoZW50cnlbJ2NvbnRlbnRfYjY0J10sIHZhbGlkYXRlPVRydWUpCiAgICAgICAgaGFyZC5yZXF1aXJlKGhhc2hsaWIuc2hhMjU2KHBheWxvYWQpLmhleGRpZ2VzdCgpID09IGVudHJ5WydzaGEyNTYnXSwgJ0NvcnJ1cHQgZW1iZWRkZWQgcGF5bG9hZCcpCiAgICAgICAgaWYgcmVsLmVuZHN3aXRoKCcucHknKToKICAgICAgICAgICAgYXN0LnBhcnNlKHBheWxvYWQsIGZpbGVuYW1lPSdlbWJlZGRlZC8nICsgcmVsKQogICAgZm9yIG5hbWUsIHNvdXJjZSBpbiBjb2RlLml0ZW1zKCk6CiAgICAgICAgc291cmNlID0gJ1xuJy5qb2luKHNvdXJjZS5zcGxpdGxpbmVzKClbMTpdKSBpZiBuYW1lID09ICdyZXZpZXdlZC1zZXR1cCcgZWxzZSBzb3VyY2UKICAgICAgICBoYXJkLnJlcXVpcmUoJ2ZvcmNlX3JlbW91bnQnIG5vdCBpbiBzb3VyY2UgYW5kICdwaXAgaW5zdGFsbCcgbm90IGluIHNvdXJjZSBhbmQgJ29zLmtpbGwoJyBub3QgaW4gc291cmNlLAogICAgICAgICAgICAgICAgICAgICAnVW5zYWZlIG5vdGVib29rIG9wZXJhdGlvbicpCiAgICBoYXJkLnJlcXVpcmUoYWxsKGMuZ2V0KCdleGVjdXRpb25fY291bnQnKSBpcyBOb25lIGFuZCBub3QgYy5nZXQoJ291dHB1dHMnKSBmb3IgYyBpbiBuYlsnY2VsbHMnXSksCiAgICAgICAgICAgICAgICAgJ1NhdmVkIG5vdGVib29rIG11c3QgYmUgdW5leGVjdXRlZCcpCiAgICAjIEdlbmVyYXRpb24gdXNlcyB0aGUgc3lzdGVtIFB5dGhvbiB3aXRoIGV4aXN0aW5nIG5iZm9ybWF0LiBDUFUgdW5pdCB0ZXN0cyBjYW4gYWxzbwogICAgIyB2YWxpZGF0ZSBhbGwgc291cmNlL0FTVCBpbnZhcmlhbnRzIHdpdGhvdXQgYWRkaW5nIG5iZm9ybWF0IHRvIHRoZSB0cmFpbmluZyB0ZXN0IHZlbnYuCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IG5iZm9ybWF0CiAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgcGFzcwogICAgZWxzZToKICAgICAgICBuYmZvcm1hdC52YWxpZGF0ZShuYikKCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgaW1wb3J0IG5iZm9ybWF0CiAgICBub3RlYm9vayA9IGJ1aWxkX25vdGVib29rKCkKICAgIG5iZm9ybWF0LnZhbGlkYXRlKG5vdGVib29rKQogICAgT1VUUFVULndyaXRlX3RleHQoanNvbi5kdW1wcyhub3RlYm9vaywgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MSkgKyAnXG4nKQogICAgcHJpbnQoT1VUUFVUKQo='}, 'tools/rollout_logger.py': {'base_sha256': 'fd6061998da3eb8a9365164f6de788c380be491edc0785eab0548a3999cbafce', 'sha256': '790ddb31421dfd8e30931e2c81f71d191698a906f1027fb240d317baeb22b222', 'content_b64': 'IiIiRGlhZ25vc2UgV0hZIGEgcG9saWN5IGZhaWxzOiBwZXItc3RlcCBsb2cgb2YgZ3JpcHBlciBjb21tYW5kLCBncmFzcCBzdGF0ZSwgVENQIGhlaWdodCBhbmQgdGhlClRDUC10by10YXJnZXQtcGFyY2VsIHh5IGVycm9yLCBwbHVzIHBlci1lcGlzb2RlIHBoYXNlIHN1bW1hcmllcy4gVXNlcyBwcml2aWxlZ2VkIGVudiBzdGF0ZSBmb3IKQU5BTFlTSVMgT05MWSAodGhlIHBvbGljeSBzdGlsbCBhY3RzIGZyb20gdGhlIG9ic2VydmF0aW9uKSAtLSBuZXZlciBwYXJ0IG9mIGEgc3VibWlzc2lvbi4KCiAgcHl0aG9uIHRvb2xzL3JvbGxvdXRfbG9nZ2VyLnB5IGRpZmZpY3VsdHk9ZWFzeSBvYnNfbW9kZT1yZ2IgcG9saWN5PXdhcmVob3VzZV9zb3J0LmlsX3BvbGljeTpsb2FkX2RwX3JnYiBcXAogICAgICBjaGVja3BvaW50PTxja3B0PiBldmFsX2NvbmZpZz1jb25mL2V2YWwvZXZhbDMyLnlhbWwgbnVtX2VudnM9OCArbG9nX2VwaXNvZGVzPTggK2xvZ19vdXQ9b3V0cHV0cy9kaWFnLmpzb24KClByaW50cywgcGVyIGVwaXNvZGU6IHBhcmNlbHMgc29ydGVkLCBzdGVwcyB0byBmaXJzdCBncmFzcCwgbnVtYmVyIG9mIGdyaXBwZXIgY2xvc2UtPm9wZW4gZmxpY2tlcnMsCmxvbmdlc3QgcnVuIG9mIGNvbnNlY3V0aXZlICJjbG9zZSIgY29tbWFuZHMgYmVmb3JlIHRoZSBmaXJzdCBsaWZ0LCBtZWFuIHh5IGVycm9yIGF0IHRoZSBtb21lbnQgdGhlCmdyaXBwZXIgZmlyc3QgY2xvc2VzLCBhbmQgd2hldGhlciB0aGUgYXJtIGV2ZXIgZ290IHdpdGhpbiAyIGNtIG9mIGVhY2ggcGFyY2VsLgoiIiIKCmltcG9ydCBqc29uCmltcG9ydCBvcwoKaW1wb3J0IGh5ZHJhCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKZnJvbSBvbWVnYWNvbmYgaW1wb3J0IE9tZWdhQ29uZgoKZnJvbSB3YXJlaG91c2Vfc29ydC51dGlscyBpbXBvcnQgbG9hZF9hZ2VudCwgbG9nX3J1bl9oZWFkZXIsIG1ha2VfZW52LCB0b19kZXZpY2UKCgpkZWYgX3BvbGljeV9rd2FyZ3NfdG9fY29udGFpbmVyKHBvbGljeV9rd2FyZ3MpOgogICAgaWYgcG9saWN5X2t3YXJncyBpcyBOb25lOgogICAgICAgIHJldHVybiB7fQogICAgaWYgT21lZ2FDb25mLmlzX2NvbmZpZyhwb2xpY3lfa3dhcmdzKToKICAgICAgICByZXR1cm4gT21lZ2FDb25mLnRvX2NvbnRhaW5lcihwb2xpY3lfa3dhcmdzLCByZXNvbHZlPVRydWUpIG9yIHt9CiAgICByZXR1cm4gZGljdChwb2xpY3lfa3dhcmdzKQoKCkBoeWRyYS5tYWluKHZlcnNpb25fYmFzZT1Ob25lLCBjb25maWdfcGF0aD0iLi4vY29uZiIsIGNvbmZpZ19uYW1lPSJjb25maWciKQpkZWYgbWFpbihjZmcpOgogICAgYXNzZXJ0IGNmZy5jaGVja3BvaW50IGFuZCBjZmcucG9saWN5CiAgICBsb2dfcnVuX2hlYWRlcihjZmcsICJkaWFnIikKICAgIGRldmljZSA9IHRvcmNoLmRldmljZShjZmcuZGV2aWNlIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKICAgIGV2YWxfY2ZnID0gT21lZ2FDb25mLmxvYWQoY2ZnLmV2YWxfY29uZmlnKSBpZiBjZmcuZ2V0KCJldmFsX2NvbmZpZyIpIGVsc2UgTm9uZQogICAgc2VlZHMgPSBsaXN0KGV2YWxfY2ZnLmV2YWwuc2VlZHMpIGlmIGV2YWxfY2ZnIGVsc2UgWzUwMDAgKyBpIGZvciBpIGluIHJhbmdlKGNmZy5udW1fZW52cyldCiAgICByYW5kb21pemF0aW9uID0gKGV2YWxfY2ZnLmdldCgicmFuZG9taXphdGlvbiIsIE5vbmUpIGlmIGV2YWxfY2ZnIGVsc2UgTm9uZSkgb3IgY2ZnLnJhbmRvbWl6YXRpb24KICAgIG5fZXBzID0gaW50KGNmZy5nZXQoImxvZ19lcGlzb2RlcyIsIGNmZy5udW1fZW52cykpCiAgICBuX2VudnMgPSBtaW4oY2ZnLm51bV9lbnZzLCBuX2VwcykKICAgIGVudiwgXyA9IG1ha2VfZW52KGNmZywgY2ZnLm9ic19tb2RlLCByYW5kb21pemF0aW9uLCBudW1fZW52cz1uX2VudnMpCiAgICBiYXNlID0gZW52LnVud3JhcHBlZAogICAgcG9saWN5X2t3YXJncyA9IF9wb2xpY3lfa3dhcmdzX3RvX2NvbnRhaW5lcihjZmcuZ2V0KCJwb2xpY3lfa3dhcmdzIikpCiAgICBhZ2VudCwgXyA9IGxvYWRfYWdlbnQoY2ZnLmNoZWNrcG9pbnQsIGVudiwgZGV2aWNlLCBlbnRyeXBvaW50PWNmZy5wb2xpY3ksIHBvbGljeV9rd2FyZ3M9cG9saWN5X2t3YXJncykKCiAgICBvYnMsIF8gPSBlbnYucmVzZXQoc2VlZD1baW50KHMpIGZvciBzIGluIHNlZWRzWzpuX2VudnNdXSkKICAgIGlmIGhhc2F0dHIoYWdlbnQsICJyZXNldCIpOgogICAgICAgIGFnZW50LnJlc2V0KCkKICAgIG9icyA9IHRvX2RldmljZShvYnMsIGRldmljZSkKICAgIFQgPSBpbnQoY2ZnLm1heF9lcGlzb2RlX3N0ZXBzKQogICAgUCA9IGJhc2UubnVtX3BhcmNlbHMKICAgIGdyaXBfY21kID0gbnAuemVyb3MoKFQsIG5fZW52cyksIG5wLmZsb2F0MzIpCiAgICBncmFzcGVkID0gbnAuemVyb3MoKFQsIG5fZW52cyksIGJvb2wpCiAgICB0Y3AgPSBucC56ZXJvcygoVCwgbl9lbnZzLCAzKSwgbnAuZmxvYXQzMikKICAgIHBhcmNlbF94eSA9IG5wLnplcm9zKChULCBuX2VudnMsIFAsIDIpLCBucC5mbG9hdDMyKQogICAgcGFyY2VsX3ogPSBucC56ZXJvcygoVCwgbl9lbnZzLCBQKSwgbnAuZmxvYXQzMikKICAgIHNvcnRlZF9jbnQgPSBucC56ZXJvcygoVCwgbl9lbnZzKSwgbnAuZmxvYXQzMikKICAgIGZvciB0IGluIHJhbmdlKFQgLSAxKToKICAgICAgICBhID0gYWdlbnQuYWN0KG9icywgZGV0ZXJtaW5pc3RpYz1UcnVlKQogICAgICAgIG9icywgXywgXywgXywgaW5mbyA9IGVudi5zdGVwKGEpCiAgICAgICAgb2JzID0gdG9fZGV2aWNlKG9icywgZGV2aWNlKQogICAgICAgIGdyaXBfY21kW3RdID0gYVs6LCAzXS5kZXRhY2goKS5jcHUoKS5udW1weSgpCiAgICAgICAgZ3Jhc3BlZFt0XSA9IGluZm9bImlzX2dyYXNwZWQiXS5kZXRhY2goKS5jcHUoKS5udW1weSgpIGlmICJpc19ncmFzcGVkIiBpbiBpbmZvIGVsc2UgYmFzZS5ldmFsdWF0ZSgpWyJpc19ncmFzcGVkIl0uY3B1KCkubnVtcHkoKQogICAgICAgIHRjcFt0XSA9IGJhc2UuYWdlbnQudGNwX3Bvc2UucC5kZXRhY2goKS5jcHUoKS5udW1weSgpCiAgICAgICAgZm9yIGosIHBhcmNlbCBpbiBlbnVtZXJhdGUoYmFzZS5wYXJjZWxzKToKICAgICAgICAgICAgcHAgPSBwYXJjZWwucG9zZS5wLmRldGFjaCgpLmNwdSgpLm51bXB5KCkKICAgICAgICAgICAgcGFyY2VsX3h5W3QsIDosIGpdID0gcHBbOiwgOjJdCiAgICAgICAgICAgIHBhcmNlbF96W3QsIDosIGpdID0gcHBbOiwgMl0KICAgICAgICBzb3J0ZWRfY250W3RdID0gYmFzZS5ldmFsdWF0ZSgpWyJzdWNjZXNzX2NvdW50Il0uY3B1KCkubnVtcHkoKQoKICAgIHJlcG9ydCA9IFtdCiAgICBmb3IgZSBpbiByYW5nZShuX2VudnMpOgogICAgICAgIGcgPSBncmlwX2NtZFs6VCAtIDEsIGVdCiAgICAgICAgY2xvc2VkID0gZyA8IDAKICAgICAgICBmaXJzdF9jbG9zZSA9IGludChucC5hcmdtYXgoY2xvc2VkKSkgaWYgY2xvc2VkLmFueSgpIGVsc2UgLTEKICAgICAgICBmaXJzdF9ncmFzcCA9IGludChucC5hcmdtYXgoZ3Jhc3BlZFs6VCAtIDEsIGVdKSkgaWYgZ3Jhc3BlZFs6VCAtIDEsIGVdLmFueSgpIGVsc2UgLTEKICAgICAgICBmbGlwcyA9IGludChucC5zdW0obnAuYWJzKG5wLmRpZmYoY2xvc2VkLmFzdHlwZShpbnQpKSkpKQogICAgICAgICMgbG9uZ2VzdCBydW4gb2YgY29uc2VjdXRpdmUgY2xvc2UgY29tbWFuZHMKICAgICAgICBiZXN0ID0gcnVuID0gMAogICAgICAgIGZvciBjIGluIGNsb3NlZDoKICAgICAgICAgICAgcnVuID0gcnVuICsgMSBpZiBjIGVsc2UgMAogICAgICAgICAgICBiZXN0ID0gbWF4KGJlc3QsIHJ1bikKICAgICAgICBkID0gbnAubGluYWxnLm5vcm0odGNwWzpUIC0gMSwgZSwgTm9uZSwgOjJdIC0gcGFyY2VsX3h5WzpUIC0gMSwgZV0sIGF4aXM9LTEpICAgIyAoVCwgUCkKICAgICAgICBtaW5feHlfZXJyID0gZC5taW4oYXhpcz0wKS50b2xpc3QoKQogICAgICAgIHh5X2Vycl9hdF9jbG9zZSA9IGZsb2F0KGRbZmlyc3RfY2xvc2VdLm1pbigpKSBpZiBmaXJzdF9jbG9zZSA+PSAwIGVsc2UgTm9uZQogICAgICAgIHJlcG9ydC5hcHBlbmQoZGljdCgKICAgICAgICAgICAgZW52PWUsIHNlZWQ9aW50KHNlZWRzW2VdKSwgc29ydGVkPWZsb2F0KHNvcnRlZF9jbnRbVCAtIDIsIGVdKSwgcGFyY2Vscz1QLAogICAgICAgICAgICBmaXJzdF9jbG9zZV9zdGVwPWZpcnN0X2Nsb3NlLCBmaXJzdF9ncmFzcF9zdGVwPWZpcnN0X2dyYXNwLCBncmlwcGVyX2ZsaXBzPWZsaXBzLAogICAgICAgICAgICBsb25nZXN0X2Nsb3NlX3J1bj1iZXN0LCB4eV9lcnJfYXRfZmlyc3RfY2xvc2U9eHlfZXJyX2F0X2Nsb3NlLAogICAgICAgICAgICBtaW5feHlfZXJyX3Blcl9wYXJjZWw9W3JvdW5kKHgsIDQpIGZvciB4IGluIG1pbl94eV9lcnJdLAogICAgICAgICAgICByZWFjaGVkX3dpdGhpbl8yY209W2Jvb2woeCA8IDAuMDIpIGZvciB4IGluIG1pbl94eV9lcnJdLAogICAgICAgICAgICBuX3BsYW5zPWdldGF0dHIoYWdlbnQsICJuX3BsYW5zIiwgTm9uZSksCiAgICAgICAgKSkKICAgICAgICBwcmludChqc29uLmR1bXBzKHJlcG9ydFstMV0pKQogICAgcHJpbnQoZiJtZWFuIHNvcnRlZCB7bnAubWVhbihbclsnc29ydGVkJ10gZm9yIHIgaW4gcmVwb3J0XSk6LjJmfS97UH0gICIKICAgICAgICAgIGYiZ3Jhc3BlZC1hdC1sZWFzdC1vbmNlIHtucC5tZWFuKFtyWydmaXJzdF9ncmFzcF9zdGVwJ10gPj0gMCBmb3IgciBpbiByZXBvcnRdKTouMmZ9IikKICAgIG91dCA9IGNmZy5nZXQoImxvZ19vdXQiKQogICAgaWYgb3V0OgogICAgICAgIG9zLm1ha2VkaXJzKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgob3V0KSksIGV4aXN0X29rPVRydWUpCiAgICAgICAgd2l0aCBvcGVuKG91dCwgInciKSBhcyBmOgogICAgICAgICAgICBqc29uLmR1bXAoZGljdChjaGVja3BvaW50PWNmZy5jaGVja3BvaW50LCBsZXZlbD1jZmcuZGlmZmljdWx0eS5uYW1lLCBlcGlzb2Rlcz1yZXBvcnQpLCBmLCBpbmRlbnQ9MSkKICAgICAgICBucC5zYXZlel9jb21wcmVzc2VkKG91dC5yZXBsYWNlKCIuanNvbiIsICJfdHJhY2VzLm5weiIpLCBncmlwX2NtZD1ncmlwX2NtZCwgZ3Jhc3BlZD1ncmFzcGVkLCB0Y3A9dGNwLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgcGFyY2VsX3h5PXBhcmNlbF94eSwgcGFyY2VsX3o9cGFyY2VsX3osIHNvcnRlZF9jbnQ9c29ydGVkX2NudCkKICAgICAgICBwcmludCgid3JvdGUiLCBvdXQpCiAgICBlbnYuY2xvc2UoKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK'}, 'tools/run_colab_hard.py': {'base_sha256': None, 'sha256': '37426e40c1f6768765f128ca29012a302fba0f49b474d1b0584806d69f9c61c0', 'content_b64': '...'}, 'tools/run_colab_medium_resume.py': {'base_sha256': None, 'sha256': '68a5b247bc4263cf561bd155fd9ffbbb1ccee19397ec4b2d35f5be55a57b998f', 'content_b64': '...'}, 'warehouse_sort/__init__.py': {'base_sha256': '5b1e0ab8aec7f8c8916905c168bf68884e8c3dae2de6c1b01a49374272cfdd75', 'sha256': '5b1e0ab8aec7f8c8916905c168bf68884e8c3dae2de6c1b01a49374272cfdd75', 'content_b64': 'IiIiV2FyZWhvdXNlIENvbG91ci1Tb3J0IGhhY2thdGhvbiBzdGFydGVyIHBhY2thZ2UuCgpJbXBvcnRpbmcgdGhpcyBwYWNrYWdlIHJlZ2lzdGVycyB0aGUgV2FyZWhvdXNlU29ydC12MSBNYW5pU2tpbGwgZW52aXJvbm1lbnQuIFRoZSByZWdpc3RyYXRpb24gaXMKZ3VhcmRlZCBzbyB0aGF0IHRoZSBwb2xpY3kgbG9hZGVycyAoYHdhcmVob3VzZV9zb3J0LmlsX3BvbGljeWApIHN0YXkgaW1wb3J0YWJsZSBvbiBtYWNoaW5lcyB3aXRob3V0Ck1hbmlTa2lsbC9TQVBJRU4gKGUuZy4gZm9yIENQVSB1bml0IHRlc3RzKTsgZXZhbC5weSAvIHRoZSBqdWRnZSBhbHdheXMgcnVuIHdpdGggTWFuaVNraWxsIGluc3RhbGxlZC4KIiIiCgp0cnk6CiAgICBmcm9tIHdhcmVob3VzZV9zb3J0LmVudiBpbXBvcnQgV2FyZWhvdXNlU29ydEVudiAgIyBub3FhOiBGNDAxICAocmVnaXN0ZXJzIFdhcmVob3VzZVNvcnQtdjEpCmV4Y2VwdCBJbXBvcnRFcnJvciBhcyBfZTogICMgcHJhZ21hOiBubyBjb3ZlciAtIG9ubHkgb24gbWFjaGluZXMgd2l0aG91dCB0aGUgc2ltdWxhdG9yCiAgICBpbXBvcnQgd2FybmluZ3MKCiAgICB3YXJuaW5ncy53YXJuKGYid2FyZWhvdXNlX3NvcnQuZW52IG5vdCBpbXBvcnRlZCAoc2ltdWxhdG9yIG1pc3Npbmc/KToge19lfSIpCiAgICBXYXJlaG91c2VTb3J0RW52ID0gTm9uZSAgIyB0eXBlOiBpZ25vcmUKCl9fYWxsX18gPSBbIldhcmVob3VzZVNvcnRFbnYiXQo='}, 'warehouse_sort/act_policy.py': {'base_sha256': '2e35f8a9266327bd48caad81fb610c1df7127e9832039a02e618a409fbd80490', 'sha256': '2e35f8a9266327bd48caad81fb610c1df7127e9832039a02e618a409fbd80490', 'content_b64': 'IiIiQUNUIChBY3Rpb24gQ2h1bmtpbmcgVHJhbnNmb3JtZXIpIHBvbGljeSBlbnRyeXBvaW50IGZvciBldmFsLnB5IC8gdGhlIGp1ZGdlLgoKU2F0aXNmaWVzIHRoZSBwb2xpY3kgY29udHJhY3Q6CiAgICBwb2xpY3kuYWN0KG9icywgZGV0ZXJtaW5pc3RpYz1UcnVlKSAtPiBUZW5zb3IgKG51bV9lbnZzLCBhY3Rpb25fZGltKSBpbiBbLTEsIDFdCgpXaXJlIGl0IGluIHZpYSB0aGUgY29uZmlnIGBwb2xpY3lgIGZpZWxkOgogICAgcGl4aSBydW4gcHl0aG9uIGV2YWwucHkgZGlmZmljdWx0eT1lYXN5IFxcCiAgICAgICAgcG9saWN5PXdhcmVob3VzZV9zb3J0LmFjdF9wb2xpY3k6bG9hZF9hY3QgXFwKICAgICAgICBjaGVja3BvaW50PTxwYXRoPiBldmFsX2NvbmZpZz1jb25mL2V2YWwvZGVmYXVsdC55YW1sCgpOb3RlOiB1c2VzIHNpbXBsZSBjaHVuay1yZXBsYXkgKHByZWRpY3QgbnVtX3F1ZXJpZXMgYWN0aW9ucywgcGxheSB0aGVtIGJhY2sgb3Blbi1sb29wLCB0aGVuCnJlLXByZWRpY3QpIHJhdGhlciB0aGFuIEFDVCdzIG9wdGlvbmFsIHRlbXBvcmFsLWVuc2VtYmxpbmcgZXZhbCBtb2RlLCB3aGljaCBuZWVkcyBtb3JlIHN0YXRlCnRoYW4gY2h1bmstcmVwbGF5IHRvIHRyYWNrIGNvcnJlY3RseS4gX0FDVFBvbGljeSBleHBvc2VzIGEgcmVzZXQoKSBtZXRob2QgdGhhdApyb2xsb3V0X21ldHJpY3MvcmVjb3JkX2V2YWxfdmlkZW8gY2FsbCBiZWZvcmUgZWFjaCBuZXcgZXBpc29kZSAoc2VlIHdhcmVob3VzZV9zb3J0L3V0aWxzLnB5KSwKc28gdGhlIGJ1ZmZlcmVkIGNodW5rIGRvZXNuJ3QgY2Fycnkgc3RhbGUgYWN0aW9ucyBhY3Jvc3MgZXBpc29kZSBib3VuZGFyaWVzLgoiIiIKCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2h2aXNpb24udHJhbnNmb3JtcyBhcyBUCgoKZGVmIF9hZGRfYmFzZWxpbmVfcGF0aChyZWwpOgogICAgaW1wb3J0IG9zLCBzeXMKICAgIHAgPSBvcy5wYXRoLmFic3BhdGgob3MucGF0aC5qb2luKG9zLnBhdGguZGlybmFtZShfX2ZpbGVfXyksICIuLiIsICJpbCIsICJiYXNlbGluZXMiLCByZWwpKQogICAgaWYgcCBub3QgaW4gc3lzLnBhdGg6CiAgICAgICAgc3lzLnBhdGguaW5zZXJ0KDAsIHApCgoKY2xhc3MgX0FDVFBvbGljeToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBhZ2VudCwgbnVtX3F1ZXJpZXMsIGFjdF9ob3Jpem9uLCBkZXZpY2UpOgogICAgICAgIHNlbGYuYWdlbnQgPSBhZ2VudC50byhkZXZpY2UpLmV2YWwoKQogICAgICAgIHNlbGYubnVtX3F1ZXJpZXMgPSBudW1fcXVlcmllcwogICAgICAgIHNlbGYuYWN0X2hvcml6b24gPSBhY3RfaG9yaXpvbiAgIyBob3cgbWFueSBwcmVkaWN0ZWQgYWN0aW9ucyB0byByZXBsYXkgYmVmb3JlIHJlLXF1ZXJ5aW5nCiAgICAgICAgc2VsZi5kZXZpY2UgPSBkZXZpY2UKICAgICAgICBzZWxmLnJlc2l6ZSA9IFQuUmVzaXplKCgyMjQsIDIyNCksIGFudGlhbGlhcz1UcnVlKQogICAgICAgIHNlbGYuX2NodW5rID0gTm9uZQogICAgICAgIHNlbGYuX3N0ZXAgPSAwCgogICAgZGVmIHJlc2V0KHNlbGYpOgogICAgICAgICIiIkNhbGxlZCBieSByb2xsb3V0X21ldHJpY3MvcmVjb3JkX2V2YWxfdmlkZW8gYmVmb3JlIGVhY2ggbmV3IGVwaXNvZGUgLS0gd2l0aG91dCB0aGlzCiAgICAgICAgdGhlIGJ1ZmZlcmVkIGNodW5rIGNhcnJpZXMgb3ZlciBhY3Jvc3MgZXBpc29kZSBib3VuZGFyaWVzLCBzbyB0aGUgZmlyc3QgZmV3IHN0ZXBzIG9mIGEKICAgICAgICBmcmVzaCBlcGlzb2RlIHdvdWxkIHJlcGxheSBhY3Rpb25zIHBsYW5uZWQgZm9yIHRoZSBQUkVWSU9VUyBlcGlzb2RlJ3Mgb2JzZXJ2YXRpb24uIiIiCiAgICAgICAgc2VsZi5fY2h1bmsgPSBOb25lCiAgICAgICAgc2VsZi5fc3RlcCA9IDAKCiAgICBkZWYgX3ByZXBfb2JzKHNlbGYsIG9icyk6CiAgICAgICAgc3RhdGUgPSBvYnNbInN0YXRlIl0uZmxvYXQoKS50byhzZWxmLmRldmljZSkKICAgICAgICByZ2IgPSBvYnNbInJnYiJdLnRvKHNlbGYuZGV2aWNlKSAgIyAoTiwgSCwgVywgMykgdWludDgKICAgICAgICByZ2IgPSByZ2IucGVybXV0ZSgwLCAzLCAxLCAyKSAgIyAoTiwgMywgSCwgVykKICAgICAgICByZ2IgPSBzZWxmLnJlc2l6ZShyZ2IpICAjIChOLCAzLCAyMjQsIDIyNCkKICAgICAgICByZ2IgPSByZ2IudW5zcXVlZXplKDEpICAjIChOLCAxLCAzLCAyMjQsIDIyNCkgLS0gbnVtX2NhbXM9MQogICAgICAgIHJldHVybiB7InN0YXRlIjogc3RhdGUsICJyZ2IiOiByZ2J9CgogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIGFjdChzZWxmLCBvYnMsIGRldGVybWluaXN0aWM9VHJ1ZSk6CiAgICAgICAgaWYgc2VsZi5fY2h1bmsgaXMgTm9uZSBvciBzZWxmLl9zdGVwID49IHNlbGYuYWN0X2hvcml6b246CiAgICAgICAgICAgIG9ic19pbiA9IHNlbGYuX3ByZXBfb2JzKG9icykKICAgICAgICAgICAgc2VsZi5fY2h1bmsgPSBzZWxmLmFnZW50LmdldF9hY3Rpb24ob2JzX2luKSAgIyAoTiwgbnVtX3F1ZXJpZXMsIGFjdF9kaW0pLCBkZWx0YSBjb250cm9sIC0+IHVubm9ybWFsaXplZAogICAgICAgICAgICBzZWxmLl9zdGVwID0gMAogICAgICAgIGFjdGlvbiA9IHNlbGYuX2NodW5rWzosIHNlbGYuX3N0ZXBdCiAgICAgICAgc2VsZi5fc3RlcCArPSAxCiAgICAgICAgcmV0dXJuIGFjdGlvbi5jbGFtcCgtMS4wLCAxLjApCgoKZGVmIGxvYWRfYWN0KGNoZWNrcG9pbnQsIHNhbXBsZV9vYnMsIGFjdGlvbl9zcGFjZSwgZGV2aWNlLAogICAgICAgICAgICAgYmFja2JvbmU9InJlc25ldDE4IiwgbnVtX3F1ZXJpZXM9MzAsIGFjdF9ob3Jpem9uPTEwLAogICAgICAgICAgICAgaGlkZGVuX2RpbT0yNTYsIGVuY19sYXllcnM9MiwgZGVjX2xheWVycz00LCBkaW1fZmVlZGZvcndhcmQ9NTEyLAogICAgICAgICAgICAgbmhlYWRzPTgsIGRyb3BvdXQ9MC4xLCBwcmVfbm9ybT1GYWxzZSwgcG9zaXRpb25fZW1iZWRkaW5nPSJzaW5lIiwKICAgICAgICAgICAgIG1hc2tzPUZhbHNlLCBkaWxhdGlvbj1GYWxzZSwgbHJfYmFja2JvbmU9MWUtNSwga2xfd2VpZ2h0PTEwKToKICAgICIiIkxvYWQgYW4gQUNUIGNoZWNrcG9pbnQgdHJhaW5lZCBieSBpbC9iYXNlbGluZXMvYWN0L3RyYWluX3JnYmQucHkgKHVzZXMgRU1BIHdlaWdodHMpLgoKICAgIElmIHlvdSBjaGFuZ2VkIGFueSBhcmNoaXRlY3R1cmUgZmxhZyBmb3IgdHJhaW5pbmcgKGJhY2tib25lLCBoaWRkZW5fZGltLCBlbmNfbGF5ZXJzLAogICAgZGVjX2xheWVycywgZGltX2ZlZWRmb3J3YXJkLCBuaGVhZHMpLCBwYXNzIHRoZSBzYW1lIHZhbHVlIGhlcmUgb3IgdGhlIGNoZWNrcG9pbnQgd29uJ3QKICAgIGxvYWQuIGBgbnVtX3F1ZXJpZXNgYCBpcyB0aGUgZXhjZXB0aW9uIC0tIGV2YWwucHkvdGhlIGp1ZGdlIGNhbGwgdGhpcyBmdW5jdGlvbiB3aXRoIG5vCiAgICBleHRyYSBrd2FyZ3MsIGJ1dCBkaWZmZXJlbnQgdHJhaW5pbmcgcnVucyB1c2UgZGlmZmVyZW50IGNodW5rIHNpemVzIChlLmcuIGEgNjAtc3RlcCBjaHVuawogICAgZm9yIGEgbG9uZ2VyIG1heF9lcGlzb2RlX3N0ZXBzKSwgc28gaXQncyBpbmZlcnJlZCBkaXJlY3RseSBmcm9tIHRoZSBjaGVja3BvaW50J3MKICAgIGBgcXVlcnlfZW1iZWQud2VpZ2h0YGAgc2hhcGUgYmVsb3cgcmF0aGVyIHRoYW4gdHJ1c3RlZCBmcm9tIHRoZSBkZWZhdWx0L0NMSSB2YWx1ZS4KCiAgICBhY3RfaG9yaXpvbiBkZWZhdWx0cyB0byAxMCwgbm90IG51bV9xdWVyaWVzOiBncmFkaW5nIHVzZXMgcGxhaW4gY2h1bmstcmVwbGF5IChubwogICAgdGVtcG9yYWxfYWdnKSwgc28gcmVwbGF5aW5nIHRoZSBmdWxsIGNodW5rIG9wZW4tbG9vcCBiZWZvcmUgcmUtb2JzZXJ2aW5nIGlzIHRoZSBsZWFzdAogICAgcmVhY3RpdmUgb3B0aW9uLiBSZS1xdWVyeWluZyBldmVyeSB+MTAgc3RlcHMgdHJhZGVzIGEgYml0IG9mIGNvbXB1dGUgZm9yIG11Y2ggYmV0dGVyCiAgICByZWNvdmVyeSBmcm9tIGppdHRlci9iaW4tc3dhcCAoc2FtZSByZWFzb25pbmcgYXMgRFAncyBhY3RfaG9yaXpvbikuCiAgICAiIiIKICAgIGltcG9ydCB0eXBlcwogICAgX2FkZF9iYXNlbGluZV9wYXRoKCJhY3QiKQogICAgZnJvbSB0cmFpbl9yZ2JkIGltcG9ydCBBZ2VudAoKICAgIGNrcHQgPSB0b3JjaC5sb2FkKGNoZWNrcG9pbnQsIG1hcF9sb2NhdGlvbj1kZXZpY2UsIHdlaWdodHNfb25seT1GYWxzZSkKICAgIHN0YXRlX2RpY3QgPSBja3B0LmdldCgiZW1hX2FnZW50IiwgY2twdC5nZXQoImFnZW50IikpCiAgICBja3B0X251bV9xdWVyaWVzID0gc3RhdGVfZGljdFsibW9kZWwucXVlcnlfZW1iZWQud2VpZ2h0Il0uc2hhcGVbMF0KICAgIGlmIGNrcHRfbnVtX3F1ZXJpZXMgIT0gbnVtX3F1ZXJpZXM6CiAgICAgICAgcHJpbnQoZiJbYWN0X3BvbGljeV0gY2hlY2twb2ludCB3YXMgdHJhaW5lZCB3aXRoIG51bV9xdWVyaWVzPXtja3B0X251bV9xdWVyaWVzfSAiCiAgICAgICAgICAgICAgZiIoZGVmYXVsdCBpcyB7bnVtX3F1ZXJpZXN9KSAtLSB1c2luZyB7Y2twdF9udW1fcXVlcmllc30gdG8gbWF0Y2ggdGhlIGNoZWNrcG9pbnQiLAogICAgICAgICAgICAgIGZsdXNoPVRydWUpCiAgICAgICAgbnVtX3F1ZXJpZXMgPSBja3B0X251bV9xdWVyaWVzCiAgICBhY3RfaG9yaXpvbiA9IG1pbihhY3RfaG9yaXpvbiBvciBudW1fcXVlcmllcywgbnVtX3F1ZXJpZXMpCgogICAgc3RhdGVfZGltID0gc2FtcGxlX29ic1sic3RhdGUiXS5zaGFwZVsxXQogICAgYXJncyA9IHR5cGVzLlNpbXBsZU5hbWVzcGFjZSgKICAgICAgICBiYWNrYm9uZT1iYWNrYm9uZSwgbnVtX3F1ZXJpZXM9bnVtX3F1ZXJpZXMsIGhpZGRlbl9kaW09aGlkZGVuX2RpbSwKICAgICAgICBlbmNfbGF5ZXJzPWVuY19sYXllcnMsIGRlY19sYXllcnM9ZGVjX2xheWVycywgZGltX2ZlZWRmb3J3YXJkPWRpbV9mZWVkZm9yd2FyZCwKICAgICAgICBuaGVhZHM9bmhlYWRzLCBkcm9wb3V0PWRyb3BvdXQsIHByZV9ub3JtPXByZV9ub3JtLCBwb3NpdGlvbl9lbWJlZGRpbmc9cG9zaXRpb25fZW1iZWRkaW5nLAogICAgICAgIG1hc2tzPW1hc2tzLCBkaWxhdGlvbj1kaWxhdGlvbiwgbHJfYmFja2JvbmU9bHJfYmFja2JvbmUsIGtsX3dlaWdodD1rbF93ZWlnaHQsCiAgICAgICAgaW5jbHVkZV9kZXB0aD1GYWxzZSwKICAgICkKICAgIHN0dWIgPSB0eXBlcy5TaW1wbGVOYW1lc3BhY2UoCiAgICAgICAgc2luZ2xlX29ic2VydmF0aW9uX3NwYWNlPXsKICAgICAgICAgICAgInN0YXRlIjogdHlwZXMuU2ltcGxlTmFtZXNwYWNlKHNoYXBlPShzdGF0ZV9kaW0sKSksCiAgICAgICAgICAgICJyZ2IiOiB0eXBlcy5TaW1wbGVOYW1lc3BhY2Uoc2hhcGU9KDEsIDMsIDIyNCwgMjI0KSksICAjIChudW1fY2FtcywgQywgSCwgVykKICAgICAgICB9LAogICAgICAgIHNpbmdsZV9hY3Rpb25fc3BhY2U9dHlwZXMuU2ltcGxlTmFtZXNwYWNlKHNoYXBlPShhY3Rpb25fc3BhY2Uuc2hhcGVbMF0sKSksCiAgICApCgogICAgYWdlbnQgPSBBZ2VudChzdHViLCBhcmdzKQogICAgYWdlbnQubG9hZF9zdGF0ZV9kaWN0KHN0YXRlX2RpY3QpCiAgICByZXR1cm4gX0FDVFBvbGljeShhZ2VudCwgbnVtX3F1ZXJpZXMsIGFjdF9ob3Jpem9uLCBkZXZpY2UpCg=='}, 'warehouse_sort/constants.py': {'base_sha256': '5213d8ffcb4c0c0003f00967b4788b51c8f2f501e15d270b3227605349d352e0', 'sha256': '5213d8ffcb4c0c0003f00967b4788b51c8f2f501e15d270b3227605349d352e0', 'content_b64': 'IiIiQ29uc3RhbnRzIHNoYXJlZCBieSB0aGUgZW52aXJvbm1lbnQgYW5kIHRoZSBwb2xpY3kgbG9hZGVycy4KCktlcHQgZnJlZSBvZiBzaW11bGF0b3IgaW1wb3J0cyBzbyBgd2FyZWhvdXNlX3NvcnQuaWxfcG9saWN5YCBjYW4gYmUgaW1wb3J0ZWQgKGFuZCB1bml0LXRlc3RlZCkKb24gYSBtYWNoaW5lIHdpdGhvdXQgTWFuaVNraWxsL1NBUElFTi4KIiIiCgojIFJvYm90IGhvbWUgY29uZmlndXJhdGlvbiB1c2VkIGJ5IFdhcmVob3VzZVNvcnRFbnYuX2luaXRpYWxpemVfZXBpc29kZSAoNyBhcm0gam9pbnRzICsgMiBmaW5nZXJzKS4KU1RBUlRfUVBPUyA9IFswLjAsIDAuMzkyNywgMC4wLCAtMS45NjM1LCAwLjAsIDIuMzU2LCAwLjc4NTQsIDAuMDQsIDAuMDRdCgojIExheW91dCBvZiB0aGUgcHJvcHJpb2NlcHRpb24gdmVjdG9yIG9ic1sic3RhdGUiXSBvbiB0aGUgcmdiIHRyYWNrICgyNiBkaW1zKS4gUHJvZHVjZWQgYnkKIyBGbGF0dGVuUkdCRE9ic2VydmF0aW9uV3JhcHBlciAvIGJ1aWxkX3N0YXRlX29ic19leHRyYWN0b3IgaW4gdGhpcyBvcmRlcjoKIyAgIGFnZW50LnFwb3MgKDkpIHwgYWdlbnQucXZlbCAoOSkgfCBleHRyYS50Y3BfcG9zZSAoNzogeHl6ICsgcXVhdCB3eHl6KSB8IGV4dHJhLmlzX2dyYXNwZWQgKDEpClBST1BSSU9fRElNID0gMjYKUVBPU19TTElDRSA9IHNsaWNlKDAsIDkpClFWRUxfU0xJQ0UgPSBzbGljZSg5LCAxOCkKVENQX1BPU0VfU0xJQ0UgPSBzbGljZSgxOCwgMjUpCklTX0dSQVNQRURfSU5ERVggPSAyNQoKIyBBY3Rpb24gbGF5b3V0IChwZF9lZV9kZWx0YV9wb3MpOiBbZHgsIGR5LCBkeiwgZ3JpcHBlcl0sIGFsbCBpbiBbLTEsIDFdLgpBQ1RJT05fRElNID0gNApHUklQUEVSX0lOREVYID0gMwo='}, 'warehouse_sort/env.py': {'base_sha256': '5b7d1f91375c676615d0f5cac32e17f168215c3988729c65890573f5d8b010f3', 'sha256': '5b7d1f91375c676615d0f5cac32e17f168215c3988729c65890573f5d8b010f3', 'content_b64': '...'}, 'warehouse_sort/il_policy.py': {'base_sha256': '07a5958d292438cc48308ffa0b451a6c8f535cc309bab759ebba0090a4487e9b', 'sha256': '07a5958d292438cc48308ffa0b451a6c8f535cc309bab759ebba0090a4487e9b', 'content_b64': 'IiIiSW1pdGF0aW9uLWxlYXJuaW5nIHBvbGljeSBlbnRyeXBvaW50cyBmb3IgZXZhbC5weSAvIHRoZSBqdWRnZS4KCkVhY2ggbG9hZGVyIHNhdGlzZmllcyB0aGUgcG9saWN5IGNvbnRyYWN0OgogICAgbG9hZF9mbihjaGVja3BvaW50LCBzYW1wbGVfb2JzLCBhY3Rpb25fc3BhY2UsIGRldmljZSkgLT4gcG9saWN5CiAgICBwb2xpY3kuYWN0KG9icywgZGV0ZXJtaW5pc3RpYz1UcnVlKSAtPiBUZW5zb3IgKG51bV9lbnZzLCBhY3Rpb25fZGltKSBpbiBbLTEsIDFdCgpXaXJlIG9uZSBpbiB2aWEgdGhlIGNvbmZpZyBgcG9saWN5YCBmaWVsZDoKICAgIHB5dGhvbiBldmFsLnB5IGRpZmZpY3VsdHk9ZWFzeSBvYnNfbW9kZT1yZ2IgXFwKICAgICAgICBwb2xpY3k9d2FyZWhvdXNlX3NvcnQuaWxfcG9saWN5OmxvYWRfZHBfcmdiIFxcCiAgICAgICAgY2hlY2twb2ludD08cGF0aD4gZXZhbF9jb25maWc9Y29uZi9ldmFsL2V2YWwzMi55YW1sCgogIGxvYWRfZHAgICAgICAtLSBzdGF0ZSBEaWZmdXNpb24gUG9saWN5IChvbmUgY2hlY2twb2ludCBQRVIgbGV2ZWw7IHN0YXRlIGRpbSBkZXBlbmRzIG9uIHBhcmNlbHMpCiAgbG9hZF9kcF9yZ2IgIC0tIFJHQiBEaWZmdXNpb24gUG9saWN5IChzY2VuZSBpbWFnZSArIHByb3ByaW9jZXB0aW9uOyBvbmUgY2twdCBjYW4gc2VydmUgYWxsIGxldmVscykKCkRlcGxveW1lbnQgbWF0Y2hlcyB0aGUgdHJhaW5pbmctdGltZSBldmFsdWF0b3IgKGRpZmZ1c2lvbl9wb2xpY3kvZXZhbHVhdGUucHkgKyBGcmFtZVN0YWNrKToKICAqIGEgcmVhbCByb2xsaW5nIGhpc3Rvcnkgb2YgdGhlIGxhc3QgYG9ic19ob3Jpem9uYCBvYnNlcnZhdGlvbnMgY29uZGl0aW9ucyB0aGUgbW9kZWwsIGFuZAogICogZWFjaCBkaWZmdXNpb24gY2FsbCB5aWVsZHMgYGFjdF9ob3Jpem9uYCBhY3Rpb25zIHRoYXQgYXJlIGV4ZWN1dGVkIGJlZm9yZSByZS1wbGFubmluZwogICAgKHJlY2VkaW5nLWhvcml6b24gY2h1bmtpbmcpLiBUaGUgb3JpZ2luYWwgdGVtcGxhdGUgcmUtc2FtcGxlZCBhIGZyZXNoIHBsYW4gZXZlcnkgc3RlcCBhbmQKICAgIGV4ZWN1dGVkIG9ubHkgaXRzIGZpcnN0IGFjdGlvbiwgd2hpY2ggbWFkZSB0aGUgZ3JpcHBlciBjb21tYW5kIGZsaWNrZXIgYmV0d2VlbiBpbmRlcGVuZGVudAogICAgc2FtcGxlcyBhbmQgbmV2ZXIgbGF0Y2ggYSBncmFzcC4KCkFyY2hpdGVjdHVyZSAvIGhvcml6b24gaHlwZXJwYXJhbWV0ZXJzIGFyZSByZWFkIGZyb20gdGhlIGNoZWNrcG9pbnQncyBgY29uZmlnYCBkaWN0ICh3cml0dGVuIGJ5CnRoZSB0cmFpbmVycyksIHNvIHRoZSBqdWRnZSBjYW4gY2FsbCB0aGUgbG9hZGVyIHdpdGggbm8gZXh0cmEga3dhcmdzLiBFeHBsaWNpdCBrd2FyZ3Mgb3ZlcnJpZGUuCiIiIgoKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgZGVxdWUKZnJvbSB0eXBlcyBpbXBvcnQgU2ltcGxlTmFtZXNwYWNlCgppbXBvcnQgdG9yY2gKCmZyb20gd2FyZWhvdXNlX3NvcnQuY29uc3RhbnRzIGltcG9ydCBHUklQUEVSX0lOREVYLCBRUE9TX1NMSUNFLCBTVEFSVF9RUE9TCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCiMgZGVmYXVsdHMgKG11c3QgbWF0Y2ggaWwvY29uZi9tZXRob2QvZHAqLnlhbWwgd2hlbiBubyBgY29uZmlnYCBpcyBzdG9yZWQgaW4gdGhlIGNoZWNrcG9pbnQpCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKREVGQVVMVF9EUF9DT05GSUcgPSBkaWN0KAogICAgb2JzX2hvcml6b249MiwKICAgIGFjdF9ob3Jpem9uPTgsCiAgICBwcmVkX2hvcml6b249MTYsCiAgICBkaWZmdXNpb25fc3RlcF9lbWJlZF9kaW09NjQsCiAgICB1bmV0X2RpbXM9WzY0LCAxMjgsIDI1Nl0sCiAgICBuX2dyb3Vwcz04LAogICAgbnVtX2RpZmZ1c2lvbl9pdGVycz0xMDAsCiAgICBudW1faW5mZXJlbmNlX3N0ZXBzPTE2LCAgICMgZXZhbC1vbmx5IGtub2IgKEREUE0gc3RlcHMpOyB0cmFpbmluZyBhbHdheXMgdXNlcyBudW1fZGlmZnVzaW9uX2l0ZXJzCiAgICBzY2hlZHVsZXI9ImRkcG0iLCAgICAgICAgICMgImRkcG0iIHwgImRkaW0iIChldmFsLW9ubHkpCiAgICBncmlwcGVyX2JpbmFyaXplPUZhbHNlLCAgICMgc25hcCB0aGUgZ3JpcHBlciBjb21tYW5kIHRvICsvLTEgYXQgZGVwbG95bWVudCAoZXZhbC1vbmx5KQogICAgc2VlZD0wLCAgICAgICAgICAgICAgICAgICAjIHNlZWQgZm9yIHRoZSBkaWZmdXNpb24gbm9pc2UgZ2VuZXJhdG9yIChyZXByb2R1Y2libGUgZXZhbHMpCiAgICBhbXBfZXZhbD1GYWxzZSwgICAgICAgICAgICMgZnAxNiBhdXRvY2FzdCBmb3IgdGhlIHZpc3VhbCBlbmNvZGVyIGF0IGV2YWwgKHJnYiBvbmx5KQogICAgIyByZ2Itb25seQogICAgdmlzdWFsX2VuY29kZXI9InJlc25ldDE4IiwKICAgIG51bV9rcD0zMiwKKQoKCmRlZiBfYWRkX2Jhc2VsaW5lX3BhdGgocmVsKToKICAgIGltcG9ydCBvcwogICAgaW1wb3J0IHN5cwoKICAgIHAgPSBvcy5wYXRoLmFic3BhdGgob3MucGF0aC5qb2luKG9zLnBhdGguZGlybmFtZShfX2ZpbGVfXyksICIuLiIsICJpbCIsICJiYXNlbGluZXMiLCByZWwpKQogICAgaWYgcCBub3QgaW4gc3lzLnBhdGg6CiAgICAgICAgc3lzLnBhdGguaW5zZXJ0KDAsIHApCgoKZGVmIHJlc29sdmVfcG9saWN5X2NvbmZpZyhja3B0LCBvdmVycmlkZXM9Tm9uZSwgZGVmYXVsdHM9REVGQVVMVF9EUF9DT05GSUcpOgogICAgIiIiZGVmYXVsdHMgPCBjaGVja3BvaW50Wydjb25maWcnXSA8IGV4cGxpY2l0IChub24tTm9uZSkgb3ZlcnJpZGVzLiIiIgogICAgY2ZnID0gZGljdChkZWZhdWx0cykKICAgIHN0b3JlZCA9IGNrcHQuZ2V0KCJjb25maWciKSBpZiBpc2luc3RhbmNlKGNrcHQsIGRpY3QpIGVsc2UgTm9uZQogICAgaWYgaXNpbnN0YW5jZShzdG9yZWQsIGRpY3QpOgogICAgICAgIGNmZy51cGRhdGUoe2s6IHYgZm9yIGssIHYgaW4gc3RvcmVkLml0ZW1zKCkgaWYgdiBpcyBub3QgTm9uZX0pCiAgICBmb3IgaywgdiBpbiAob3ZlcnJpZGVzIG9yIHt9KS5pdGVtcygpOgogICAgICAgIGlmIHYgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGNmZ1trXSA9IHYKICAgIGlmIGlzaW5zdGFuY2UoY2ZnLmdldCgidW5ldF9kaW1zIiksICh0dXBsZSwgbGlzdCkpOgogICAgICAgIGNmZ1sidW5ldF9kaW1zIl0gPSBbaW50KHgpIGZvciB4IGluIGNmZ1sidW5ldF9kaW1zIl1dCiAgICByZXR1cm4gY2ZnCgoKZGVmIF9sb2FkX3N0YXRlX2RpY3QoY2twdCk6CiAgICAiIiJBY2NlcHQgZnVsbCB0cmFpbmVyIGNoZWNrcG9pbnRzICh7J2FnZW50JywnZW1hX2FnZW50JywuLi59KSBhbmQgc3RyaXBwZWQgc3VibWlzc2lvbgogICAgY2hlY2twb2ludHMgKHsnbW9kZWwnOiAuLi4sICdjb25maWcnOiAuLi59KS4gRU1BIHdlaWdodHMgYXJlIHByZWZlcnJlZC4iIiIKICAgIGZvciBrZXkgaW4gKCJlbWFfYWdlbnQiLCAibW9kZWwiLCAiYWdlbnQiKToKICAgICAgICBpZiBrZXkgaW4gY2twdCBhbmQgaXNpbnN0YW5jZShja3B0W2tleV0sIGRpY3QpOgogICAgICAgICAgICByZXR1cm4gY2twdFtrZXldCiAgICByYWlzZSBLZXlFcnJvcihmIm5vIHdlaWdodHMgZm91bmQgaW4gY2hlY2twb2ludCAoa2V5czoge2xpc3QoY2twdC5rZXlzKCkpfSkiKQoKCmRlZiBtYWtlX3NjaGVkdWxlcihraW5kLCBudW1fdHJhaW5fdGltZXN0ZXBzKToKICAgICIiIlNhbWUgbm9pc2Ugc2NoZWR1bGUgYXMgdHJhaW5pbmcgKHNxdWFyZWRjb3NfY2FwX3YyLCBlcHNpbG9uLCBjbGlwX3NhbXBsZSkuIiIiCiAgICBpZiBraW5kID09ICJkZGltIjoKICAgICAgICBmcm9tIGRpZmZ1c2Vycy5zY2hlZHVsZXJzLnNjaGVkdWxpbmdfZGRpbSBpbXBvcnQgRERJTVNjaGVkdWxlcgoKICAgICAgICByZXR1cm4gRERJTVNjaGVkdWxlcihudW1fdHJhaW5fdGltZXN0ZXBzPW51bV90cmFpbl90aW1lc3RlcHMsIGJldGFfc2NoZWR1bGU9InNxdWFyZWRjb3NfY2FwX3YyIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjbGlwX3NhbXBsZT1UcnVlLCBwcmVkaWN0aW9uX3R5cGU9ImVwc2lsb24iKQogICAgZnJvbSBkaWZmdXNlcnMuc2NoZWR1bGVycy5zY2hlZHVsaW5nX2RkcG0gaW1wb3J0IEREUE1TY2hlZHVsZXIKCiAgICByZXR1cm4gRERQTVNjaGVkdWxlcihudW1fdHJhaW5fdGltZXN0ZXBzPW51bV90cmFpbl90aW1lc3RlcHMsIGJldGFfc2NoZWR1bGU9InNxdWFyZWRjb3NfY2FwX3YyIiwKICAgICAgICAgICAgICAgICAgICAgICAgIGNsaXBfc2FtcGxlPVRydWUsIHByZWRpY3Rpb25fdHlwZT0iZXBzaWxvbiIpCgoKZGVmIGRpZmZ1c2lvbl9zYW1wbGUobmV0LCBzY2hlZHVsZXIsIG9ic19jb25kLCBwcmVkX2hvcml6b24sIGFjdF9kaW0sIGdlbmVyYXRvcj1Ob25lKToKICAgICIiIlJldmVyc2UtZGlmZnVzZSBhbiBhY3Rpb24gY2h1bmsgKEIsIHByZWRfaG9yaXpvbiwgYWN0X2RpbSkgY29uZGl0aW9uZWQgb24gb2JzX2NvbmQuIiIiCiAgICBCID0gb2JzX2NvbmQuc2hhcGVbMF0KICAgIGRldmljZSA9IG9ic19jb25kLmRldmljZQogICAgbmFjdGlvbiA9IHRvcmNoLnJhbmRuKChCLCBwcmVkX2hvcml6b24sIGFjdF9kaW0pLCBkZXZpY2U9ZGV2aWNlLCBnZW5lcmF0b3I9Z2VuZXJhdG9yKQogICAgZm9yIGsgaW4gc2NoZWR1bGVyLnRpbWVzdGVwczoKICAgICAgICBub2lzZV9wcmVkID0gbmV0KHNhbXBsZT1uYWN0aW9uLCB0aW1lc3RlcD1rLCBnbG9iYWxfY29uZD1vYnNfY29uZCkKICAgICAgICAjIGdlbmVyYXRvciBhbHNvIGRyaXZlcyB0aGUgRERQTSB2YXJpYW5jZSBub2lzZSBpbnNpZGUgc3RlcCgpOyB3aXRob3V0IGl0IGV2YWxzIGFyZSBub3QgcmVwcm9kdWNpYmxlCiAgICAgICAgbmFjdGlvbiA9IHNjaGVkdWxlci5zdGVwKG1vZGVsX291dHB1dD1ub2lzZV9wcmVkLCB0aW1lc3RlcD1rLCBzYW1wbGU9bmFjdGlvbiwgZ2VuZXJhdG9yPWdlbmVyYXRvcikucHJldl9zYW1wbGUKICAgIHJldHVybiBuYWN0aW9uCgoKY2xhc3MgQ2h1bmtlZFBvbGljeToKICAgICIiIlJlY2VkaW5nLWhvcml6b24gZXhlY3V0b3Igd2l0aCByb2xsaW5nIG9ic2VydmF0aW9uIGhpc3RvcnkgYW5kIGVwaXNvZGUtcmVzZXQgZGV0ZWN0aW9uLgoKICAgIFN1YmNsYXNzZXMgaW1wbGVtZW50IGBfcHJlcChvYnMpIC0+IGRpY3Rbc3RyLCBUZW5zb3JdYCAodGVuc29ycyBvbiBzZWxmLmRldmljZSwgYmF0Y2ggZmlyc3QpCiAgICBhbmQgYF9wbGFuKG9ic19zZXEpIC0+IFRlbnNvciAoQiwgYWN0X2hvcml6b24sIGFjdF9kaW0pYCB3aGVyZSBvYnNfc2VxIHN0YWNrcyB0aGUgaGlzdG9yeSBhbG9uZwogICAgZGltIDEuIGByZXNldCgpYCBpcyBjYWxsZWQgYnkgd2FyZWhvdXNlX3NvcnQudXRpbHMgYmVmb3JlIGV2ZXJ5IGVwaXNvZGUgYmF0Y2g7IGJlY2F1c2UgdGhlIGp1ZGdlCiAgICBtaWdodCBydW4gYSBoYXJuZXNzIHRoYXQgbmV2ZXIgY2FsbHMgaXQsIGEgcmVzZXQgaXMgQUxTTyBkZXRlY3RlZCBmcm9tIHByb3ByaW9jZXB0aW9uOiBhIGZyZXNoCiAgICBlcGlzb2RlIHN0YXJ0cyBhdCB0aGUgZXhhY3QgaG9tZSBjb25maWd1cmF0aW9uIChTVEFSVF9RUE9TKSBvciB0ZWxlcG9ydHMgdGhlIGpvaW50cy4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBvYnNfaG9yaXpvbiwgYWN0X2hvcml6b24sIGRldmljZSwgZ3JpcHBlcl9iaW5hcml6ZT1GYWxzZSwKICAgICAgICAgICAgICAgICBob21lX3RvbD0wLjAzLCBqdW1wX3RvbD0wLjMpOgogICAgICAgIHNlbGYub2JzX2hvcml6b24gPSBpbnQob2JzX2hvcml6b24pCiAgICAgICAgc2VsZi5hY3RfaG9yaXpvbiA9IGludChhY3RfaG9yaXpvbikKICAgICAgICBzZWxmLmRldmljZSA9IHRvcmNoLmRldmljZShkZXZpY2UpCiAgICAgICAgc2VsZi5ncmlwcGVyX2JpbmFyaXplID0gYm9vbChncmlwcGVyX2JpbmFyaXplKQogICAgICAgIHNlbGYuaG9tZV90b2wgPSBmbG9hdChob21lX3RvbCkKICAgICAgICBzZWxmLmp1bXBfdG9sID0gZmxvYXQoanVtcF90b2wpCiAgICAgICAgc2VsZi5faG9tZSA9IHRvcmNoLnRlbnNvcihTVEFSVF9RUE9TWzo3XSwgZHR5cGU9dG9yY2guZmxvYXQzMiwgZGV2aWNlPXNlbGYuZGV2aWNlKQogICAgICAgIHNlbGYucmVzZXQoKQoKICAgICMgLS0gZXBpc29kZSBib29ra2VlcGluZyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKICAgIGRlZiByZXNldChzZWxmKToKICAgICAgICBzZWxmLl9oaXN0ID0gTm9uZSAgICAgICMgZGVxdWUgb2YgcHJlcGFyZWQgb2JzIGRpY3RzIChsZW4gPD0gb2JzX2hvcml6b24pCiAgICAgICAgc2VsZi5fcXVldWUgPSBbXSAgICAgICAjIG5vdC15ZXQtZXhlY3V0ZWQgYWN0aW9ucyBmcm9tIHRoZSBsYXN0IHBsYW4KICAgICAgICBzZWxmLl9iYXRjaCA9IE5vbmUgICAgICMgYmF0Y2ggc2l6ZSBvZiB0aGUgY3VycmVudCByb2xsb3V0CiAgICAgICAgc2VsZi5fcHJldl9xcG9zID0gTm9uZQogICAgICAgIHNlbGYubl9wbGFucyA9IDAKCiAgICBkZWYgX2F0X2hvbWUoc2VsZiwgcXBvczcpOgogICAgICAgIHJldHVybiAocXBvczcgLSBzZWxmLl9ob21lKS5hYnMoKS5tYXgoZGltPTEpLnZhbHVlcyA8IHNlbGYuaG9tZV90b2wKCiAgICBkZWYgX2RldGVjdF9yZXNldChzZWxmLCBxcG9zNyk6CiAgICAgICAgaWYgc2VsZi5fcHJldl9xcG9zIGlzIE5vbmUgb3Igc2VsZi5fcHJldl9xcG9zLnNoYXBlICE9IHFwb3M3LnNoYXBlOgogICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgICAgIHJldHVybmVkX2hvbWUgPSBzZWxmLl9hdF9ob21lKHFwb3M3KSAmIH5zZWxmLl9hdF9ob21lKHNlbGYuX3ByZXZfcXBvcykKICAgICAgICBqdW1wZWQgPSAocXBvczcgLSBzZWxmLl9wcmV2X3Fwb3MpLmFicygpLm1heChkaW09MSkudmFsdWVzID4gc2VsZi5qdW1wX3RvbAogICAgICAgIHJldHVybiBib29sKChyZXR1cm5lZF9ob21lIHwganVtcGVkKS5hbnkoKSkKCiAgICBkZWYgX3N0YWNrKHNlbGYpOgogICAgICAgIGtleXMgPSBzZWxmLl9oaXN0WzBdLmtleXMoKQogICAgICAgIHJldHVybiB7azogdG9yY2guc3RhY2soW2hba10gZm9yIGggaW4gc2VsZi5faGlzdF0sIGRpbT0xKSBmb3IgayBpbiBrZXlzfQoKICAgICMgLS0gY29udHJhY3QgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKICAgIEB0b3JjaC5ub19ncmFkKCkKICAgIGRlZiBhY3Qoc2VsZiwgb2JzLCBkZXRlcm1pbmlzdGljPVRydWUpOgogICAgICAgIGN1ciA9IHNlbGYuX3ByZXAob2JzKQogICAgICAgIHFwb3M3ID0gY3VyWyJzdGF0ZSJdWzosIFFQT1NfU0xJQ0VdWzosIDo3XS5mbG9hdCgpCiAgICAgICAgQiA9IHFwb3M3LnNoYXBlWzBdCiAgICAgICAgaWYgc2VsZi5faGlzdCBpcyBOb25lIG9yIHNlbGYuX2JhdGNoICE9IEIgb3Igc2VsZi5fZGV0ZWN0X3Jlc2V0KHFwb3M3KToKICAgICAgICAgICAgc2VsZi5faGlzdCA9IGRlcXVlKFtjdXJdICogc2VsZi5vYnNfaG9yaXpvbiwgbWF4bGVuPXNlbGYub2JzX2hvcml6b24pCiAgICAgICAgICAgIHNlbGYuX3F1ZXVlID0gW10KICAgICAgICAgICAgc2VsZi5fYmF0Y2ggPSBCCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2VsZi5faGlzdC5hcHBlbmQoY3VyKQogICAgICAgIHNlbGYuX3ByZXZfcXBvcyA9IHFwb3M3CgogICAgICAgIGlmIG5vdCBzZWxmLl9xdWV1ZToKICAgICAgICAgICAgcGxhbiA9IHNlbGYuX3BsYW4oc2VsZi5fc3RhY2soKSkuY2xhbXAoLTEuMCwgMS4wKSAgICAgICMgKEIsIGFjdF9ob3Jpem9uLCBhY3RfZGltKQogICAgICAgICAgICBzZWxmLl9xdWV1ZSA9IGxpc3QocGxhbi51bmJpbmQoZGltPTEpKQogICAgICAgICAgICBzZWxmLm5fcGxhbnMgKz0gMQogICAgICAgIGEgPSBzZWxmLl9xdWV1ZS5wb3AoMCkKICAgICAgICBpZiBzZWxmLmdyaXBwZXJfYmluYXJpemU6CiAgICAgICAgICAgIGEgPSBhLmNsb25lKCkKICAgICAgICAgICAgYVs6LCBHUklQUEVSX0lOREVYXSA9IHRvcmNoLndoZXJlKGFbOiwgR1JJUFBFUl9JTkRFWF0gPj0gMCwgMS4wLCAtMS4wKQogICAgICAgIHJldHVybiBhCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwojIFN0YXRlIERpZmZ1c2lvbiBQb2xpY3kgKHByaXZpbGVnZWQgbG93LWRpbSBvYnM7IG9uZSBjaGVja3BvaW50IHBlciBsZXZlbCkKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwpjbGFzcyBfRFBQb2xpY3koQ2h1bmtlZFBvbGljeSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgbmV0LCBzY2hlZHVsZXIsIGNmZywgYWN0X2RpbSwgZGV2aWNlKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKGNmZ1sib2JzX2hvcml6b24iXSwgY2ZnWyJhY3RfaG9yaXpvbiJdLCBkZXZpY2UsIGNmZy5nZXQoImdyaXBwZXJfYmluYXJpemUiLCBGYWxzZSkpCiAgICAgICAgc2VsZi5uZXQgPSBuZXQudG8oZGV2aWNlKS5ldmFsKCkKICAgICAgICBzZWxmLnNjaGVkdWxlciA9IHNjaGVkdWxlcgogICAgICAgIHNlbGYuc2NoZWR1bGVyLnNldF90aW1lc3RlcHMoaW50KGNmZ1sibnVtX2luZmVyZW5jZV9zdGVwcyJdKSkKICAgICAgICBzZWxmLnByZWRfaG9yaXpvbiA9IGludChjZmdbInByZWRfaG9yaXpvbiJdKQogICAgICAgIHNlbGYuYWN0X2RpbSA9IGludChhY3RfZGltKQogICAgICAgIHNlbGYuZ2VuZXJhdG9yID0gdG9yY2guR2VuZXJhdG9yKGRldmljZT1zZWxmLmRldmljZSkubWFudWFsX3NlZWQoaW50KGNmZy5nZXQoInNlZWQiLCAwKSkpCgogICAgZGVmIF9wcmVwKHNlbGYsIG9icyk6CiAgICAgICAgc3RhdGUgPSBvYnNbInN0YXRlIl0gaWYgaXNpbnN0YW5jZShvYnMsIGRpY3QpIGVsc2Ugb2JzCiAgICAgICAgcmV0dXJuIHsic3RhdGUiOiBzdGF0ZS5mbG9hdCgpLnRvKHNlbGYuZGV2aWNlKX0KCiAgICBkZWYgX3BsYW4oc2VsZiwgb2JzX3NlcSk6CiAgICAgICAgb2JzX2NvbmQgPSBvYnNfc2VxWyJzdGF0ZSJdLmZsYXR0ZW4oc3RhcnRfZGltPTEpICAgICAgICAgICMgKEIsIG9ic19ob3Jpem9uICogb2JzX2RpbSkKICAgICAgICBuYWN0aW9uID0gZGlmZnVzaW9uX3NhbXBsZShzZWxmLm5ldCwgc2VsZi5zY2hlZHVsZXIsIG9ic19jb25kLCBzZWxmLnByZWRfaG9yaXpvbiwgc2VsZi5hY3RfZGltLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGdlbmVyYXRvcj1zZWxmLmdlbmVyYXRvcikKICAgICAgICBzdGFydCA9IHNlbGYub2JzX2hvcml6b24gLSAxCiAgICAgICAgcmV0dXJuIG5hY3Rpb25bOiwgc3RhcnQ6c3RhcnQgKyBzZWxmLmFjdF9ob3Jpem9uXQoKCmRlZiBsb2FkX2RwKGNoZWNrcG9pbnQsIHNhbXBsZV9vYnMsIGFjdGlvbl9zcGFjZSwgZGV2aWNlLCAqKm92ZXJyaWRlcyk6CiAgICAiIiJMb2FkIGEgc3RhdGUgRGlmZnVzaW9uIFBvbGljeSBjaGVja3BvaW50IChFTUEgd2VpZ2h0cykuIEh5cGVycGFyYW1ldGVycyBjb21lIGZyb20gdGhlCiAgICBjaGVja3BvaW50J3MgYGNvbmZpZ2A7IGt3YXJncyBvdmVycmlkZSAoZS5nLiBhY3RfaG9yaXpvbj00LCBudW1faW5mZXJlbmNlX3N0ZXBzPTMyKS4iIiIKICAgIF9hZGRfYmFzZWxpbmVfcGF0aCgiZGlmZnVzaW9uX3BvbGljeSIpCiAgICBmcm9tIGRpZmZ1c2lvbl9wb2xpY3kuY29uZGl0aW9uYWxfdW5ldDFkIGltcG9ydCBDb25kaXRpb25hbFVuZXQxRAoKICAgIGNrcHQgPSB0b3JjaC5sb2FkKGNoZWNrcG9pbnQsIG1hcF9sb2NhdGlvbj1kZXZpY2UsIHdlaWdodHNfb25seT1GYWxzZSkKICAgIGNmZyA9IHJlc29sdmVfcG9saWN5X2NvbmZpZyhja3B0LCBvdmVycmlkZXMpCiAgICBzdGF0ZSA9IHNhbXBsZV9vYnNbInN0YXRlIl0gaWYgaXNpbnN0YW5jZShzYW1wbGVfb2JzLCBkaWN0KSBlbHNlIHNhbXBsZV9vYnMKICAgIG9ic19kaW0gPSBzdGF0ZS5zaGFwZVsxXQogICAgYWN0X2RpbSA9IGFjdGlvbl9zcGFjZS5zaGFwZVswXQogICAgbmV0ID0gQ29uZGl0aW9uYWxVbmV0MUQoCiAgICAgICAgaW5wdXRfZGltPWFjdF9kaW0sIGdsb2JhbF9jb25kX2RpbT1jZmdbIm9ic19ob3Jpem9uIl0gKiBvYnNfZGltLAogICAgICAgIGRpZmZ1c2lvbl9zdGVwX2VtYmVkX2RpbT1jZmdbImRpZmZ1c2lvbl9zdGVwX2VtYmVkX2RpbSJdLAogICAgICAgIGRvd25fZGltcz1saXN0KGNmZ1sidW5ldF9kaW1zIl0pLCBuX2dyb3Vwcz1jZmdbIm5fZ3JvdXBzIl0sCiAgICApCiAgICBzZCA9IF9sb2FkX3N0YXRlX2RpY3QoY2twdCkKICAgIG5ldF9zZCA9IHtrLnJlcGxhY2UoIm5vaXNlX3ByZWRfbmV0LiIsICIiLCAxKTogdiBmb3IgaywgdiBpbiBzZC5pdGVtcygpIGlmIGsuc3RhcnRzd2l0aCgibm9pc2VfcHJlZF9uZXQuIil9CiAgICBuZXQubG9hZF9zdGF0ZV9kaWN0KG5ldF9zZCkKICAgIHNjaGVkdWxlciA9IG1ha2Vfc2NoZWR1bGVyKGNmZy5nZXQoInNjaGVkdWxlciIsICJkZHBtIiksIGNmZ1sibnVtX2RpZmZ1c2lvbl9pdGVycyJdKQogICAgcmV0dXJuIF9EUFBvbGljeShuZXQsIHNjaGVkdWxlciwgY2ZnLCBhY3RfZGltLCBkZXZpY2UpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwojIFJHQiBEaWZmdXNpb24gUG9saWN5IChzY2VuZSBpbWFnZSArIHByb3ByaW9jZXB0aW9uOyBOTyBwcml2aWxlZ2VkIHN0YXRlKQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCmNsYXNzIF9EUFJnYlBvbGljeShDaHVua2VkUG9saWN5KToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBhZ2VudCwgY2ZnLCBkZXZpY2UpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oY2ZnWyJvYnNfaG9yaXpvbiJdLCBjZmdbImFjdF9ob3Jpem9uIl0sIGRldmljZSwgY2ZnLmdldCgiZ3JpcHBlcl9iaW5hcml6ZSIsIEZhbHNlKSkKICAgICAgICBzZWxmLmFnZW50ID0gYWdlbnQudG8oZGV2aWNlKS5ldmFsKCkKICAgICAgICBzZWxmLmFnZW50Lm5vaXNlX3NjaGVkdWxlciA9IG1ha2Vfc2NoZWR1bGVyKGNmZy5nZXQoInNjaGVkdWxlciIsICJkZHBtIiksIGNmZ1sibnVtX2RpZmZ1c2lvbl9pdGVycyJdKQogICAgICAgIHNlbGYuYWdlbnQubm9pc2Vfc2NoZWR1bGVyLnNldF90aW1lc3RlcHMoaW50KGNmZ1sibnVtX2luZmVyZW5jZV9zdGVwcyJdKSkKICAgICAgICBzZWxmLmFnZW50LmFtcCA9IGJvb2woY2ZnLmdldCgiYW1wX2V2YWwiLCBGYWxzZSkpCiAgICAgICAgc2VsZi5nZW5lcmF0b3IgPSB0b3JjaC5HZW5lcmF0b3IoZGV2aWNlPXNlbGYuZGV2aWNlKS5tYW51YWxfc2VlZChpbnQoY2ZnLmdldCgic2VlZCIsIDApKSkKICAgICAgICBzZWxmLmNmZyA9IGNmZwoKICAgIGRlZiBfcHJlcChzZWxmLCBvYnMpOgogICAgICAgIHJldHVybiB7InN0YXRlIjogb2JzWyJzdGF0ZSJdLmZsb2F0KCkudG8oc2VsZi5kZXZpY2UpLCAicmdiIjogb2JzWyJyZ2IiXS50byhzZWxmLmRldmljZSl9CgogICAgZGVmIF9wbGFuKHNlbGYsIG9ic19zZXEpOgogICAgICAgICMgb2JzX3NlcVsicmdiIl06IChCLCBvYnNfaG9yaXpvbiwgSCwgVywgMykgdWludDggLS0gQWdlbnQuZ2V0X2FjdGlvbiBwZXJtdXRlcyB0byBjaGFubmVscy1maXJzdAogICAgICAgIHJldHVybiBzZWxmLmFnZW50LmdldF9hY3Rpb24ob2JzX3NlcSwgZ2VuZXJhdG9yPXNlbGYuZ2VuZXJhdG9yKSAgICMgKEIsIGFjdF9ob3Jpem9uLCBhY3RfZGltKQoKCmRlZiBsb2FkX2RwX3JnYihjaGVja3BvaW50LCBzYW1wbGVfb2JzLCBhY3Rpb25fc3BhY2UsIGRldmljZSwgKipvdmVycmlkZXMpOgogICAgIiIiTG9hZCBhbiBSR0IgRGlmZnVzaW9uIFBvbGljeSBjaGVja3BvaW50ICh2ZW5kb3JlZCB0cmFpbl9yZ2JkLkFnZW50OyBFTUEgd2VpZ2h0cykuIiIiCiAgICBpbXBvcnQgbnVtcHkgYXMgbnAKICAgIGltcG9ydCBneW1uYXNpdW0uc3BhY2VzIGFzIHNwYWNlcwoKICAgIF9hZGRfYmFzZWxpbmVfcGF0aCgiZGlmZnVzaW9uX3BvbGljeSIpCiAgICBmcm9tIHRyYWluX3JnYmQgaW1wb3J0IEFnZW50CgogICAgY2twdCA9IHRvcmNoLmxvYWQoY2hlY2twb2ludCwgbWFwX2xvY2F0aW9uPWRldmljZSwgd2VpZ2h0c19vbmx5PUZhbHNlKQogICAgY2ZnID0gcmVzb2x2ZV9wb2xpY3lfY29uZmlnKGNrcHQsIG92ZXJyaWRlcykKICAgIGgsIHcsIGMgPSBzYW1wbGVfb2JzWyJyZ2IiXS5zaGFwZVsxOl0KICAgIHN0YXRlX2RpbSA9IHNhbXBsZV9vYnNbInN0YXRlIl0uc2hhcGVbMV0KICAgIG9oID0gaW50KGNmZ1sib2JzX2hvcml6b24iXSkKICAgIHN0dWIgPSBTaW1wbGVOYW1lc3BhY2UoCiAgICAgICAgc2luZ2xlX29ic2VydmF0aW9uX3NwYWNlPXNwYWNlcy5EaWN0KHsKICAgICAgICAgICAgInN0YXRlIjogc3BhY2VzLkJveCgtbnAuaW5mLCBucC5pbmYsIChvaCwgc3RhdGVfZGltKSwgbnAuZmxvYXQzMiksCiAgICAgICAgICAgICJyZ2IiOiBzcGFjZXMuQm94KDAsIDI1NSwgKG9oLCBoLCB3LCBjKSwgbnAudWludDgpLAogICAgICAgIH0pLAogICAgICAgIHNpbmdsZV9hY3Rpb25fc3BhY2U9c3BhY2VzLkJveCgtMS4wLCAxLjAsIChhY3Rpb25fc3BhY2Uuc2hhcGVbMF0sKSwgbnAuZmxvYXQzMiksCiAgICApCiAgICBhcmdzID0gU2ltcGxlTmFtZXNwYWNlKAogICAgICAgIG9ic19ob3Jpem9uPW9oLCBhY3RfaG9yaXpvbj1pbnQoY2ZnWyJhY3RfaG9yaXpvbiJdKSwgcHJlZF9ob3Jpem9uPWludChjZmdbInByZWRfaG9yaXpvbiJdKSwKICAgICAgICBkaWZmdXNpb25fc3RlcF9lbWJlZF9kaW09aW50KGNmZ1siZGlmZnVzaW9uX3N0ZXBfZW1iZWRfZGltIl0pLCB1bmV0X2RpbXM9bGlzdChjZmdbInVuZXRfZGltcyJdKSwKICAgICAgICBuX2dyb3Vwcz1pbnQoY2ZnWyJuX2dyb3VwcyJdKSwgdmlzdWFsX2VuY29kZXI9Y2ZnWyJ2aXN1YWxfZW5jb2RlciJdLCBudW1fa3A9aW50KGNmZ1sibnVtX2twIl0pLAogICAgICAgIGltYWdlX2F1Z19wYWQ9MCwgcHJvcHJpb19ub2lzZV9zdGQ9MC4wLCBhbXA9RmFsc2UsCiAgICApCiAgICBhZ2VudCA9IEFnZW50KHN0dWIsIGFyZ3MpCiAgICBhZ2VudC5sb2FkX3N0YXRlX2RpY3QoX2xvYWRfc3RhdGVfZGljdChja3B0KSkKICAgIHJldHVybiBfRFBSZ2JQb2xpY3koYWdlbnQsIGNmZywgZGV2aWNlKQoKCiMgQ29udmVuaWVuY2UgZW50cnlwb2ludHMgZm9yIHN1Ym1pc3Npb24ueWFtbCAodGhlIGp1ZGdlIHBhc3NlcyBubyBrd2FyZ3MpLiBBZGQgbW9yZSBhcyBuZWVkZWQuCmRlZiBsb2FkX2RwX3JnYl9haDQoY2hlY2twb2ludCwgc2FtcGxlX29icywgYWN0aW9uX3NwYWNlLCBkZXZpY2UsICoqa3cpOgogICAga3cuc2V0ZGVmYXVsdCgiYWN0X2hvcml6b24iLCA0KQogICAgcmV0dXJuIGxvYWRfZHBfcmdiKGNoZWNrcG9pbnQsIHNhbXBsZV9vYnMsIGFjdGlvbl9zcGFjZSwgZGV2aWNlLCAqKmt3KQo='}, 'warehouse_sort/utils.py': {'base_sha256': '586e47affc15bd80ab66ae2244a63ea218922a37f84f9c6a363339d0655cafd6', 'sha256': '699eec100ea65c59fb94dc795e15df03bd75c413e8b7f6528e7791ab97a5bfde', 'content_b64': 'IiIiU2hhcmVkIGhlbHBlcnMgZm9yIGV2YWwucHkgLyB0aGUganVkZ2U6IGVudiBjb25zdHJ1Y3Rpb24sIGNvbmZpZyBsb2dnaW5nLCByb2xsb3V0LiIiIgoKaW1wb3J0IHN1YnByb2Nlc3MKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsCgppbXBvcnQgZ3ltbmFzaXVtIGFzIGd5bQppbXBvcnQgdG9yY2gKZnJvbSBvbWVnYWNvbmYgaW1wb3J0IE9tZWdhQ29uZgoKaW1wb3J0IHdhcmVob3VzZV9zb3J0ICAjIG5vcWE6IEY0MDEgIChyZWdpc3RlcnMgV2FyZWhvdXNlU29ydC12MSkKZnJvbSBtYW5pX3NraWxsLnV0aWxzLndyYXBwZXJzLmZsYXR0ZW4gaW1wb3J0IEZsYXR0ZW5SR0JET2JzZXJ2YXRpb25XcmFwcGVyCmZyb20gbWFuaV9za2lsbC52ZWN0b3Iud3JhcHBlcnMuZ3ltbmFzaXVtIGltcG9ydCBNYW5pU2tpbGxWZWN0b3JFbnYKCgpkZWYgX2d5bV9tYWtlKGNmZywgb2JzX21vZGUsIHJhbmRvbWl6YXRpb24sIG4sIHJlbmRlcl9tb2RlKToKICAgIHJldHVybiBneW0ubWFrZSgKICAgICAgICAiV2FyZWhvdXNlU29ydC12MSIsCiAgICAgICAgbnVtX2VudnM9biwKICAgICAgICBvYnNfbW9kZT1vYnNfbW9kZSwKICAgICAgICBjb250cm9sX21vZGU9Y2ZnLmNvbnRyb2xfbW9kZSwKICAgICAgICBzaW1fYmFja2VuZD0iZ3B1IiwKICAgICAgICByZW5kZXJfbW9kZT1yZW5kZXJfbW9kZSwKICAgICAgICByZXdhcmRfbW9kZT0ic3BhcnNlIiwKICAgICAgICBtYXhfZXBpc29kZV9zdGVwcz1jZmcubWF4X2VwaXNvZGVfc3RlcHMsCiAgICAgICAgZGlmZmljdWx0eT1jZmcuZGlmZmljdWx0eS5uYW1lLAogICAgICAgIG51bV9wYXJjZWxzPWNmZy5kaWZmaWN1bHR5Lm51bV9wYXJjZWxzLAogICAgICAgIGZpeGVkX3Bvc2VzPWNmZy5kaWZmaWN1bHR5LmZpeGVkX3Bvc2VzLAogICAgICAgIGNhbWVyYV93aWR0aD1jZmcuY2FtZXJhLndpZHRoLAogICAgICAgIGNhbWVyYV9oZWlnaHQ9Y2ZnLmNhbWVyYS5oZWlnaHQsCiAgICAgICAgb2JzX2NhbWVyYT1jZmcuZ2V0KCJvYnNfY2FtZXJhIiwgInNjZW5lIiksCiAgICAgICAgcmFuZG9taXphdGlvbj1PbWVnYUNvbmYudG9fY29udGFpbmVyKHJhbmRvbWl6YXRpb24sIHJlc29sdmU9VHJ1ZSksCiAgICApCgoKZGVmIGNvbXBvc2VfY2ZnKG92ZXJyaWRlcz1Ob25lLCBjb25maWdfZGlyPU5vbmUpOgogICAgIiIiTG9hZCB0aGUgSHlkcmEgY29uZmlnIG91dHNpZGUgdGhlIEBoeWRyYS5tYWluIHNjcmlwdHMgKHVzZWQgYnkgdGhlIG5vdGVib29rKS4iIiIKICAgIGltcG9ydCBvcwogICAgZnJvbSBoeWRyYSBpbXBvcnQgY29tcG9zZSwgaW5pdGlhbGl6ZV9jb25maWdfZGlyCiAgICBmcm9tIGh5ZHJhLmNvcmUuZ2xvYmFsX2h5ZHJhIGltcG9ydCBHbG9iYWxIeWRyYQoKICAgIGNvbmZpZ19kaXIgPSBvcy5wYXRoLmFic3BhdGgoY29uZmlnX2RpciBvciBvcy5wYXRoLmpvaW4ob3MuZ2V0Y3dkKCksICJjb25mIikpCiAgICBHbG9iYWxIeWRyYS5pbnN0YW5jZSgpLmNsZWFyKCkKICAgIHdpdGggaW5pdGlhbGl6ZV9jb25maWdfZGlyKHZlcnNpb25fYmFzZT1Ob25lLCBjb25maWdfZGlyPWNvbmZpZ19kaXIpOgogICAgICAgIHJldHVybiBjb21wb3NlKGNvbmZpZ19uYW1lPSJjb25maWciLCBvdmVycmlkZXM9b3ZlcnJpZGVzIG9yIFtdKQoKCmRlZiB0b19kZXZpY2Uob2JzLCBkZXZpY2UpOgogICAgaWYgaXNpbnN0YW5jZShvYnMsIGRpY3QpOgogICAgICAgIHJldHVybiB7azogdi50byhkZXZpY2UpIGZvciBrLCB2IGluIG9icy5pdGVtcygpfQogICAgcmV0dXJuIG9icy50byhkZXZpY2UpCgoKZGVmIGV4cGFuZF9zZWVkcyhzZWVkcywgbl9lcGlzb2Rlcyk6CiAgICAiIiJEZXRlcm1pbmlzdGljYWxseSBleHBhbmQgYSBiYXNlIHNlZWQgbGlzdCB0byBleGFjdGx5IG5fZXBpc29kZXMgc2VlZHMuIiIiCiAgICBzZWVkcyA9IGxpc3Qoc2VlZHMpCiAgICBvdXQgPSBbXQogICAgayA9IDAKICAgIHdoaWxlIGxlbihvdXQpIDwgbl9lcGlzb2RlczoKICAgICAgICBvdXQuYXBwZW5kKGludChzZWVkc1trICUgbGVuKHNlZWRzKV0pICsgKGsgLy8gbGVuKHNlZWRzKSkgKiAxMDAwMDMpCiAgICAgICAgayArPSAxCiAgICByZXR1cm4gb3V0WzpuX2VwaXNvZGVzXQoKCkB0b3JjaC5ub19ncmFkKCkKZGVmIHJvbGxvdXRfbWV0cmljcyhlbnYsIGFnZW50LCBkZXZpY2UsIG5fZXBpc29kZXMsIHNlZWRzLCBtYXhfc3RlcHMsIGRldGVybWluaXN0aWM9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAqLCBvbl9yZXNldD1Ob25lLCBvbl9iYXRjaD1Ob25lKToKICAgICIiIlJ1biBuX2VwaXNvZGVzIGFuZCBhZ2dyZWdhdGUgdGhlIMKnOS4xIG1ldHJpY3MuCgogICAgT3B0aW9uYWwgZGlhZ25vc3RpYyBvYnNlcnZlcnMgcmVjZWl2ZSB0aGUgZXhpc3RpbmcgcmVzZXQgYW5kIGZpbmFsIGV2YWx1YXRpb247CiAgICB0aGV5IG11c3Qgbm90IHN0ZXAvcmVzZXQgdGhlIGVudmlyb25tZW50IG9yIGNoYW5nZSB0aGUgcG9saWN5LiBObyBleHRyYSByb2xsb3V0cy4KICAgIE9ubHkgdGhlIGZpcnN0IGBgdGFrZWBgIHNsb3RzIGJlbG9uZyB0byB0aGUgcmVxdWVzdGVkIHNlZWQgbGlzdCBpbiBhIHBhZGRlZCBiYXRjaC4KICAgICIiIgogICAgYmFzZSA9IGVudi51bndyYXBwZWQKICAgIG5iID0gYmFzZS5udW1fZW52cwogICAgYWxsX3NlZWRzID0gZXhwYW5kX3NlZWRzKHNlZWRzLCBuX2VwaXNvZGVzKQogICAgdG90X3NvcnRlZCA9IHRvdF9taXMgPSB0b3RfcGFyY2VscyA9IDAuMAogICAgbl9hbGxfcGxhY2VkID0gMAogICAgc3RlcHNfc3VtID0gMC4wCiAgICBjb3VudGVkID0gMAogICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsIG5fZXBpc29kZXMsIG5iKToKICAgICAgICBiYXRjaF9zZWVkcyA9IGFsbF9zZWVkc1tzdGFydDpzdGFydCArIG5iXQogICAgICAgIHRha2UgPSBsZW4oYmF0Y2hfc2VlZHMpCiAgICAgICAgaWYgdGFrZSA8IG5iOgogICAgICAgICAgICBiYXRjaF9zZWVkcyA9IGJhdGNoX3NlZWRzICsgYWxsX3NlZWRzWzogbmIgLSB0YWtlXQogICAgICAgIG9icywgXyA9IGVudi5yZXNldChzZWVkPWJhdGNoX3NlZWRzKQogICAgICAgIGlmIG9uX3Jlc2V0IGlzIG5vdCBOb25lOgogICAgICAgICAgICBvbl9yZXNldChiYXNlLCBvYnMsIGJhdGNoX3NlZWRzWzp0YWtlXSkKICAgICAgICBpZiBoYXNhdHRyKGFnZW50LCAicmVzZXQiKToKICAgICAgICAgICAgYWdlbnQucmVzZXQoKSAgICMgZHJvcCBidWZmZXJlZCBhY3Rpb25zIC8gb2JzIGhpc3RvcnkgZnJvbSB0aGUgcHJldmlvdXMgYmF0Y2gKICAgICAgICBvYnMgPSB0b19kZXZpY2Uob2JzLCBkZXZpY2UpCiAgICAgICAgZm9yIF8gaW4gcmFuZ2UobWF4X3N0ZXBzIC0gMSk6CiAgICAgICAgICAgIG9icywgXywgXywgXywgXyA9IGVudi5zdGVwKGFnZW50LmFjdChvYnMsIGRldGVybWluaXN0aWM9ZGV0ZXJtaW5pc3RpYykpCiAgICAgICAgICAgIG9icyA9IHRvX2RldmljZShvYnMsIGRldmljZSkKICAgICAgICBldiA9IGJhc2UuZXZhbHVhdGUoKQogICAgICAgIGlmIG9uX2JhdGNoIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvbl9iYXRjaChiYXNlLCBldiwgYmF0Y2hfc2VlZHNbOnRha2VdKQogICAgICAgIHNjID0gZXZbInN1Y2Nlc3NfY291bnQiXVs6dGFrZV0KICAgICAgICB0b3Rfc29ydGVkICs9IHNjLnN1bSgpLml0ZW0oKQogICAgICAgIHRvdF9taXMgKz0gZXZbIm1pc19zb3J0X2NvdW50Il1bOnRha2VdLnN1bSgpLml0ZW0oKQogICAgICAgIHRvdF9wYXJjZWxzICs9IGJhc2UubnVtX3BhcmNlbHMgKiB0YWtlCiAgICAgICAgbl9hbGxfcGxhY2VkICs9IGV2WyJhbGxfcGxhY2VkIl1bOnRha2VdLnN1bSgpLml0ZW0oKQogICAgICAgIHN0ZXBzX3N1bSArPSBldlsic3RlcHNfdG9fY29tcGxldGUiXVs6dGFrZV0uZmxvYXQoKS5zdW0oKS5pdGVtKCkKICAgICAgICBjb3VudGVkICs9IHRha2UKICAgIHJldHVybiBkaWN0KAogICAgICAgIG5fZXBpc29kZXM9Y291bnRlZCwKICAgICAgICBudW1fcGFyY2Vscz1iYXNlLm51bV9wYXJjZWxzLAogICAgICAgIHNvcnRfYWNjdXJhY3k9dG90X3NvcnRlZCAvIG1heCh0b3RfcGFyY2VscywgMSksCiAgICAgICAgbWVhbl9zb3J0ZWQ9dG90X3NvcnRlZCAvIG1heChjb3VudGVkLCAxKSwKICAgICAgICBhbGxfcGxhY2VkX3JhdGU9bl9hbGxfcGxhY2VkIC8gbWF4KGNvdW50ZWQsIDEpLAogICAgICAgIG1lYW5fc3RlcHM9c3RlcHNfc3VtIC8gbWF4KGNvdW50ZWQsIDEpLAogICAgICAgIG1pc19zb3J0X3JhdGU9dG90X21pcyAvIG1heCh0b3RfcGFyY2VscywgMSksCiAgICApCgoKZGVmIHByaW50X21ldHJpY3Mocm9sZSwgZGlmZmljdWx0eSwgb2JzX21vZGUsIG0sIGhhcmQ9RmFsc2UpOgogICAgcHJpbnQoIi0iICogNTApCiAgICBwcmludChmIntyb2xlfSAgZGlmZmljdWx0eT17ZGlmZmljdWx0eX0gIG5fZXBpc29kZXM9e21bJ25fZXBpc29kZXMnXX0gIG9ic19tb2RlPXtvYnNfbW9kZX0iKQogICAgcHJpbnQoZiIgIFNPUlQgQUNDVVJBQ1k6ICAgICAgICB7bVsnc29ydF9hY2N1cmFjeSddICogMTAwOjUuMWZ9ICUgICAgICAjIFBSSU1BUlkgTUVUUklDIikKICAgIHByaW50KGYiICBtZWFuX3NvcnRlZC9lcGlzb2RlOiAge21bJ21lYW5fc29ydGVkJ106LjJmfSAvIHttWydudW1fcGFyY2VscyddfSIpCiAgICBwcmludChmIiAgYWxsX3BsYWNlZF9yYXRlOiAgICAgIHttWydhbGxfcGxhY2VkX3JhdGUnXTouM2Z9IikKICAgIHByaW50KGYiICBtaXNfc29ydF9yYXRlOiAgICAgICAge21bJ21pc19zb3J0X3JhdGUnXTouM2Z9ICAgICAgICAjIGRpYWdub3N0aWMiKQogICAgcHJpbnQoIi0iICogNTAsIGZsdXNoPVRydWUpCgoKZGVmIGxvYWRfYWdlbnQoY2twdF9wYXRoLCBlbnYsIGRldmljZSwgZW50cnlwb2ludD1Ob25lLCBwb2xpY3lfa3dhcmdzPU5vbmUpOgogICAgIiIiTG9hZCBhIHBvbGljeSBmb3IgZXZhbCAvIHRoZSBqdWRnZS4gUmVxdWlyZXMgYSBwb2xpY3kgZW50cnlwb2ludC4KCiAgICBlbnRyeXBvaW50IGZvcm1hdDogIm1vZHVsZTpmdW5jdGlvbiIgd2hlcmUKICAgICAgZnVuY3Rpb24oY2hlY2twb2ludCwgc2FtcGxlX29icywgYWN0aW9uX3NwYWNlLCBkZXZpY2UpIC0+IHBvbGljeSB3aXRoIC5hY3Qob2JzLCBkZXRlcm1pbmlzdGljPVRydWUpCgogICAgRXhhbXBsZToKICAgICAgcG9saWN5PXdhcmVob3VzZV9zb3J0LmlsX3BvbGljeTpsb2FkX2RwX3JnYiAgICAoUkdCIERpZmZ1c2lvbiBQb2xpY3kpCiAgICAiIiIKICAgIGlmIG5vdCBlbnRyeXBvaW50OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICJBIHBvbGljeSBlbnRyeXBvaW50IGlzIHJlcXVpcmVkLlxuIgogICAgICAgICAgICAiICBSR0IgRFA6ICBwb2xpY3k9d2FyZWhvdXNlX3NvcnQuaWxfcG9saWN5OmxvYWRfZHBfcmdiXG4iCiAgICAgICAgICAgICIgIEN1c3RvbTogIHBvbGljeT1teV9tb2R1bGU6bG9hZF9mblxuIgogICAgICAgICAgICAiICAgIHdoZXJlIGxvYWRfZm4oY2hlY2twb2ludCwgc2FtcGxlX29icywgYWN0aW9uX3NwYWNlLCBkZXZpY2UpIC0+IHBvbGljeSIKICAgICAgICApCiAgICBpbXBvcnQgaW1wb3J0bGliCiAgICBzYW1wbGVfb2JzID0gdG9fZGV2aWNlKGVudi5yZXNldChzZWVkPTApWzBdLCBkZXZpY2UpCiAgICBhY3Rpb25fc3BhY2UgPSBlbnYuc2luZ2xlX2FjdGlvbl9zcGFjZQogICAgbW9kX25hbWUsIGZuX25hbWUgPSBlbnRyeXBvaW50LnNwbGl0KCI6IikKICAgIGZuID0gZ2V0YXR0cihpbXBvcnRsaWIuaW1wb3J0X21vZHVsZShtb2RfbmFtZSksIGZuX25hbWUpCiAgICBwb2xpY3kgPSBmbihja3B0X3BhdGgsIHNhbXBsZV9vYnMsIGFjdGlvbl9zcGFjZSwgZGV2aWNlLCAqKmRpY3QocG9saWN5X2t3YXJncyBvciB7fSkpCiAgICBhc3NlcnQgaGFzYXR0cihwb2xpY3ksICJhY3QiKSwgZiJwb2xpY3kgZnJvbSB7ZW50cnlwb2ludH0gbXVzdCBkZWZpbmUgLmFjdChvYnMsIGRldGVybWluaXN0aWM9VHJ1ZSkiCiAgICByZXR1cm4gcG9saWN5LCBOb25lCgoKZGVmIHJlY29yZF9ldmFsX3ZpZGVvKGNmZywgb2JzX21vZGUsIHJhbmRvbWl6YXRpb24sIGFnZW50LCBkZXZpY2UsIG91dF9kaXIsCiAgICAgICAgICAgICAgICAgICAgICBuX2VudnM9MSwgc2VlZD0wLCBtYXhfc3RlcHM9Tm9uZSk6CiAgICAiIiJSZWNvcmQgYSBwb2xpY3kgcm9sbG91dCB0byBtcDQgdXNpbmcgTWFuaVNraWxsJ3MgUmVjb3JkRXBpc29kZSB3cmFwcGVyLiIiIgogICAgZnJvbSBtYW5pX3NraWxsLnV0aWxzLndyYXBwZXJzLnJlY29yZCBpbXBvcnQgUmVjb3JkRXBpc29kZQoKICAgIGVudiA9IF9neW1fbWFrZShjZmcsIG9ic19tb2RlLCByYW5kb21pemF0aW9uLCBuX2VudnMsIHJlbmRlcl9tb2RlPSJhbGwiKQogICAgaWYgb2JzX21vZGUgPT0gInJnYiI6CiAgICAgICAgZW52ID0gRmxhdHRlblJHQkRPYnNlcnZhdGlvbldyYXBwZXIoZW52LCByZ2I9VHJ1ZSwgZGVwdGg9RmFsc2UsIHN0YXRlPVRydWUpCiAgICBlbnYgPSBSZWNvcmRFcGlzb2RlKAogICAgICAgIGVudiwgb3V0cHV0X2Rpcj1vdXRfZGlyLCBzYXZlX3RyYWplY3Rvcnk9RmFsc2UsIHNhdmVfdmlkZW89VHJ1ZSwKICAgICAgICB2aWRlb19mcHM9MjAsIG1heF9zdGVwc19wZXJfdmlkZW89Y2ZnLm1heF9lcGlzb2RlX3N0ZXBzLAogICAgKQogICAgb2JzLCBfID0gZW52LnJlc2V0KHNlZWQ9c2VlZCkKICAgIGlmIGhhc2F0dHIoYWdlbnQsICJyZXNldCIpOgogICAgICAgIGFnZW50LnJlc2V0KCkKICAgIHN0ZXBzID0gbWF4X3N0ZXBzIG9yIGNmZy5tYXhfZXBpc29kZV9zdGVwcwogICAgZm9yIF8gaW4gcmFuZ2Uoc3RlcHMpOgogICAgICAgIG9icywgXywgXywgXywgXyA9IGVudi5zdGVwKGFnZW50LmFjdCh0b19kZXZpY2Uob2JzLCBkZXZpY2UpLCBkZXRlcm1pbmlzdGljPVRydWUpKQogICAgZW52LmNsb3NlKCkKICAgIHJldHVybiBvdXRfZGlyCgoKZGVmIGdpdF9oYXNoKCkgLT4gc3RyOgogICAgdHJ5OgogICAgICAgIHJldHVybiBzdWJwcm9jZXNzLmNoZWNrX291dHB1dCgKICAgICAgICAgICAgWyJnaXQiLCAicmV2LXBhcnNlIiwgIkhFQUQiXSwgc3RkZXJyPXN1YnByb2Nlc3MuREVWTlVMTAogICAgICAgICkuZGVjb2RlKCkuc3RyaXAoKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gInVua25vd24iCgoKZGVmIGxvZ19ydW5faGVhZGVyKGNmZywgcm9sZTogc3RyKToKICAgIHByaW50KCI9IiAqIDcwKQogICAgcHJpbnQoZiJbe3JvbGV9XSBnaXQ9e2dpdF9oYXNoKCl9IikKICAgIHByaW50KCItIiAqIDcwKQogICAgcHJpbnQoT21lZ2FDb25mLnRvX3lhbWwoY2ZnLCByZXNvbHZlPVRydWUpLnJzdHJpcCgpKQogICAgcHJpbnQoIj0iICogNzAsIGZsdXNoPVRydWUpCgoKZGVmIG1ha2VfZW52KAogICAgY2ZnLAogICAgb2JzX21vZGU6IHN0ciwKICAgIHJhbmRvbWl6YXRpb246IGRpY3QsCiAgICBudW1fZW52czogT3B0aW9uYWxbaW50XSA9IE5vbmUsCiAgICByZW5kZXJfbW9kZTogT3B0aW9uYWxbc3RyXSA9IE5vbmUsCiAgICByZWNvcmRfbWV0cmljczogYm9vbCA9IFRydWUsCiAgICBpZ25vcmVfdGVybWluYXRpb25zOiBib29sID0gVHJ1ZSwKICAgIHZpZGVvX2RpcjogT3B0aW9uYWxbc3RyXSA9IE5vbmUsCik6CiAgICAiIiJDb25zdHJ1Y3QgdGhlIFdhcmVob3VzZVNvcnQgZW52ICsgc3RhbmRhcmQgTWFuaVNraWxsIHZlY3RvciB3cmFwcGVycy4KCiAgICBSZXR1cm5zICh2ZWN0b3JfZW52LCBpc19yZ2IpLiBGb3IgcmdiIG9icyB0aGUgb2JzZXJ2YXRpb24gaXMgeyJyZ2IiLCAic3RhdGUifTsKICAgIGZvciBzdGF0ZSBvYnMgaXQgaXMgYSBmbGF0IHRlbnNvci4KICAgICIiIgogICAgZnJvbSBtYW5pX3NraWxsLnV0aWxzLndyYXBwZXJzLnJlY29yZCBpbXBvcnQgUmVjb3JkRXBpc29kZQoKICAgIG4gPSBpbnQobnVtX2VudnMgaWYgbnVtX2VudnMgaXMgbm90IE5vbmUgZWxzZSBjZmcubnVtX2VudnMpCiAgICBpc19yZ2IgPSBvYnNfbW9kZSA9PSAicmdiIgogICAgaWYgdmlkZW9fZGlyIGlzIG5vdCBOb25lIGFuZCByZW5kZXJfbW9kZSBpcyBOb25lOgogICAgICAgIHJlbmRlcl9tb2RlID0gImFsbCIKICAgIGVudiA9IF9neW1fbWFrZShjZmcsIG9ic19tb2RlLCByYW5kb21pemF0aW9uLCBuLCByZW5kZXJfbW9kZSkKICAgIGlmIGlzX3JnYjoKICAgICAgICBlbnYgPSBGbGF0dGVuUkdCRE9ic2VydmF0aW9uV3JhcHBlcihlbnYsIHJnYj1UcnVlLCBkZXB0aD1GYWxzZSwgc3RhdGU9VHJ1ZSkKICAgIGlmIHZpZGVvX2RpciBpcyBub3QgTm9uZToKICAgICAgICBlbnYgPSBSZWNvcmRFcGlzb2RlKAogICAgICAgICAgICBlbnYsIG91dHB1dF9kaXI9dmlkZW9fZGlyLCBzYXZlX3RyYWplY3Rvcnk9RmFsc2UsIHNhdmVfdmlkZW89VHJ1ZSwKICAgICAgICAgICAgdmlkZW9fZnBzPTIwLCBtYXhfc3RlcHNfcGVyX3ZpZGVvPWNmZy5tYXhfZXBpc29kZV9zdGVwcywKICAgICAgICApCiAgICBlbnYgPSBNYW5pU2tpbGxWZWN0b3JFbnYoCiAgICAgICAgZW52LCBudW1fZW52cz1uLCBpZ25vcmVfdGVybWluYXRpb25zPWlnbm9yZV90ZXJtaW5hdGlvbnMsIHJlY29yZF9tZXRyaWNzPXJlY29yZF9tZXRyaWNzCiAgICApCiAgICByZXR1cm4gZW52LCBpc19yZ2IKCgpkZWYgYXBwZW5kX2pzb25sKHBhdGgsIHJlY29yZCk6CiAgICAiIiJBcHBlbmQgb25lIEpTT04gbGluZSAoZXZhbCBib29ra2VlcGluZyBhY3Jvc3MgQ29sYWIgc2Vzc2lvbnMpLiIiIgogICAgaW1wb3J0IGpzb24KICAgIGltcG9ydCBvcwoKICAgIG9zLm1ha2VkaXJzKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgocGF0aCkpLCBleGlzdF9vaz1UcnVlKQogICAgd2l0aCBvcGVuKHBhdGgsICJhIikgYXMgZjoKICAgICAgICBmLndyaXRlKGpzb24uZHVtcHMocmVjb3JkLCBkZWZhdWx0PXN0cikgKyAiXG4iKQo='}}

import base64, hashlib, json, os, subprocess, time
from pathlib import Path
APPLY_HARD_SETUP = False
if APPLY_HARD_SETUP:
    # This namespace executes only the hash-checked, embedded stdlib Hard bootstrap.
    entry = EMBEDDED_SOURCES['tools/run_colab_hard.py']
    payload = base64.b64decode(entry['content_b64'], validate=True)
    assert hashlib.sha256(payload).hexdigest() == entry['sha256']
    hard_setup = {'__name__': 'hard_bootstrap'}
    exec(compile(payload, 'embedded/tools/run_colab_hard.py', 'exec'), hard_setup)
    EXP = 'hard_' + time.strftime('%Y%m%d_%H%M%S')
    PLAN = hard_setup['bootstrap'](EMBEDDED_SOURCES, EXP)
    print('Prepared new run:', PLAN)
    print('Reviewed originals backed up under source-backups; snapshots under sources.')
else:
    print('Setup skipped. Review this cell, then set APPLY_HARD_SETUP=True and execute only this cell.')


In [ ]:
# No kernel restart, clone, package installation, environment replacement, or data download.
REPO = Path('/content/berlin-marso-hackathon')
PY = '/content/marso-py312/bin/python'
ENV = dict(os.environ, DISPLAY='', PYOPENGL_PLATFORM='egl', HDF5_USE_FILE_LOCKING='FALSE',
           PYTHONUNBUFFERED='1', PYTHONPATH=str(REPO), PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
def hard_phase(phase):
    assert 'PLAN' in globals(), 'Run the reviewed setup cell first (or restore the exact saved PLAN path)'
    subprocess.run([PY, '-u', str(REPO/'tools/run_colab_hard.py'), str(PLAN), '--phase', phase],
                   cwd=REPO, env=ENV, check=True)

RUN_CHECKS = False
if RUN_CHECKS:
    hard_phase('checks')
else:
    print('Checks skipped. After parent stages local h5/json, enable only this cell.')


In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    hard_phase('smoke')
else:
    print('Smoke skipped. Enable after checks_passed: 100 iterations, all 200 demos, separate weights.')


In [ ]:
RUN_HARD = False
if RUN_HARD:
    hard_phase('run')
else:
    print('Full training skipped. Review smoke_passed, then enable only this cell.')
    print('40,000 iterations from fixed seed 1; no smoke checkpoint resume. Six-hour train deadline.')


In [ ]:
# Read-only. Set the exact PLAN path after reconnect; this cell never chooses or launches a run.
from pathlib import Path
import json
PLAN_PATH = str(PLAN) if 'PLAN' in globals() else ''
if PLAN_PATH:
    root = Path(PLAN_PATH).parent.parent
    for rel in ('run/status.json', 'run/summary.json', 'run/durability.json', 'run/train.log'):
        path = root / rel
        if path.is_file():
            with path.open('rb') as stream:
                stream.seek(max(0, path.stat().st_size - 5000))
                print(rel, stream.read().decode(errors='replace'))
else:
    print('Set PLAN_PATH to /content/marso-hard/hard_<timestamp>/run/plan.json. Do not relaunch blindly.')


In [ ]:
RETRY_FINAL_SYNC = False
if RETRY_FINAL_SYNC:
    hard_phase('finalize')
else:
    print('Optional: after compute_status=completed and remounting Drive, retry durability only.')
